In [ ]:
# ------------------------------------------------------------
# Dereddened SED for Fiber 215
# - Uses BDBS ugrizy photometry from your uploaded file
# - Dereddens magnitudes with Au, Ag, Ar, Ai, Az, Ay
# - Plots nu*Fnu points with error bars
# - NO line connecting the photometry points
# - Overlays a blackbody model using SP_Ace Teff
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from astropy import constants as const

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

from dotenv import load_dotenv
import os

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "README.md").exists() and (Path.cwd().parent / "README.md").exists():
    PROJECT_ROOT = Path.cwd().parent.resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

input_file = str(Path(os.getenv("INTERIM_DATA_DIR", PROJECT_ROOT / "data/interim")).resolve() / "CaT_summary_BDBS_XCSAO.csv")
fiber_target = 215

# Fill this in from your SP_Ace fit:
TEFF_SP_ACE = 4500.0   # <-- replace with the SP_Ace Teff for Fiber 215

# Output file
#output_png = f"SED_Fiber_{fiber_target}_dereddened_BB.png"

# ------------------------------------------------------------
# READ FILE
# ------------------------------------------------------------

df = pd.read_csv(input_file)

obj = df[df["fiberid"] == fiber_target].copy()
if len(obj) == 0:
    raise ValueError(f"No row found for fiberid = {fiber_target}")

row = obj.iloc[0]

print("Selected object:")
print(row[["fiberid", "RV", "S/N"]])

# ------------------------------------------------------------
# BAND DEFINITIONS
# BDBS effective wavelengths (approx, in microns)
# Zero points in Jy
# ugriz treated with AB zeropoint = 3631 Jy
# y band approximate zeropoint included
# ------------------------------------------------------------

bands = [
    # label, mag_col, err_col, A_col, lambda_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 3631.0),
]

# ------------------------------------------------------------
# BUILD DEREDDENED SED TABLE
# ------------------------------------------------------------

c_cgs = const.c.cgs.value          # cm/s
h_cgs = const.h.cgs.value          # erg s
k_B_cgs = const.k_B.cgs.value      # erg/K

sed_rows = []

for band, mag_col, err_col, A_col, lam_um, zp_jy in bands:

    mag = pd.to_numeric(row[mag_col], errors="coerce")
    emag = pd.to_numeric(row[err_col], errors="coerce")
    A_lambda = pd.to_numeric(row[A_col], errors="coerce")

    if not np.isfinite(mag):
        continue
    if mag > 90 or mag < -90:
        continue

    # Dereddened magnitude
    mag0 = mag - A_lambda

    # Convert to Fnu
    # Fnu[Jy] = zp * 10^(-0.4 * mag0)
    fnu_jy = zp_jy * 10 ** (-0.4 * mag0)
    fnu_cgs = fnu_jy * 1e-23   # erg/s/cm^2/Hz

    # Frequency
    lam_cm = lam_um * 1e-4
    nu_hz = c_cgs / lam_cm

    # nuFnu
    nu_fnu = nu_hz * fnu_cgs

    # magnitude error propagation
    if np.isfinite(emag) and emag > 0 and emag < 5:
        frac_err = np.log(10) * 0.4 * emag
        nu_fnu_err = nu_fnu * frac_err
    else:
        nu_fnu_err = np.nan

    sed_rows.append({
        "band": band,
        "lambda_um": lam_um,
        "lambda_cm": lam_cm,
        "nu_hz": nu_hz,
        "mag_obs": mag,
        "mag0": mag0,
        "mag_err": emag,
        "A_lambda": A_lambda,
        "fnu_jy": fnu_jy,
        "nu_fnu": nu_fnu,
        "nu_fnu_err": nu_fnu_err
    })

sed = pd.DataFrame(sed_rows)
sed = sed.sort_values("lambda_um").reset_index(drop=True)

print("\nDereddened photometry used:")
print(sed[["band", "mag_obs", "A_lambda", "mag0", "mag_err", "nu_fnu", "nu_fnu_err"]])

# ------------------------------------------------------------
# BLACKBODY MODEL
# We use SP_Ace Teff and solve only for a scale factor.
# Shape is from Planck function B_nu(T)
# ------------------------------------------------------------

def planck_Bnu_cgs(nu, T):
    """
    Planck function B_nu in cgs:
    erg s^-1 cm^-2 Hz^-1 sr^-1
    """
    x = h_cgs * nu / (k_B_cgs * T)
    # avoid overflow
    x = np.clip(x, 1e-8, 700)
    return (2.0 * h_cgs * nu**3 / c_cgs**2) / (np.exp(x) - 1.0)

def bb_nuFnu_model(nu, T, scale):
    """
    Returns scaled nu*B_nu(T); scale absorbs radius/distance/etc.
    """
    return scale * nu * planck_Bnu_cgs(nu, T)

# observed data for fit
nu_obs = sed["nu_hz"].to_numpy(dtype=float)
y_obs = sed["nu_fnu"].to_numpy(dtype=float)
yerr_obs = sed["nu_fnu_err"].to_numpy(dtype=float)

# if any missing errors, replace with fractional 10% just so fit works
bad_err = ~np.isfinite(yerr_obs) | (yerr_obs <= 0)
if np.any(bad_err):
    yerr_obs[bad_err] = 0.10 * y_obs[bad_err]

def chi2_logscale(log_scale):
    scale = 10 ** log_scale
    model = bb_nuFnu_model(nu_obs, TEFF_SP_ACE, scale)
    return np.sum(((y_obs - model) / yerr_obs) ** 2)

res = minimize_scalar(chi2_logscale, bounds=(-40, 20), method="bounded")
best_log_scale = res.x
best_scale = 10 ** best_log_scale

print(f"\nSP_Ace Teff used: {TEFF_SP_ACE:.1f} K")
print(f"Best-fit BB scale: {best_scale:.5e}")

# ------------------------------------------------------------
# MODEL CURVE
# ------------------------------------------------------------

lam_grid_um = np.logspace(np.log10(0.30), np.log10(1.20), 500)
lam_grid_cm = lam_grid_um * 1e-4
nu_grid = c_cgs / lam_grid_cm

model_nuFnu = bb_nuFnu_model(nu_grid, TEFF_SP_ACE, best_scale)

# sort for plotting left-to-right in wavelength
sort_idx = np.argsort(lam_grid_um)
lam_grid_um = lam_grid_um[sort_idx]
model_nuFnu = model_nuFnu[sort_idx]

# ------------------------------------------------------------
# AXIS LIMITS
# Include error bars in y-axis range
# ------------------------------------------------------------

y = sed["nu_fnu"].to_numpy(dtype=float)
yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

y_low = y.copy()
y_high = y.copy()

good = np.isfinite(yerr) & (yerr > 0)
y_low[good] = y[good] - yerr[good]
y_high[good] = y[good] + yerr[good]

# log axis requires positive lower bound
pos_low = np.isfinite(y_low) & (y_low > 0)
pos_high = np.isfinite(y_high) & (y_high > 0)

ymin = np.nanmin(np.concatenate([y_low[pos_low], model_nuFnu[model_nuFnu > 0]]))
ymax = np.nanmax(np.concatenate([y_high[pos_high], model_nuFnu[model_nuFnu > 0]]))

log_ymin = np.log10(ymin)
log_ymax = np.log10(ymax)
ypad = 0.12 * (log_ymax - log_ymin if log_ymax > log_ymin else 1.0)

ylim_low = 10 ** (log_ymin - ypad)
ylim_high = 10 ** (log_ymax + ypad)

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

# photometry points WITH error bars, NO connecting line
ax.errorbar(
    sed["lambda_um"],
    sed["nu_fnu"],
    yerr=sed["nu_fnu_err"],
    fmt="o",
    markersize=7,
    capsize=3,
    linestyle="none",
    label=f"Fiber {fiber_target} dereddened photometry"
)

# annotate band names
for _, r in sed.iterrows():
    ax.annotate(
        r["band"],
        (r["lambda_um"], r["nu_fnu"]),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=9
    )

# blackbody model
ax.plot(
    lam_grid_um,
    model_nuFnu,
    linewidth=1.8,
    label=f"Blackbody, T = {TEFF_SP_ACE:.0f} K"
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.32, 1.10)
ax.set_ylim(ylim_low, ylim_high)

ax.set_xlabel("Wavelength [micron]")
ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_title(f"Fiber {fiber_target}: Dereddened SED + BB overlay")

ax.grid(alpha=0.25, which="both")
ax.legend(loc="best", fontsize=9)

plt.tight_layout()
plt.savefig(output_png, dpi=300)
plt.show()

#print(f"\nSaved: {output_png}")

In [ ]:
# ------------------------------------------------------------
# Repair/download MARCS wavelength file
# Fixes the '<!DOCTYPE' error
# ------------------------------------------------------------

import os
import gzip
import urllib.request
import numpy as np

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

wave_gz = os.path.join(MARCS_DIR, "flx_wavelengths.vac.gz")
wave_txt = os.path.join(MARCS_DIR, "flx_wavelengths.vac")

# Correct MARCS wavelength-file URL.
# Note: this is NOT in /data/.
url = "https://marcs.astro.uu.se/flx_wavelengths.vac.gz"


def looks_like_html(path):
    """
    Return True if the file is actually an HTML page rather than numeric data.
    """
    if not os.path.exists(path):
        return False

    with open(path, "rb") as f:
        start = f.read(200).lstrip().lower()

    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def is_gzip_file(path):
    """
    Check real gzip header.
    """
    if not os.path.exists(path):
        return False

    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


# Remove bad HTML wavelength files
for path in [wave_gz, wave_txt]:
    if looks_like_html(path):
        print(f"Removing bad HTML file: {path}")
        os.remove(path)

# Download the correct file if needed
if not os.path.exists(wave_gz) and not os.path.exists(wave_txt):
    print("Downloading correct MARCS wavelength file...")
    urllib.request.urlretrieve(url, wave_gz)
    print(f"Downloaded: {wave_gz}")

# Test-read the wavelength file
if os.path.exists(wave_gz):
    if looks_like_html(wave_gz):
        raise RuntimeError(
            "Downloaded file is still HTML. Open the MARCS documentation page "
            "in a browser and manually download flx_wavelengths.vac.gz."
        )

    if is_gzip_file(wave_gz):
        with gzip.open(wave_gz, "rt") as f:
            wave_A = np.loadtxt(f)
    else:
        # Sometimes a file is plain text even though it has .gz in the name
        wave_A = np.loadtxt(wave_gz)

elif os.path.exists(wave_txt):
    if looks_like_html(wave_txt):
        raise RuntimeError("flx_wavelengths.vac is HTML, not numeric data.")
    wave_A = np.loadtxt(wave_txt)

else:
    raise FileNotFoundError("No MARCS wavelength file found.")

wave_A = np.ravel(wave_A)

print("Wavelength file is OK.")
print(f"N wavelengths = {len(wave_A)}")
print("First five wavelengths [Angstrom]:")
print(wave_A[:5])
print("Last five wavelengths [Angstrom]:")
print(wave_A[-5:])

In [ ]:
# ------------------------------------------------------------
# Dereddened BDBS SED + nearest MARCS model
#
# Uses:
#   1. CaT_summary_BDBS_XCSAO.csv for photometry/extinction
#   2. SP_Ace_averages_with_FeHDP_avg_std.csv for Teff/logg/[Fe/H]/[alpha/Fe]
#
# Change only fiber_to_plot.
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import constants as const


# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

fiber_to_plot = 215

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

# The code will use whichever of these exists
summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

output_png = rf"/Users/kerrycheon/repos/SpectraReductions2026/space\SED_Fiber_{fiber_to_plot}_MARCS.png"


# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

c_cgs = const.c.cgs.value


# ------------------------------------------------------------
# FIND SUMMARY FILE
# ------------------------------------------------------------

summary_file = None

for f in summary_file_candidates:
    if os.path.exists(f):
        summary_file = f
        break

if summary_file is None:
    raise FileNotFoundError(
        "Could not find the SP_Ace summary file. Tried:\n"
        + "\n".join(summary_file_candidates)
    )

print("Using photometry file:")
print(phot_file)

print("\nUsing SP_Ace summary file:")
print(summary_file)

print("\nUsing MARCS directory:")
print(MARCS_DIR)


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def is_gzip_file(path):
    """
    Check the true file header, not just the .gz extension.
    """
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def read_numeric_file(path):
    """
    Read whitespace numeric files.
    Handles true gzip files and plain-text files mislabeled as .gz.
    """
    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    """
    Find MARCS wavelength file.
    """
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            "Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz "
            f"inside {marcs_dir}"
        )

    return matches[0]


def parse_marcs_filename(path):
    """
    Parse MARCS model filenames.

    Example:
    s4750_g+3.0_m1.0_t02_st_z-0.75_a+0.30_c+0.00_n+0.00_o+0.30_r+0.00_s+0.00.flx.gz
    """
    name = os.path.basename(path)

    pattern = (
        r"^(?P<geom>[sp])"
        r"(?P<teff>\d{4,5})"
        r"_g(?P<logg>[+-]\d+\.\d)"
        r"_m(?P<mass>[0-9]+\.[0-9])"
        r"_t(?P<vturb>\d{2})"
        r"_(?P<abund>[a-z]{2})"
        r"_z(?P<feh>[+-]\d+\.\d{2})"
        r"_a(?P<alpha>[+-]\d+\.\d{2})"
    )

    m = re.search(pattern, name)

    if m is None:
        return None

    d = m.groupdict()

    return {
        "path": path,
        "filename": name,
        "geom": d["geom"],
        "teff": float(d["teff"]),
        "logg": float(d["logg"]),
        "mass": float(d["mass"]),
        "vturb": float(d["vturb"]) / 10.0,
        "abund": d["abund"],
        "feh": float(d["feh"]),
        "alpha": float(d["alpha"]),
    }


def find_nearest_marcs_model(marcs_dir, teff_target, logg_target, feh_target, alpha_target):
    """
    Find nearest MARCS .flx or .flx.gz model to SP_Ace parameters.
    """

    flx_files = (
        glob.glob(os.path.join(marcs_dir, "**", "*.flx"), recursive=True)
        + glob.glob(os.path.join(marcs_dir, "**", "*.flx.gz"), recursive=True)
    )

    if len(flx_files) == 0:
        raise FileNotFoundError(
            f"No .flx or .flx.gz files found inside:\n{marcs_dir}"
        )

    records = []

    for path in flx_files:
        info = parse_marcs_filename(path)
        if info is not None:
            records.append(info)

    if len(records) == 0:
        raise ValueError(
            "Found MARCS flux files, but could not parse their filenames."
        )

    grid = pd.DataFrame(records)

    print(f"\nParsed {len(grid)} MARCS flux files.")

    # Geometry choice:
    # low-gravity giants: spherical
    # high-gravity dwarfs: plane parallel
    if logg_target <= 3.5:
        preferred_geom = "s"
    else:
        preferred_geom = "p"

    sub = grid[grid["geom"] == preferred_geom].copy()
    if len(sub) > 0:
        grid = sub
        print(f"Using MARCS geometry: {preferred_geom}")
    else:
        print("Preferred geometry unavailable; using all geometries.")

    # Prefer standard abundance grid, if available
    sub = grid[grid["abund"] == "st"].copy()
    if len(sub) > 0:
        grid = sub
        print("Using MARCS abundance grid: st")
    else:
        print("No st abundance grid found; using available grid.")

    # Weighted distance in grid units
    grid["distance"] = (
        ((grid["teff"] - teff_target) / 100.0) ** 2
        + ((grid["logg"] - logg_target) / 0.5) ** 2
        + ((grid["feh"] - feh_target) / 0.25) ** 2
    )

    if np.isfinite(alpha_target):
        grid["distance"] += ((grid["alpha"] - alpha_target) / 0.20) ** 2

    best = grid.sort_values("distance").iloc[0]

    return best, grid


# ------------------------------------------------------------
# READ AND MERGE FILES
# ------------------------------------------------------------

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(
    summary,
    on="fiberid",
    how="left",
    suffixes=("", "_SPsummary")
)

obj = df[df["fiberid"] == fiber_to_plot].copy()

if len(obj) == 0:
    raise ValueError(f"No object found with fiberid = {fiber_to_plot}")

if len(obj) > 1:
    print(f"Warning: found {len(obj)} rows for Fiber {fiber_to_plot}. Using the first row.")

row = obj.iloc[0]


# ------------------------------------------------------------
# READ MODEL PARAMETERS FROM SUMMARY FILE
# ------------------------------------------------------------

teff = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
logg = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
feh = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
alpha = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

if not np.isfinite(teff) or not np.isfinite(logg) or not np.isfinite(feh):
    raise ValueError(
        f"Fiber {fiber_to_plot} does not have finite SP_Ace parameters:\n"
        f"Teff_avg_SP = {teff}\n"
        f"logg_avg_SP = {logg}\n"
        f"FeH_avg_SP = {feh}\n"
    )

print("\nSelected object:")
print(f"Fiber = {fiber_to_plot}")

if "RV" in df.columns:
    print(f"RV = {row['RV']}")
if "S/N" in df.columns:
    print(f"S/N = {row['S/N']}")

print("\nSP_Ace parameters:")
print(f"Teff_avg_SP = {teff:.1f} K")
print(f"logg_avg_SP = {logg:.3f}")
print(f"FeH_avg_SP  = {feh:.3f}")
print(f"aFe_avg_SP  = {alpha:.3f}")


# ------------------------------------------------------------
# BUILD DEREDDENED BDBS SED
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_um, zeropoint_Jy
    ("u", "umag", "uerr", "Au", 0.36, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 3631.0),
]

sed_rows = []

for band, mag_col, err_col, A_col, lam_um, zp_jy in bands:

    mag = pd.to_numeric(row[mag_col], errors="coerce")
    emag = pd.to_numeric(row[err_col], errors="coerce")
    A_lambda = pd.to_numeric(row[A_col], errors="coerce")

    if not np.isfinite(mag):
        continue
    if mag > 90 or mag < -90:
        continue
    if not np.isfinite(A_lambda):
        A_lambda = 0.0

    # Dereddened magnitude
    mag0 = mag - A_lambda

    # Convert dereddened magnitude to Fnu
    fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
    fnu_cgs = fnu_jy * 1.0e-23

    lam_cm = lam_um * 1.0e-4
    nu_hz = c_cgs / lam_cm

    nu_fnu = nu_hz * fnu_cgs

    # Error propagation from magnitude error
    if np.isfinite(emag) and 0 < emag < 5:
        frac_err = 0.4 * np.log(10.0) * emag
        nu_fnu_err = nu_fnu * frac_err
    else:
        nu_fnu_err = np.nan

    sed_rows.append({
        "band": band,
        "lambda_um": lam_um,
        "nu_hz": nu_hz,
        "mag_obs": mag,
        "A_lambda": A_lambda,
        "mag0": mag0,
        "mag_err": emag,
        "nu_fnu": nu_fnu,
        "nu_fnu_err": nu_fnu_err,
    })

sed = pd.DataFrame(sed_rows).sort_values("lambda_um").reset_index(drop=True)

if len(sed) == 0:
    raise ValueError("No usable BDBS photometry found for this fiber.")

print("\nDereddened photometry:")
print(sed[["band", "lambda_um", "mag_obs", "A_lambda", "mag0", "mag_err", "nu_fnu", "nu_fnu_err"]])


# ------------------------------------------------------------
# FIND NEAREST MARCS MODEL
# ------------------------------------------------------------

best_model, marcs_grid = find_nearest_marcs_model(
    MARCS_DIR,
    teff,
    logg,
    feh,
    alpha
)

print("\nNearest MARCS model:")
print(best_model[["filename", "geom", "teff", "logg", "feh", "alpha", "abund", "vturb", "distance"]])


# ------------------------------------------------------------
# READ MARCS WAVELENGTH GRID AND FLUX MODEL
# ------------------------------------------------------------

wave_file = find_wavelength_file(MARCS_DIR)

print("\nUsing MARCS wavelength file:")
print(wave_file)

print("\nUsing MARCS flux file:")
print(best_model["path"])

wave_A = read_numeric_file(wave_file)
flux_lambda = read_numeric_file(best_model["path"])

n = min(len(wave_A), len(flux_lambda))
wave_A = wave_A[:n]
flux_lambda = flux_lambda[:n]

good = (
    np.isfinite(wave_A)
    & np.isfinite(flux_lambda)
    & (wave_A > 0)
    & (flux_lambda > 0)
)

wave_A = wave_A[good]
flux_lambda = flux_lambda[good]

# MARCS .flx units:
# F_lambda = erg cm^-2 s^-1 Angstrom^-1 at stellar surface.
#
# Convert to nu Fnu:
# F_lambda per cm = F_lambda per Angstrom * 1e8
# F_nu = F_lambda * lambda^2 / c
# nu Fnu = nu * F_nu

wave_cm = wave_A * 1.0e-8
lambda_um = wave_A * 1.0e-4
nu_hz = c_cgs / wave_cm

flux_lambda_per_cm = flux_lambda * 1.0e8
flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
model_nu_fnu_surface = nu_hz * flux_nu

# Restrict model to photometry wavelength region
model_mask = (lambda_um >= 0.30) & (lambda_um <= 1.20)

lambda_um_model = lambda_um[model_mask]
model_nu_fnu_surface = model_nu_fnu_surface[model_mask]

idx = np.argsort(lambda_um_model)
lambda_um_model = lambda_um_model[idx]
model_nu_fnu_surface = model_nu_fnu_surface[idx]


# ------------------------------------------------------------
# FIT ONLY MODEL NORMALIZATION
# Weighted least-squares scale:
# scale = sum(model * data / sigma^2) / sum(model^2 / sigma^2)
# ------------------------------------------------------------

obs_lam = sed["lambda_um"].to_numpy(dtype=float)
obs_y = sed["nu_fnu"].to_numpy(dtype=float)
obs_yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

obs_yerr_fit = obs_yerr.copy()

bad_err = ~np.isfinite(obs_yerr_fit) | (obs_yerr_fit <= 0)
obs_yerr_fit[bad_err] = 0.10 * obs_y[bad_err]

model_interp = np.interp(obs_lam, lambda_um_model, model_nu_fnu_surface)

good_fit = (
    np.isfinite(obs_y)
    & np.isfinite(obs_yerr_fit)
    & np.isfinite(model_interp)
    & (obs_y > 0)
    & (obs_yerr_fit > 0)
    & (model_interp > 0)
)

if np.sum(good_fit) < 2:
    raise ValueError("Not enough good photometry points to normalize the MARCS model.")

w = 1.0 / obs_yerr_fit[good_fit]**2

best_scale = np.sum(model_interp[good_fit] * obs_y[good_fit] * w) / np.sum(model_interp[good_fit]**2 * w)

scaled_model_nu_fnu = best_scale * model_nu_fnu_surface

model_at_points = best_scale * model_interp[good_fit]
chi2 = np.sum(((obs_y[good_fit] - model_at_points) / obs_yerr_fit[good_fit])**2)
dof = np.sum(good_fit) - 1

print(f"\nBest-fit normalization scale = {best_scale:.5e}")
print(f"chi2 = {chi2:.2f}")
print(f"dof = {dof}")
if dof > 0:
    print(f"reduced chi2 = {chi2/dof:.2f}")


# ------------------------------------------------------------
# AXIS LIMITS INCLUDING ERROR BARS AND MODEL
# ------------------------------------------------------------

y_low = obs_y.copy()
y_high = obs_y.copy()

good_err = np.isfinite(obs_yerr) & (obs_yerr > 0)
y_low[good_err] = obs_y[good_err] - obs_yerr[good_err]
y_high[good_err] = obs_y[good_err] + obs_yerr[good_err]

positive_data_low = y_low[np.isfinite(y_low) & (y_low > 0)]
positive_data_high = y_high[np.isfinite(y_high) & (y_high > 0)]
positive_model = scaled_model_nu_fnu[
    np.isfinite(scaled_model_nu_fnu) & (scaled_model_nu_fnu > 0)
]

ymin = np.nanmin(np.concatenate([positive_data_low, positive_model]))
ymax = np.nanmax(np.concatenate([positive_data_high, positive_model]))

log_ymin = np.log10(ymin)
log_ymax = np.log10(ymax)

ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

ylim_low = 10.0 ** (log_ymin - ypad)
ylim_high = 10.0 ** (log_ymax + ypad)


# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.8, 5.6))

# Photometry points only. No connecting line.
ax.errorbar(
    sed["lambda_um"],
    sed["nu_fnu"],
    yerr=sed["nu_fnu_err"],
    fmt="o",
    markersize=7,
    capsize=3,
    linestyle="none",
    label=f"Fiber {fiber_to_plot} dereddened BDBS photometry"
)

for _, r in sed.iterrows():
    ax.annotate(
        r["band"],
        (r["lambda_um"], r["nu_fnu"]),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=9
    )

model_label = (
    "Nearest MARCS "
    f"T={best_model['teff']:.0f} K, "
    f"logg={best_model['logg']:.1f}, "
    f"[Fe/H]={best_model['feh']:.2f}, "
    f"[α/Fe]={best_model['alpha']:.2f}"
)

ax.plot(
    lambda_um_model,
    scaled_model_nu_fnu,
    linewidth=2.0,
    label=model_label
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(0.32, 1.10)
ax.set_ylim(ylim_low, ylim_high)

ax.set_xlabel("Wavelength [micron]")
ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_title(f"Fiber {fiber_to_plot}: dereddened SED with MARCS model")

ax.grid(alpha=0.25, which="both")
ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.savefig(output_png, dpi=300)
plt.show()

print(f"\nSaved plot to:\n{output_png}")

In [ ]:
# ------------------------------------------------------------
# Dereddened BDBS SED + smoothed nearest MARCS model
#
# Uses:
#   /Users/kerrycheon/repos/SpectraReductions2026/space/CaT_summary_BDBS_XCSAO.csv
#   /Users/kerrycheon/repos/SpectraReductions2026/space/SP_Ace_averages_with_FeHDP_avg_std.csv
#   /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS
#
# Produces:
#   Dereddened ugrizy photometry with error bars
#   NO connecting line between photometry points
#   Nearest MARCS model from SP_Ace Teff/logg/[Fe/H]/[alpha/Fe]
#   Smoothed/binned MARCS model for clean SED display
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

fiber_to_plot = 69

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

output_png = rf"/Users/kerrycheon/repos/SpectraReductions2026/space\SED_Fiber_{fiber_to_plot}_MARCS_smoothed.png"

# Number of bins for the displayed MARCS curve.
# Smaller = smoother. Try 80, 120, 180.
model_smoothing_bins = 200

# Optional: if one band is clearly bad, list it here.
# Example: exclude_bands = ["r"]
exclude_bands = []


# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

c_cgs = 2.99792458e10   # cm/s


# ------------------------------------------------------------
# FIND SUMMARY FILE
# ------------------------------------------------------------

summary_file = None

for f in summary_file_candidates:
    if os.path.exists(f):
        summary_file = f
        break

if summary_file is None:
    raise FileNotFoundError(
        "Could not find the SP_Ace summary file. Tried:\n"
        + "\n".join(summary_file_candidates)
    )

print("Using photometry file:")
print(phot_file)

print("\nUsing SP_Ace summary file:")
print(summary_file)

print("\nUsing MARCS directory:")
print(MARCS_DIR)


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def is_gzip_file(path):
    """
    Check the real file header, not the extension.
    """
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    """
    Detect failed downloads saved as HTML.
    """
    if not os.path.exists(path):
        return False

    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()

    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    """
    Read a whitespace numeric file.
    Handles:
      - real gzip files
      - plain text files
      - plain text files mislabeled .gz
    """
    if looks_like_html(path):
        raise ValueError(
            f"This file is HTML, not numeric MARCS data:\n{path}\n\n"
            "It was probably a failed download. Re-download flx_wavelengths.vac.gz "
            "from the MARCS site."
        )

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    """
    Find the MARCS wavelength file.
    """
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            "Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz "
            f"inside {marcs_dir}"
        )

    return matches[0]


def parse_marcs_filename(path):
    """
    Parse MARCS flux filename.

    Example:
    s4750_g+3.0_m1.0_t02_st_z-0.75_a+0.30_c+0.00_n+0.00_o+0.30_r+0.00_s+0.00.flx.gz
    """
    name = os.path.basename(path)

    pattern = (
        r"^(?P<geom>[sp])"
        r"(?P<teff>\d{4,5})"
        r"_g(?P<logg>[+-]\d+\.\d)"
        r"_m(?P<mass>[0-9]+\.[0-9])"
        r"_t(?P<vturb>\d{2})"
        r"_(?P<abund>[a-z]{2})"
        r"_z(?P<feh>[+-]\d+\.\d{2})"
        r"_a(?P<alpha>[+-]\d+\.\d{2})"
    )

    m = re.search(pattern, name)

    if m is None:
        return None

    d = m.groupdict()

    return {
        "path": path,
        "filename": name,
        "geom": d["geom"],
        "teff": float(d["teff"]),
        "logg": float(d["logg"]),
        "mass": float(d["mass"]),
        "vturb": float(d["vturb"]) / 10.0,
        "abund": d["abund"],
        "feh": float(d["feh"]),
        "alpha": float(d["alpha"]),
    }


def find_nearest_marcs_model(marcs_dir, teff_target, logg_target, feh_target, alpha_target):
    """
    Find nearest MARCS .flx or .flx.gz model.
    """

    flx_files = (
        glob.glob(os.path.join(marcs_dir, "**", "*.flx"), recursive=True)
        + glob.glob(os.path.join(marcs_dir, "**", "*.flx.gz"), recursive=True)
    )

    if len(flx_files) == 0:
        raise FileNotFoundError(
            f"No .flx or .flx.gz files found inside:\n{marcs_dir}"
        )

    records = []

    for path in flx_files:
        info = parse_marcs_filename(path)
        if info is not None:
            records.append(info)

    if len(records) == 0:
        raise ValueError("Found MARCS flux files, but could not parse their filenames.")

    grid = pd.DataFrame(records)

    print(f"\nParsed {len(grid)} MARCS flux files.")

    # Geometry choice:
    # low-gravity giants: spherical
    # high-gravity dwarfs: plane-parallel
    if logg_target <= 3.5:
        preferred_geom = "s"
    else:
        preferred_geom = "p"

    sub = grid[grid["geom"] == preferred_geom].copy()
    if len(sub) > 0:
        grid = sub
        print(f"Using MARCS geometry: {preferred_geom}")
    else:
        print("Preferred geometry unavailable; using all geometries.")

    # Prefer standard abundance grid if available
    sub = grid[grid["abund"] == "st"].copy()
    if len(sub) > 0:
        grid = sub
        print("Using MARCS abundance grid: st")
    else:
        print("No st abundance grid found; using available grid.")

    # Weighted distance in grid units
    grid["distance"] = (
        ((grid["teff"] - teff_target) / 100.0) ** 2
        + ((grid["logg"] - logg_target) / 0.5) ** 2
        + ((grid["feh"] - feh_target) / 0.25) ** 2
    )

    if np.isfinite(alpha_target):
        grid["distance"] += ((grid["alpha"] - alpha_target) / 0.20) ** 2

    best = grid.sort_values("distance").iloc[0]

    return best, grid


def bin_model_in_log_lambda(lambda_um, flux, n_bins=120, statistic="median"):
    """
    Bin the high-resolution MARCS model in log wavelength.

    This makes the displayed curve represent the broadband SED shape
    instead of every narrow absorption feature.

    statistic options:
      "median" = good default
      "mean"   = averages all line blanketing
      "p85"    = smoother upper-envelope continuum-like curve
    """
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lambda_binned = []
    flux_binned = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lambda_binned.append(10 ** np.nanmean(loglam[m]))

        if statistic == "mean":
            flux_binned.append(np.nanmean(flux[m]))
        elif statistic == "p85":
            flux_binned.append(np.nanpercentile(flux[m], 85))
        else:
            flux_binned.append(np.nanmedian(flux[m]))

    return np.array(lambda_binned), np.array(flux_binned)


# ------------------------------------------------------------
# READ AND MERGE DATA
# ------------------------------------------------------------

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(
    summary,
    on="fiberid",
    how="left",
    suffixes=("", "_SPsummary")
)

obj = df[df["fiberid"] == fiber_to_plot].copy()

if len(obj) == 0:
    raise ValueError(f"No object found with fiberid = {fiber_to_plot}")

if len(obj) > 1:
    print(f"Warning: found {len(obj)} rows for Fiber {fiber_to_plot}. Using the first row.")

row = obj.iloc[0]


# ------------------------------------------------------------
# READ SP_ACE PARAMETERS
# ------------------------------------------------------------

teff = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
logg = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
feh = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
alpha = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

if not np.isfinite(teff) or not np.isfinite(logg) or not np.isfinite(feh):
    raise ValueError(
        f"Fiber {fiber_to_plot} does not have finite SP_Ace parameters:\n"
        f"Teff_avg_SP = {teff}\n"
        f"logg_avg_SP = {logg}\n"
        f"FeH_avg_SP = {feh}\n"
    )

print("\nSelected object:")
print(f"Fiber = {fiber_to_plot}")

if "RV" in df.columns:
    print(f"RV = {row['RV']}")
if "S/N" in df.columns:
    print(f"S/N = {row['S/N']}")

print("\nSP_Ace parameters:")
print(f"Teff_avg_SP = {teff:.1f} K")
print(f"logg_avg_SP = {logg:.3f}")
print(f"FeH_avg_SP  = {feh:.3f}")
print(f"aFe_avg_SP  = {alpha:.3f}")


# ------------------------------------------------------------
# BUILD DEREDDENED BDBS SED
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_um, zeropoint_Jy
    ("u", "umag", "uerr", "Au", 0.36, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 3631.0),
]

sed_rows = []

for band, mag_col, err_col, A_col, lam_um, zp_jy in bands:

    if band in exclude_bands:
        continue

    mag = pd.to_numeric(row[mag_col], errors="coerce")
    emag = pd.to_numeric(row[err_col], errors="coerce")
    A_lambda = pd.to_numeric(row[A_col], errors="coerce")

    if not np.isfinite(mag):
        continue
    if mag > 90 or mag < -90:
        continue
    if not np.isfinite(A_lambda):
        A_lambda = 0.0

    # Dereddened magnitude
    mag0 = mag - A_lambda

    # Convert dereddened magnitude to Fnu
    fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
    fnu_cgs = fnu_jy * 1.0e-23

    lam_cm = lam_um * 1.0e-4
    nu_hz = c_cgs / lam_cm

    nu_fnu = nu_hz * fnu_cgs

    # Error propagation from magnitude uncertainty
    if np.isfinite(emag) and 0 < emag < 5:
        frac_err = 0.4 * np.log(10.0) * emag
        nu_fnu_err = nu_fnu * frac_err
    else:
        nu_fnu_err = np.nan

    sed_rows.append({
        "band": band,
        "lambda_um": lam_um,
        "nu_hz": nu_hz,
        "mag_obs": mag,
        "A_lambda": A_lambda,
        "mag0": mag0,
        "mag_err": emag,
        "nu_fnu": nu_fnu,
        "nu_fnu_err": nu_fnu_err,
    })

sed = pd.DataFrame(sed_rows).sort_values("lambda_um").reset_index(drop=True)

if len(sed) == 0:
    raise ValueError("No usable BDBS photometry found for this fiber.")

print("\nDereddened photometry:")
print(sed[["band", "lambda_um", "mag_obs", "A_lambda", "mag0", "mag_err", "nu_fnu", "nu_fnu_err"]])


# ------------------------------------------------------------
# FIND NEAREST MARCS MODEL
# ------------------------------------------------------------

best_model, marcs_grid = find_nearest_marcs_model(
    MARCS_DIR,
    teff,
    logg,
    feh,
    alpha
)

print("\nNearest MARCS model:")
print(best_model[["filename", "geom", "teff", "logg", "feh", "alpha", "abund", "vturb", "distance"]])


# ------------------------------------------------------------
# READ MARCS WAVELENGTH GRID AND FLUX MODEL
# ------------------------------------------------------------

wave_file = find_wavelength_file(MARCS_DIR)

print("\nUsing MARCS wavelength file:")
print(wave_file)

print("\nUsing MARCS flux file:")
print(best_model["path"])

wave_A = read_numeric_file(wave_file)
flux_lambda = read_numeric_file(best_model["path"])

n = min(len(wave_A), len(flux_lambda))
wave_A = wave_A[:n]
flux_lambda = flux_lambda[:n]

good = (
    np.isfinite(wave_A)
    & np.isfinite(flux_lambda)
    & (wave_A > 0)
    & (flux_lambda > 0)
)

wave_A = wave_A[good]
flux_lambda = flux_lambda[good]

# MARCS .flx units:
# F_lambda = erg cm^-2 s^-1 Angstrom^-1 at stellar surface.
#
# Convert to nu Fnu:
# F_lambda per cm = F_lambda per Angstrom * 1e8
# F_nu = F_lambda * lambda^2 / c
# nu Fnu = nu * F_nu

wave_cm = wave_A * 1.0e-8
lambda_um = wave_A * 1.0e-4
nu_hz = c_cgs / wave_cm

flux_lambda_per_cm = flux_lambda * 1.0e8
flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
model_nu_fnu_surface = nu_hz * flux_nu

# Restrict model to photometry wavelength region
model_mask = (lambda_um >= 0.30) & (lambda_um <= 1.20)

lambda_um_model = lambda_um[model_mask]
model_nu_fnu_surface = model_nu_fnu_surface[model_mask]

idx = np.argsort(lambda_um_model)
lambda_um_model = lambda_um_model[idx]
model_nu_fnu_surface = model_nu_fnu_surface[idx]


# ------------------------------------------------------------
# FIT ONLY MODEL NORMALIZATION USING THE FULL MODEL
# ------------------------------------------------------------

obs_lam = sed["lambda_um"].to_numpy(dtype=float)
obs_y = sed["nu_fnu"].to_numpy(dtype=float)
obs_yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

obs_yerr_fit = obs_yerr.copy()

bad_err = ~np.isfinite(obs_yerr_fit) | (obs_yerr_fit <= 0)
obs_yerr_fit[bad_err] = 0.10 * obs_y[bad_err]

model_interp = np.interp(obs_lam, lambda_um_model, model_nu_fnu_surface)

good_fit = (
    np.isfinite(obs_y)
    & np.isfinite(obs_yerr_fit)
    & np.isfinite(model_interp)
    & (obs_y > 0)
    & (obs_yerr_fit > 0)
    & (model_interp > 0)
)

if np.sum(good_fit) < 2:
    raise ValueError("Not enough good photometry points to normalize the MARCS model.")

w = 1.0 / obs_yerr_fit[good_fit]**2

best_scale = np.sum(model_interp[good_fit] * obs_y[good_fit] * w) / np.sum(model_interp[good_fit]**2 * w)

scaled_model_nu_fnu = best_scale * model_nu_fnu_surface

model_at_points = best_scale * model_interp[good_fit]
chi2 = np.sum(((obs_y[good_fit] - model_at_points) / obs_yerr_fit[good_fit])**2)
dof = np.sum(good_fit) - 1

print(f"\nBest-fit normalization scale = {best_scale:.5e}")
print(f"chi2 = {chi2:.2f}")
print(f"dof = {dof}")
if dof > 0:
    print(f"reduced chi2 = {chi2/dof:.2f}")


# ------------------------------------------------------------
# SMOOTH / DERESOLVE THE MODEL FOR DISPLAY
# ------------------------------------------------------------

lambda_um_model_smooth, scaled_model_nu_fnu_smooth = bin_model_in_log_lambda(
    lambda_um_model,
    scaled_model_nu_fnu,
    n_bins=model_smoothing_bins,
    statistic="median"
)

# For an even cleaner continuum-like display, use this instead:
# statistic="p85"


# ------------------------------------------------------------
# AXIS LIMITS INCLUDING ERROR BARS AND SMOOTHED MODEL
# ------------------------------------------------------------

y_low = obs_y.copy()
y_high = obs_y.copy()

good_err = np.isfinite(obs_yerr) & (obs_yerr > 0)
y_low[good_err] = obs_y[good_err] - obs_yerr[good_err]
y_high[good_err] = obs_y[good_err] + obs_yerr[good_err]

positive_data_low = y_low[np.isfinite(y_low) & (y_low > 0)]
positive_data_high = y_high[np.isfinite(y_high) & (y_high > 0)]
positive_model = scaled_model_nu_fnu_smooth[
    np.isfinite(scaled_model_nu_fnu_smooth) & (scaled_model_nu_fnu_smooth > 0)
]

ymin = np.nanmin(np.concatenate([positive_data_low, positive_model]))
ymax = np.nanmax(np.concatenate([positive_data_high, positive_model]))

log_ymin = np.log10(ymin)
log_ymax = np.log10(ymax)

ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

ylim_low = 10.0 ** (log_ymin - ypad)
ylim_high = 10.0 ** (log_ymax + ypad)


# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.8, 5.6))

# Photometry points only. No connecting line.
ax.errorbar(
    sed["lambda_um"],
    sed["nu_fnu"],
    yerr=sed["nu_fnu_err"],
    fmt="o",
    markersize=6,
    capsize=3,
    elinewidth=1.0,
    markeredgewidth=0.8,
    linestyle="none",
    label=f"Fiber {fiber_to_plot} dereddened BDBS photometry"
)

for _, r in sed.iterrows():
    ax.annotate(
        r["band"],
        (r["lambda_um"], r["nu_fnu"]),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=9
    )

model_label = (
    "Nearest MARCS, smoothed "
    f"T={best_model['teff']:.0f} K, "
    f"logg={best_model['logg']:.1f}, "
    f"[Fe/H]={best_model['feh']:.2f}, "
    f"[α/Fe]={best_model['alpha']:.2f}"
)

ax.plot(
    lambda_um_model_smooth,
    scaled_model_nu_fnu_smooth,
    linewidth=1.4,
    alpha=0.9,
    label=model_label
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(0.32, 1.10)
ax.set_ylim(ylim_low, ylim_high)

ax.set_xlabel("Wavelength [micron]")
ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_title(f"Fiber {fiber_to_plot}: dereddened SED with smoothed MARCS model")

ax.grid(alpha=0.25, which="both")
ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.savefig(output_png, dpi=300)
plt.show()

print(f"\nSaved plot to:\n{output_png}")

In [ ]:
# ------------------------------------------------------------
# CMD + SP_Ace restricted MARCS photometric grid search
#
# This is the safer version to compare with the older unrestricted grid search.
#
# It:
#   1. Reads BDBS photometry/extinction
#   2. Reads SP_Ace average parameters
#   3. Computes dereddened CMD position: i0 and (u-i)0
#   4. Assigns a broad CMD evolutionary region
#   5. Restricts MARCS grid using CMD + SP_Ace priors
#   6. Runs photometric chi2 over the restricted grid
#   7. Plots the best restricted MARCS model
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

fiber_to_plot = 69

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

output_png = rf"/Users/kerrycheon/repos/SpectraReductions2026/space\SED_Fiber_{fiber_to_plot}_MARCS_CMD_SPprior_bestfit.png"
output_csv = rf"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_CMD_SPprior_grid_results_Fiber_{fiber_to_plot}.csv"

# Optional: exclude a bad band if needed
# Example: exclude_bands = ["r"]
exclude_bands = []

# Add systematic error floor because formal BDBS errors are too small
# for model-atmosphere/broadband comparison.
phot_error_floor_mag = 0.05

# MARCS displayed model smoothing
model_smoothing_bins = 240

# Search width around SP_Ace values.
# Low-SNR stars are automatically widened below.
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Set to None for full restricted-grid search.
# Use 200 for a quick test.
max_models_to_try = None

# Optional MARCS filters before CMD/SP_Ace cuts
# None = allow all
abundance_filter = None      # e.g. "st" or "ae"
geometry_filter = None       # e.g. "s" or "p"


# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# Top-hat windows are approximate and only for broadband SED fitting.
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(
            f"This file is HTML, not numeric MARCS data:\n{path}\n"
            "It was probably a failed download."
        )

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


# ------------------------------------------------------------
# MARCS HELPERS
# ------------------------------------------------------------

def parse_marcs_filename(path):
    name = os.path.basename(path)

    pattern = (
        r"^(?P<geom>[sp])"
        r"(?P<teff>\d{4,5})"
        r"_g(?P<logg>[+-]\d+\.\d)"
        r"_m(?P<mass>[0-9]+\.[0-9])"
        r"_t(?P<vturb>\d{2})"
        r"_(?P<abund>[a-z]{2})"
        r"_z(?P<feh>[+-]\d+\.\d{2})"
        r"_a(?P<alpha>[+-]\d+\.\d{2})"
    )

    m = re.search(pattern, name)

    if m is None:
        return None

    d = m.groupdict()

    return {
        "path": path,
        "filename": name,
        "geom": d["geom"],
        "teff": float(d["teff"]),
        "logg": float(d["logg"]),
        "mass": float(d["mass"]),
        "vturb": float(d["vturb"]) / 10.0,
        "abund": d["abund"],
        "feh": float(d["feh"]),
        "alpha": float(d["alpha"]),
    }


def load_marcs_grid(marcs_dir, abundance_filter=None, geometry_filter=None):
    flx_files = (
        glob.glob(os.path.join(marcs_dir, "**", "*.flx"), recursive=True)
        + glob.glob(os.path.join(marcs_dir, "**", "*.flx.gz"), recursive=True)
    )

    records = []

    for path in flx_files:
        info = parse_marcs_filename(path)
        if info is not None:
            records.append(info)

    if len(records) == 0:
        raise ValueError("No parseable MARCS .flx or .flx.gz files found.")

    grid = pd.DataFrame(records)

    if abundance_filter is not None:
        grid = grid[grid["abund"] == abundance_filter].copy()

    if geometry_filter is not None:
        grid = grid[grid["geom"] == geometry_filter].copy()

    if len(grid) == 0:
        raise ValueError("MARCS grid is empty after abundance/geometry filters.")

    return grid.reset_index(drop=True)


def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    """
    MARCS .flx:
        F_lambda in erg cm^-2 s^-1 Angstrom^-1 at stellar surface.

    Returns:
        lambda_um, nuFnu_surface
    """
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def synthetic_band_fluxes(lambda_um, model_nu_fnu, sed):
    """
    Approximate synthetic photometry from MARCS model using top-hat windows.
    """
    model_fluxes = []

    for _, r in sed.iterrows():
        lo = r["band_min_um"]
        hi = r["band_max_um"]
        lam_eff = r["lambda_um"]

        m = (
            np.isfinite(lambda_um)
            & np.isfinite(model_nu_fnu)
            & (lambda_um >= lo)
            & (lambda_um <= hi)
            & (model_nu_fnu > 0)
        )

        if np.sum(m) >= 5:
            val = np.nanmedian(model_nu_fnu[m])
        else:
            val = np.interp(lam_eff, lambda_um, model_nu_fnu)

        model_fluxes.append(val)

    return np.array(model_fluxes, dtype=float)


def fit_scale_and_chi2(model_band_flux, obs_flux, obs_err):
    good = (
        np.isfinite(model_band_flux)
        & np.isfinite(obs_flux)
        & np.isfinite(obs_err)
        & (model_band_flux > 0)
        & (obs_flux > 0)
        & (obs_err > 0)
    )

    if np.sum(good) < 2:
        return np.nan, np.nan, np.nan, np.sum(good)

    m = model_band_flux[good]
    y = obs_flux[good]
    s = obs_err[good]

    w = 1.0 / s**2

    scale = np.sum(m * y * w) / np.sum(m**2 * w)

    chi2 = np.sum(((y - scale * m) / s) ** 2)
    dof = np.sum(good) - 1
    red_chi2 = chi2 / dof if dof > 0 else np.nan

    return scale, chi2, red_chi2, np.sum(good)


def bin_model_in_log_lambda(lambda_um, flux, n_bins=120):
    """
    Smooth high-resolution MARCS model for plotting only.
    """
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


# ------------------------------------------------------------
# READ DATA
# ------------------------------------------------------------

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

obj = df[df["fiberid"] == fiber_to_plot].copy()

if len(obj) == 0:
    raise ValueError(f"No object found with fiberid = {fiber_to_plot}")

row = obj.iloc[0]

teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
    raise ValueError(
        f"Fiber {fiber_to_plot} has missing SP_Ace values:\n"
        f"Teff={teff_sp}, logg={logg_sp}, [Fe/H]={feh_sp}"
    )

print(f"Fiber {fiber_to_plot}")
print(f"SP_Ace: Teff={teff_sp:.1f}, logg={logg_sp:.3f}, [Fe/H]={feh_sp:.3f}, [alpha/Fe]={afe_sp:.3f}")

if "RV" in row.index:
    print(f"RV = {row['RV']}")
if "S/N" in row.index:
    print(f"S/N = {row['S/N']}")


# ------------------------------------------------------------
# CMD POSITION AND CMD PRIOR
# ------------------------------------------------------------

u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
ui0 = u0 - i0

print("\nCMD position:")
print(f"u0     = {u0:.3f}")
print(f"i0     = {i0:.3f}")
print(f"(u-i)0 = {ui0:.3f}")

# Broad CMD/evolutionary priors
if (ui0 >= 3.8) and (i0 <= 14.5):
    cmd_region = "bright RGB/AGB"
    teff_min, teff_max = 3300, 4800
    logg_min, logg_max = -0.5, 2.5

elif (ui0 >= 3.0):
    cmd_region = "RGB"
    teff_min, teff_max = 3500, 5200
    logg_min, logg_max = 0.0, 3.2

elif (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
    cmd_region = "red clump / red HB / lower RGB"
    teff_min, teff_max = 4000, 6000
    logg_min, logg_max = 1.0, 3.8

elif (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
    cmd_region = "blue HB / warmer star"
    teff_min, teff_max = 5500, 8500
    logg_min, logg_max = 2.0, 4.8

else:
    cmd_region = "broad fallback"
    teff_min, teff_max = 3300, 7000
    logg_min, logg_max = -0.5, 4.5

print(f"\nCMD region assigned: {cmd_region}")
print(f"CMD Teff range: {teff_min} to {teff_max} K")
print(f"CMD logg range: {logg_min} to {logg_max}")


# ------------------------------------------------------------
# BUILD DEREDDENED PHOTOMETRIC SED
# ------------------------------------------------------------

sed_rows = []

for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

    if band in exclude_bands:
        continue

    mag = pd.to_numeric(row[mag_col], errors="coerce")
    emag_formal = pd.to_numeric(row[err_col], errors="coerce")
    A_lambda = pd.to_numeric(row[A_col], errors="coerce")

    if not np.isfinite(mag):
        continue
    if mag > 90 or mag < -90:
        continue

    if not np.isfinite(emag_formal) or emag_formal <= 0:
        emag_formal = np.nan

    if not np.isfinite(A_lambda):
        A_lambda = 0.0

    mag0 = mag - A_lambda

    # Fit error includes floor
    if np.isfinite(emag_formal):
        emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
    else:
        emag_fit = phot_error_floor_mag

    fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
    fnu_cgs = fnu_jy * 1.0e-23

    lam_cm = lam_eff * 1.0e-4
    nu_hz = c_cgs / lam_cm
    nu_fnu = nu_hz * fnu_cgs

    frac_err_fit = 0.4 * np.log(10.0) * emag_fit
    nu_fnu_err = nu_fnu * frac_err_fit

    sed_rows.append({
        "band": band,
        "lambda_um": lam_eff,
        "band_min_um": band_min,
        "band_max_um": band_max,
        "mag_obs": mag,
        "A_lambda": A_lambda,
        "mag0": mag0,
        "mag_err_formal": emag_formal,
        "mag_err_fit": emag_fit,
        "nu_fnu": nu_fnu,
        "nu_fnu_err": nu_fnu_err,
    })

sed = pd.DataFrame(sed_rows).sort_values("lambda_um").reset_index(drop=True)

if len(sed) < 2:
    raise ValueError("Not enough usable photometry points.")

print("\nDereddened photometry used:")
print(sed[["band", "mag_obs", "A_lambda", "mag0", "mag_err_fit", "nu_fnu", "nu_fnu_err"]])


# ------------------------------------------------------------
# LOAD MARCS GRID AND APPLY CMD + SP_ACE PRIORS
# ------------------------------------------------------------

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

grid = load_marcs_grid(
    MARCS_DIR,
    abundance_filter=abundance_filter,
    geometry_filter=geometry_filter
)

n_before = len(grid)

# Widen priors for lower S/N
teff_half_width = teff_half_width_default
logg_half_width = logg_half_width_default
feh_half_width = feh_half_width_default
alpha_half_width = alpha_half_width_default

snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

if np.isfinite(snr_here) and snr_here < 25:
    teff_half_width = 800
    logg_half_width = 1.25
    feh_half_width = 0.60
    alpha_half_width = 0.50

# Combine CMD box with SP_Ace neighborhood
teff_low = max(teff_min, teff_sp - teff_half_width)
teff_high = min(teff_max, teff_sp + teff_half_width)

logg_low = max(logg_min, logg_sp - logg_half_width)
logg_high = min(logg_max, logg_sp + logg_half_width)

feh_low = feh_sp - feh_half_width
feh_high = feh_sp + feh_half_width

alpha_low = afe_sp - alpha_half_width
alpha_high = afe_sp + alpha_half_width

print("\nFinal MARCS search box:")
print(f"Teff:       {teff_low:.0f} to {teff_high:.0f} K")
print(f"logg:       {logg_low:.2f} to {logg_high:.2f}")
print(f"[Fe/H]:     {feh_low:.2f} to {feh_high:.2f}")
print(f"[alpha/Fe]: {alpha_low:.2f} to {alpha_high:.2f}")

grid = grid[
    (grid["teff"] >= teff_low)
    & (grid["teff"] <= teff_high)
    & (grid["logg"] >= logg_low)
    & (grid["logg"] <= logg_high)
    & (grid["feh"] >= feh_low)
    & (grid["feh"] <= feh_high)
].copy()

if np.isfinite(afe_sp):
    grid = grid[
        (grid["alpha"] >= alpha_low)
        & (grid["alpha"] <= alpha_high)
    ].copy()

# Prefer spherical for giants; plane-parallel for high-gravity stars
preferred_geom = "s" if logg_sp <= 3.5 else "p"

sub = grid[grid["geom"] == preferred_geom].copy()
if len(sub) > 0:
    grid = sub
    print(f"\nUsing preferred MARCS geometry: {preferred_geom}")

n_after = len(grid)

print(f"\nMARCS models before CMD/SP_Ace cuts: {n_before}")
print(f"MARCS models after CMD/SP_Ace cuts:  {n_after}")

if n_after == 0:
    raise ValueError(
        "The MARCS grid is empty after CMD/SP_Ace cuts. "
        "Widen the prior half-widths near the top of the cell."
    )

if max_models_to_try is not None:
    grid = grid.iloc[:max_models_to_try].copy()
    print(f"Testing first {len(grid)} models only.")

grid = grid.reset_index(drop=True)


# ------------------------------------------------------------
# GRID SEARCH OVER RESTRICTED MARCS MODELS
# ------------------------------------------------------------

obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)

results = []

print("\nStarting restricted MARCS photometric grid search...")

for i, model_row in grid.iterrows():

    if (i % 50) == 0:
        print(f"Testing model {i+1} / {len(grid)}", flush=True)

    try:
        flux_lambda = read_numeric_file(model_row["path"])

        n = min(len(wave_A), len(flux_lambda))
        lam_um, nu_fnu_surface = marcs_flux_to_nuFnu(wave_A[:n], flux_lambda[:n])

        mrange = (lam_um >= 0.30) & (lam_um <= 1.12)
        lam_um = lam_um[mrange]
        nu_fnu_surface = nu_fnu_surface[mrange]

        model_band_flux = synthetic_band_fluxes(lam_um, nu_fnu_surface, sed)

        scale, chi2, red_chi2, nfit = fit_scale_and_chi2(
            model_band_flux,
            obs_flux,
            obs_err
        )

        results.append({
            "filename": model_row["filename"],
            "path": model_row["path"],
            "geom": model_row["geom"],
            "teff": model_row["teff"],
            "logg": model_row["logg"],
            "feh": model_row["feh"],
            "alpha": model_row["alpha"],
            "abund": model_row["abund"],
            "vturb": model_row["vturb"],
            "scale": scale,
            "chi2": chi2,
            "red_chi2": red_chi2,
            "nfit": nfit,
            "cmd_region": cmd_region,
            "u0": u0,
            "i0": i0,
            "ui0": ui0,
            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,
            "dTeff_from_SP": model_row["teff"] - teff_sp,
            "dlogg_from_SP": model_row["logg"] - logg_sp,
            "dfeh_from_SP": model_row["feh"] - feh_sp,
            "dalpha_from_SP": model_row["alpha"] - afe_sp,
        })

    except Exception as e:
        results.append({
            "filename": model_row["filename"],
            "path": model_row["path"],
            "geom": model_row["geom"],
            "teff": model_row["teff"],
            "logg": model_row["logg"],
            "feh": model_row["feh"],
            "alpha": model_row["alpha"],
            "abund": model_row["abund"],
            "vturb": model_row["vturb"],
            "scale": np.nan,
            "chi2": np.nan,
            "red_chi2": np.nan,
            "nfit": 0,
            "error": str(e),
        })

results_df = pd.DataFrame(results)
results_df = results_df[np.isfinite(results_df["chi2"])].copy()
results_df = results_df.sort_values("chi2").reset_index(drop=True)

if len(results_df) == 0:
    raise ValueError("No valid MARCS fits were produced.")

results_df.to_csv(output_csv, index=False)

print("\nTop 10 restricted CMD/SP_Ace MARCS fits:")
print(results_df.head(10)[[
    "teff", "logg", "feh", "alpha", "geom", "abund",
    "chi2", "red_chi2", "scale",
    "dTeff_from_SP", "dlogg_from_SP", "dfeh_from_SP", "dalpha_from_SP",
    "filename"
]])

print(f"\nSaved restricted grid-search results to:\n{output_csv}")


# ------------------------------------------------------------
# PLOT BEST RESTRICTED MODEL
# ------------------------------------------------------------

best = results_df.iloc[0]

print("\nBest restricted photometric MARCS model:")
print(best[[
    "teff", "logg", "feh", "alpha", "geom", "abund",
    "chi2", "red_chi2", "scale", "filename"
]])

flux_lambda_best = read_numeric_file(best["path"])

n = min(len(wave_A), len(flux_lambda_best))
lam_um_best, nu_fnu_surface_best = marcs_flux_to_nuFnu(wave_A[:n], flux_lambda_best[:n])

mrange = (lam_um_best >= 0.30) & (lam_um_best <= 1.20)
lam_um_best = lam_um_best[mrange]
nu_fnu_surface_best = nu_fnu_surface_best[mrange]

scaled_best = best["scale"] * nu_fnu_surface_best

lam_smooth, scaled_smooth = bin_model_in_log_lambda(
    lam_um_best,
    scaled_best,
    n_bins=model_smoothing_bins
)

# Axis limits
y = sed["nu_fnu"].to_numpy(dtype=float)
yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

y_low = y - yerr
y_high = y + yerr

positive_data = np.concatenate([
    y_low[np.isfinite(y_low) & (y_low > 0)],
    y_high[np.isfinite(y_high) & (y_high > 0)]
])

positive_model = scaled_smooth[np.isfinite(scaled_smooth) & (scaled_smooth > 0)]

ymin = np.nanmin(np.concatenate([positive_data, positive_model]))
ymax = np.nanmax(np.concatenate([positive_data, positive_model]))

log_ymin = np.log10(ymin)
log_ymax = np.log10(ymax)
ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

ylim_low = 10.0 ** (log_ymin - ypad)
ylim_high = 10.0 ** (log_ymax + ypad)

fig, ax = plt.subplots(figsize=(7.8, 5.6))

ax.errorbar(
    sed["lambda_um"],
    sed["nu_fnu"],
    yerr=sed["nu_fnu_err"],
    fmt="o",
    markersize=6,
    capsize=3,
    elinewidth=1.0,
    markeredgewidth=0.8,
    linestyle="none",
    label=f"Fiber {fiber_to_plot} dereddened BDBS photometry"
)

for _, r in sed.iterrows():
    ax.annotate(
        r["band"],
        (r["lambda_um"], r["nu_fnu"]),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=9
    )

model_label = (
    "Best CMD/SP_Ace-restricted MARCS "
    f"T={best['teff']:.0f} K, "
    f"logg={best['logg']:.1f}, "
    f"[Fe/H]={best['feh']:.2f}, "
    f"[α/Fe]={best['alpha']:.2f}, "
    f"χ²ν={best['red_chi2']:.2f}"
)

ax.plot(
    lam_smooth,
    scaled_smooth,
    linewidth=1.6,
    alpha=0.95,
    label=model_label
)

# Also mark the SP_Ace solution in the text box
textstr = (
    f"CMD region: {cmd_region}\n"
    f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
    f"Best: T={best['teff']:.0f} K, logg={best['logg']:.1f}, [Fe/H]={best['feh']:.2f}, [α/Fe]={best['alpha']:.2f}"
)

ax.text(
    0.03,
    0.04,
    textstr,
    transform=ax.transAxes,
    fontsize=8,
    va="bottom",
    ha="left",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(0.32, 1.10)
ax.set_ylim(ylim_low, ylim_high)

ax.set_xlabel("Wavelength [micron]")
ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_title(f"Fiber {fiber_to_plot}: CMD/SP_Ace-restricted MARCS SED fit")

ax.grid(alpha=0.25, which="both")
ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.savefig(output_png, dpi=300)
plt.show()

print(f"\nSaved plot to:\n{output_png}")

In [ ]:
# ------------------------------------------------------------
# BUILD MARCS SYNTHETIC ugrizy CACHE
#
# Run this once.
#
# It reads every MARCS .flx or .flx.gz file, computes approximate
# top-hat synthetic nuFnu fluxes in BDBS-like ugrizy bands,
# and saves the result to:
#
#   /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS/MARCS_synthetic_ugrizy_cache.csv
#
# After this, each fiber fit can use the cache instead of rereading
# every MARCS model.
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

cache_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS\MARCS_synthetic_ugrizy_cache.csv"

overwrite_cache = False   # set True if you want to rebuild from scratch

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# Approximate top-hat BDBS-like windows
# ------------------------------------------------------------

bands = [
    # band, lambda_eff_um, band_min_um, band_max_um
    ("u", 0.36, 0.32, 0.40),
    ("g", 0.48, 0.40, 0.55),
    ("r", 0.625, 0.55, 0.70),
    ("i", 0.77, 0.70, 0.85),
    ("z", 0.91, 0.85, 0.96),
    ("y", 1.00, 0.96, 1.08),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    """
    Robust numeric reader for:
      - real gzip files
      - plain text files
      - plain text files mislabeled .gz
    """
    if looks_like_html(path):
        raise ValueError(
            f"This file is HTML, not numeric MARCS data:\n{path}\n"
            "It was probably a failed download."
        )

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


# ------------------------------------------------------------
# MARCS FILENAME PARSER
# ------------------------------------------------------------

def parse_marcs_filename(path):
    """
    Example:
    s4750_g+3.0_m1.0_t02_st_z-0.75_a+0.30_c+0.00_n+0.00_o+0.30_r+0.00_s+0.00.flx.gz
    """
    name = os.path.basename(path)

    pattern = (
        r"^(?P<geom>[sp])"
        r"(?P<teff>\d{4,5})"
        r"_g(?P<logg>[+-]\d+\.\d)"
        r"_m(?P<mass>[0-9]+\.[0-9])"
        r"_t(?P<vturb>\d{2})"
        r"_(?P<abund>[a-z]{2})"
        r"_z(?P<feh>[+-]\d+\.\d{2})"
        r"_a(?P<alpha>[+-]\d+\.\d{2})"
    )

    m = re.search(pattern, name)

    if m is None:
        return None

    d = m.groupdict()

    return {
        "path": path,
        "filename": name,
        "geom": d["geom"],
        "teff": float(d["teff"]),
        "logg": float(d["logg"]),
        "mass": float(d["mass"]),
        "vturb": float(d["vturb"]) / 10.0,
        "abund": d["abund"],
        "feh": float(d["feh"]),
        "alpha": float(d["alpha"]),
    }


def load_marcs_file_list(marcs_dir):
    flx_files = (
        glob.glob(os.path.join(marcs_dir, "**", "*.flx"), recursive=True)
        + glob.glob(os.path.join(marcs_dir, "**", "*.flx.gz"), recursive=True)
    )

    records = []

    for path in flx_files:
        info = parse_marcs_filename(path)
        if info is not None:
            records.append(info)

    if len(records) == 0:
        raise ValueError("No parseable MARCS .flx or .flx.gz files found.")

    return pd.DataFrame(records).reset_index(drop=True)


# ------------------------------------------------------------
# FLUX CONVERSION AND SYNTHETIC BANDS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    """
    MARCS .flx:
        F_lambda in erg cm^-2 s^-1 Angstrom^-1 at stellar surface.

    Returns:
        lambda_um, nuFnu_surface
    """
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def synthetic_band_flux(lambda_um, nu_fnu, band_min_um, band_max_um, lambda_eff_um):
    """
    Approximate synthetic broadband nuFnu flux using a top-hat band.
    Median suppresses narrow absorption-line structure.
    """
    m = (
        np.isfinite(lambda_um)
        & np.isfinite(nu_fnu)
        & (lambda_um >= band_min_um)
        & (lambda_um <= band_max_um)
        & (nu_fnu > 0)
    )

    if np.sum(m) >= 5:
        return np.nanmedian(nu_fnu[m])

    return np.interp(lambda_eff_um, lambda_um, nu_fnu)


# ------------------------------------------------------------
# BUILD CACHE
# ------------------------------------------------------------

if os.path.exists(cache_file) and not overwrite_cache:
    print("Cache already exists. Not rebuilding.")
    print(cache_file)

else:
    wave_file = find_wavelength_file(MARCS_DIR)
    print("Using wavelength file:")
    print(wave_file)

    wave_A = read_numeric_file(wave_file)

    grid = load_marcs_file_list(MARCS_DIR)

    print(f"\nFound {len(grid)} MARCS flux files.")
    print("Building synthetic ugrizy cache. This may take a while...")

    rows = []

    for j, model in grid.iterrows():

        if (j % 100) == 0:
            print(f"Processing model {j+1} / {len(grid)}", flush=True)

        try:
            flux_lambda = read_numeric_file(model["path"])

            n = min(len(wave_A), len(flux_lambda))
            lam_um, nu_fnu_surface = marcs_flux_to_nuFnu(
                wave_A[:n],
                flux_lambda[:n]
            )

            # Restrict for speed
            use = (lam_um >= 0.30) & (lam_um <= 1.12)
            lam_um = lam_um[use]
            nu_fnu_surface = nu_fnu_surface[use]

            row = {
                "filename": model["filename"],
                "path": model["path"],
                "geom": model["geom"],
                "teff": model["teff"],
                "logg": model["logg"],
                "mass": model["mass"],
                "vturb": model["vturb"],
                "abund": model["abund"],
                "feh": model["feh"],
                "alpha": model["alpha"],
            }

            for band, lam_eff, band_min, band_max in bands:
                row[f"model_{band}"] = synthetic_band_flux(
                    lam_um,
                    nu_fnu_surface,
                    band_min,
                    band_max,
                    lam_eff
                )

            rows.append(row)

        except Exception as e:
            print(f"Skipping model because of error: {model['filename']}")
            print(e)

    cache = pd.DataFrame(rows)

    cache.to_csv(cache_file, index=False)

    print("\nFinished building MARCS synthetic photometry cache.")
    print(f"Saved {len(cache)} models to:")
    print(cache_file)

    print("\nFirst few cached models:")
    print(cache.head())

In [ ]:
# ------------------------------------------------------------
# FULL RESTRICTED BATCH MARCS SED FITTING CELL
#
# Runs all fiberid values in the photometry file.
#
# Requires:
#   1. /Users/kerrycheon/repos/SpectraReductions2026/space/CaT_summary_BDBS_XCSAO.csv
#   2. /Users/kerrycheon/repos/SpectraReductions2026/space/SP_Ace_averages_with_FeHDP_avg_std.csv
#   3. /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS/MARCS_synthetic_ugrizy_cache.csv
#   4. MARCS .flx/.flx.gz files for plotting the best model curves
#
# Outputs:
#   1. Master table:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_batch_restricted_fit_summary.csv
#   2. One PNG per modeled fiber in:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_SED_PLOTS_RESTRICTED
# ------------------------------------------------------------

import os
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# LOAD CACHED MARCS SYNTHETIC PHOTOMETRY GRID
# ------------------------------------------------------------


MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))
cache_file = os.path.join(MARCS_DIR, "MARCS_synthetic_ugrizy_cache.csv")

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "The MARCS synthetic photometry cache does not exist yet.\n"
        "Run the cache-building cell first.\n\n"
        f"Expected file:\n{cache_file}"
    )

marcs_cache = pd.read_csv(cache_file)

print("Loaded cached MARCS synthetic photometry grid:")
print(f"N models = {len(marcs_cache)}")
print(marcs_cache.head())
# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))

cache_file = os.path.join(MARCS_DIR, "MARCS_synthetic_ugrizy_cache.csv")

output_dir = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_SED_PLOTS_RESTRICTED"
os.makedirs(output_dir, exist_ok=True)

master_output_csv = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_batch_restricted_fit_summary.csv"

# For testing, set this to e.g. 10.
# For the full run, set to None.
max_fibers_to_run = None

# Make one PNG per successfully modeled fiber.
make_plots = True

# If True, do not remake PNG files that already exist.
skip_existing_png = False

# Global bad-band exclusion, if needed.
# Example: exclude_bands_global = ["r"]
exclude_bands_global = []

# Formal BDBS errors are often too small for model/photometry comparison.
phot_error_floor_mag = 0.05

# Smoothing for plotted MARCS curve. Smaller = smoother.
model_smoothing_bins = 1800

# Strict CMD + SP_Ace prior widths.
# These are widened automatically for low-S/N spectra.
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Low-S/N widening
low_snr_threshold = 25
teff_half_width_lowsnr = 800
logg_half_width_lowsnr = 1.25
feh_half_width_lowsnr = 0.60
alpha_half_width_lowsnr = 0.50

# Optional cache filters before CMD/SP_Ace restrictions.
# Usually keep both as None.
abundance_filter = None     # None, "st", or "ae"
geometry_filter = None      # None, "s", or "p"

# If a fiber has zero models after cuts, set this True to retry with wider cuts.
# For a clean restricted run, keep False.
allow_fallback_wider_box = False

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not numeric MARCS data:\n{path}")

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


def resolve_model_path(path_from_cache, filename, marcs_dir):
    """
    Use the cached full path if it exists.
    If not, search by filename in MARCS_DIR.
    """
    if isinstance(path_from_cache, str) and os.path.exists(path_from_cache):
        return path_from_cache

    matches = glob.glob(
        os.path.join(marcs_dir, "**", filename),
        recursive=True
    )

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find MARCS model file:\n{filename}\nCached path was:\n{path_from_cache}"
    )


# ------------------------------------------------------------
# MODEL/SED HELPERS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    """
    Convert MARCS F_lambda to nu Fnu.

    MARCS .flx units:
        F_lambda = erg cm^-2 s^-1 Angstrom^-1 at stellar surface
    """
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def bin_model_in_log_lambda(lambda_um, flux, n_bins=120):
    """
    Smooth high-resolution MARCS model for plotting only.
    """
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    if len(lambda_um) < 5:
        return lambda_um, flux

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


def build_dereddened_sed(row, exclude_bands=None):
    """
    Build dereddened observed SED from BDBS ugrizy photometry.
    """
    if exclude_bands is None:
        exclude_bands = []

    sed_rows = []

    for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

        if band in exclude_bands:
            continue

        if mag_col not in row.index:
            continue

        mag = pd.to_numeric(row[mag_col], errors="coerce")
        emag_formal = pd.to_numeric(row[err_col], errors="coerce") if err_col in row.index else np.nan
        A_lambda = pd.to_numeric(row[A_col], errors="coerce") if A_col in row.index else 0.0

        if not np.isfinite(mag):
            continue
        if mag > 90 or mag < -90:
            continue

        if not np.isfinite(A_lambda):
            A_lambda = 0.0

        mag0 = mag - A_lambda

        if np.isfinite(emag_formal) and emag_formal > 0:
            emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
        else:
            emag_fit = phot_error_floor_mag

        fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
        fnu_cgs = fnu_jy * 1.0e-23

        lam_cm = lam_eff * 1.0e-4
        nu_hz = c_cgs / lam_cm
        nu_fnu = nu_hz * fnu_cgs

        frac_err = 0.4 * np.log(10.0) * emag_fit
        nu_fnu_err = nu_fnu * frac_err

        sed_rows.append({
            "band": band,
            "lambda_um": lam_eff,
            "mag_obs": mag,
            "A_lambda": A_lambda,
            "mag0": mag0,
            "mag_err_fit": emag_fit,
            "nu_fnu": nu_fnu,
            "nu_fnu_err": nu_fnu_err,
            "model_col": f"model_{band}",
        })

    sed = pd.DataFrame(sed_rows)

    if len(sed) > 0:
        sed = sed.sort_values("lambda_um").reset_index(drop=True)

    return sed


def get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Assign CMD region and combine CMD priors with SP_Ace local prior.
    """

    u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
    i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
    ui0 = u0 - i0

    # Broad CMD/evolutionary priors
    if np.isfinite(ui0) and np.isfinite(i0) and (ui0 >= 3.8) and (i0 <= 14.5):
        cmd_region = "bright RGB/AGB"
        teff_min, teff_max = 3300, 4800
        logg_min, logg_max = -0.5, 2.5

    elif np.isfinite(ui0) and (ui0 >= 3.0):
        cmd_region = "RGB"
        teff_min, teff_max = 3500, 5200
        logg_min, logg_max = 0.0, 3.2

    elif np.isfinite(ui0) and np.isfinite(i0) and (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
        cmd_region = "red clump / red HB / lower RGB"
        teff_min, teff_max = 4000, 6000
        logg_min, logg_max = 1.0, 3.8

    elif np.isfinite(ui0) and np.isfinite(i0) and (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
        cmd_region = "blue HB / warmer star"
        teff_min, teff_max = 5500, 8500
        logg_min, logg_max = 2.0, 4.8

    else:
        cmd_region = "broad fallback"
        teff_min, teff_max = 3300, 7000
        logg_min, logg_max = -0.5, 4.5

    teff_half_width = teff_half_width_default
    logg_half_width = logg_half_width_default
    feh_half_width = feh_half_width_default
    alpha_half_width = alpha_half_width_default

    snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

    if np.isfinite(snr_here) and snr_here < low_snr_threshold:
        teff_half_width = teff_half_width_lowsnr
        logg_half_width = logg_half_width_lowsnr
        feh_half_width = feh_half_width_lowsnr
        alpha_half_width = alpha_half_width_lowsnr

    teff_low = max(teff_min, teff_sp - teff_half_width)
    teff_high = min(teff_max, teff_sp + teff_half_width)

    logg_low = max(logg_min, logg_sp - logg_half_width)
    logg_high = min(logg_max, logg_sp + logg_half_width)

    feh_low = feh_sp - feh_half_width
    feh_high = feh_sp + feh_half_width

    alpha_low = afe_sp - alpha_half_width
    alpha_high = afe_sp + alpha_half_width

    return {
        "u0": u0,
        "i0": i0,
        "ui0": ui0,
        "cmd_region": cmd_region,

        "teff_low": teff_low,
        "teff_high": teff_high,
        "logg_low": logg_low,
        "logg_high": logg_high,
        "feh_low": feh_low,
        "feh_high": feh_high,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,

        "teff_min_cmd": teff_min,
        "teff_max_cmd": teff_max,
        "logg_min_cmd": logg_min,
        "logg_max_cmd": logg_max,

        "teff_half_width": teff_half_width,
        "logg_half_width": logg_half_width,
        "feh_half_width": feh_half_width,
        "alpha_half_width": alpha_half_width,
    }


def restrict_grid_for_fiber(grid_base, row, teff_sp, logg_sp, feh_sp, afe_sp, cmd_info):
    """
    Apply strict CMD + SP_Ace restrictions to cached MARCS grid.
    """

    grid = grid_base.copy()

    grid = grid[
        (grid["teff"] >= cmd_info["teff_low"])
        & (grid["teff"] <= cmd_info["teff_high"])
        & (grid["logg"] >= cmd_info["logg_low"])
        & (grid["logg"] <= cmd_info["logg_high"])
        & (grid["feh"] >= cmd_info["feh_low"])
        & (grid["feh"] <= cmd_info["feh_high"])
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= cmd_info["alpha_low"])
            & (grid["alpha"] <= cmd_info["alpha_high"])
        ].copy()

    # Prefer spherical for giants, plane-parallel for high-gravity stars
    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    return grid.reset_index(drop=True), used_geom


def restrict_grid_fallback(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Optional wider fallback if the strict restricted grid is empty.
    """
    grid = grid_base[
        (grid_base["teff"] >= teff_sp - 1000)
        & (grid_base["teff"] <= teff_sp + 1000)
        & (grid_base["logg"] >= logg_sp - 1.5)
        & (grid_base["logg"] <= logg_sp + 1.5)
        & (grid_base["feh"] >= feh_sp - 0.75)
        & (grid_base["feh"] <= feh_sp + 0.75)
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= afe_sp - 0.6)
            & (grid["alpha"] <= afe_sp + 0.6)
        ].copy()

    return grid.reset_index(drop=True)


def vectorized_cached_fit(grid, sed):
    """
    Fit all models in restricted cached grid at once.
    """
    obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)
    model_cols = sed["model_col"].tolist()

    missing_cols = [c for c in model_cols if c not in grid.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Cache is missing model columns: {missing_cols}")

    M = grid[model_cols].to_numpy(dtype=float)

    y = obs_flux.astype(float)
    s = obs_err.astype(float)

    good_band = (
        np.isfinite(y)
        & np.isfinite(s)
        & (y > 0)
        & (s > 0)
    )

    if np.sum(good_band) < 2:
        raise ValueError("Not enough valid observed bands for SED fit.")

    M = M[:, good_band]
    y = y[good_band]
    s = s[good_band]

    good_model = np.all(np.isfinite(M) & (M > 0), axis=1)

    grid_fit = grid.loc[good_model].copy().reset_index(drop=True)
    M = M[good_model, :]

    if len(grid_fit) == 0:
        raise ValueError("No valid model rows after checking cached model fluxes.")

    w = 1.0 / s**2

    numerator = np.sum(M * y * w, axis=1)
    denominator = np.sum(M**2 * w, axis=1)

    scale = numerator / denominator

    resid = y[None, :] - scale[:, None] * M
    chi2 = np.sum((resid / s[None, :])**2, axis=1)

    nfit = int(np.sum(good_band))
    dof = nfit - 1

    if dof > 0:
        red_chi2 = chi2 / dof
    else:
        red_chi2 = np.full_like(chi2, np.nan)

    results_df = grid_fit.copy()
    results_df["scale"] = scale
    results_df["chi2"] = chi2
    results_df["red_chi2"] = red_chi2
    results_df["nfit"] = nfit

    results_df = results_df[np.isfinite(results_df["chi2"])].copy()
    results_df = results_df.sort_values("chi2").reset_index(drop=True)

    if len(results_df) == 0:
        raise ValueError("No valid cached fits produced.")

    return results_df


def plot_best_model_for_fiber(
    fiberid,
    row,
    sed,
    best,
    cmd_info,
    wave_A,
    model_curve_cache,
    output_png
):
    """
    Save one PNG showing dereddened photometry and the best MARCS curve.
    """

    if skip_existing_png and os.path.exists(output_png):
        return output_png

    best_path = resolve_model_path(best["path"], best["filename"], MARCS_DIR)

    # Cache unscaled smoothed model curves to avoid rereading repeats
    if best_path in model_curve_cache:
        lam_smooth_unscaled, smooth_unscaled = model_curve_cache[best_path]
    else:
        flux_lambda_best = read_numeric_file(best_path)

        n = min(len(wave_A), len(flux_lambda_best))

        lam_um_best, nu_fnu_surface_best = marcs_flux_to_nuFnu(
            wave_A[:n],
            flux_lambda_best[:n]
        )

        mrange = (lam_um_best >= 0.30) & (lam_um_best <= 1.20)
        lam_um_best = lam_um_best[mrange]
        nu_fnu_surface_best = nu_fnu_surface_best[mrange]

        lam_smooth_unscaled, smooth_unscaled = bin_model_in_log_lambda(
            lam_um_best,
            nu_fnu_surface_best,
            n_bins=model_smoothing_bins
        )

        model_curve_cache[best_path] = (lam_smooth_unscaled, smooth_unscaled)

    scaled_smooth = best["scale"] * smooth_unscaled

    y = sed["nu_fnu"].to_numpy(dtype=float)
    yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

    y_low = y - yerr
    y_high = y + yerr

    positive_data = np.concatenate([
        y_low[np.isfinite(y_low) & (y_low > 0)],
        y_high[np.isfinite(y_high) & (y_high > 0)]
    ])

    positive_model = scaled_smooth[np.isfinite(scaled_smooth) & (scaled_smooth > 0)]

    ymin = np.nanmin(np.concatenate([positive_data, positive_model]))
    ymax = np.nanmax(np.concatenate([positive_data, positive_model]))

    log_ymin = np.log10(ymin)
    log_ymax = np.log10(ymax)

    ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

    ylim_low = 10.0 ** (log_ymin - ypad)
    ylim_high = 10.0 ** (log_ymax + ypad)

    teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
    logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
    feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
    afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

    fig, ax = plt.subplots(figsize=(7.8, 5.6))

    ax.errorbar(
        sed["lambda_um"],
        sed["nu_fnu"],
        yerr=sed["nu_fnu_err"],
        fmt="o",
        markersize=6,
        capsize=3,
        elinewidth=1.0,
        markeredgewidth=0.8,
        linestyle="none",
        label=f"Fiber {fiberid} dereddened BDBS photometry"
    )

    for _, r in sed.iterrows():
        ax.annotate(
            r["band"],
            (r["lambda_um"], r["nu_fnu"]),
            textcoords="offset points",
            xytext=(0, 7),
            ha="center",
            fontsize=9
        )

    model_label = (
        "Restricted cached MARCS "
        f"T={best['teff']:.0f} K, "
        f"logg={best['logg']:.1f}, "
        f"[Fe/H]={best['feh']:.2f}, "
        f"[α/Fe]={best['alpha']:.2f}, "
        f"χ²ν={best['red_chi2']:.2f}"
    )

    ax.plot(
        lam_smooth_unscaled,
        scaled_smooth,
        linewidth=1.6,
        alpha=0.95,
        label=model_label
    )

    textstr = (
        f"CMD region: {cmd_info['cmd_region']}\n"
        f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
        f"Best: T={best['teff']:.0f} K, logg={best['logg']:.1f}, [Fe/H]={best['feh']:.2f}, [α/Fe]={best['alpha']:.2f}"
    )

    ax.text(
        0.03,
        0.04,
        textstr,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(0.32, 1.10)
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_xlabel("Wavelength [micron]")
    ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_title(f"Fiber {fiberid}: restricted cached MARCS SED fit")

    ax.grid(alpha=0.25, which="both")
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)

    return output_png


# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "Cache file does not exist. Run the MARCS synthetic cache-building cell first:\n"
        f"{cache_file}"
    )

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)
cache = pd.read_csv(cache_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

df = df[np.isfinite(df["fiberid"])].copy()
df["fiberid"] = df["fiberid"].astype(int)

df = df.drop_duplicates(subset=["fiberid"]).sort_values("fiberid").reset_index(drop=True)

if max_fibers_to_run is not None:
    df = df.iloc[:max_fibers_to_run].copy()

grid_base = cache.copy()

if abundance_filter is not None:
    grid_base = grid_base[grid_base["abund"] == abundance_filter].copy()

if geometry_filter is not None:
    grid_base = grid_base[grid_base["geom"] == geometry_filter].copy()

if len(grid_base) == 0:
    raise ValueError("Cached MARCS grid is empty after abundance/geometry filters.")

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

print(f"Number of fibers to process: {len(df)}")
print(f"Number of cached MARCS models before per-fiber restrictions: {len(grid_base)}")
print(f"Master output table:\n{master_output_csv}")
print(f"PNG output directory:\n{output_dir}")


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

master_rows = []
model_curve_cache = {}

for idx, row in df.iterrows():

    fiberid = int(row["fiberid"])

    if (idx % 10) == 0:
        print(f"\nProcessing fiber {idx+1} / {len(df)}: fiberid={fiberid}", flush=True)

    output_png = os.path.join(output_dir, f"SED_Fiber_{fiberid:04d}_MARCS_restricted_cached.png")

    failed_result = {
        "fiberid": fiberid,
        "status": "failed",
        "error": "",
        "png_file": output_png,
    }

    try:
        teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
        logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
        feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
        afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

        if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
            raise ValueError(
                f"Missing SP_Ace parameters: Teff={teff_sp}, logg={logg_sp}, FeH={feh_sp}"
            )

        sed = build_dereddened_sed(row, exclude_bands=exclude_bands_global)

        if len(sed) < 2:
            raise ValueError("Fewer than two usable photometric bands.")

        cmd_info = get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp)

        n_cache_before = len(grid_base)

        grid_restricted, used_geom = restrict_grid_for_fiber(
            grid_base,
            row,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            cmd_info
        )

        fallback_used = False

        if len(grid_restricted) == 0 and allow_fallback_wider_box:
            grid_restricted = restrict_grid_fallback(
                grid_base,
                teff_sp,
                logg_sp,
                feh_sp,
                afe_sp
            )
            used_geom = "fallback"
            fallback_used = True

        n_cache_after = len(grid_restricted)

        if n_cache_after == 0:
            raise ValueError("No cached MARCS models left after restricted CMD/SP_Ace cuts.")

        if (idx % 10) == 0:
            print(f"  restricted models: {n_cache_after} / {n_cache_before}", flush=True)

        results_df = vectorized_cached_fit(grid_restricted, sed)

        best = results_df.iloc[0]

        if make_plots:
            png_file = plot_best_model_for_fiber(
                fiberid,
                row,
                sed,
                best,
                cmd_info,
                wave_A,
                model_curve_cache,
                output_png
            )
        else:
            png_file = ""

        used_bands = ",".join(sed["band"].tolist())

        result_row = {
            "fiberid": fiberid,
            "status": "modeled",
            "error": "",
            "png_file": png_file,

            "RV": row["RV"] if "RV" in row.index else np.nan,
            "e_RV": row["e_RV"] if "e_RV" in row.index else np.nan,
            "S/N": row["S/N"] if "S/N" in row.index else np.nan,

            "u0": cmd_info["u0"],
            "i0": cmd_info["i0"],
            "ui0": cmd_info["ui0"],
            "cmd_region": cmd_info["cmd_region"],
            "used_bands": used_bands,
            "n_bands": len(sed),

            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,

            "best_Teff": best["teff"],
            "best_logg": best["logg"],
            "best_FeH": best["feh"],
            "best_aFe": best["alpha"],
            "best_geom": best["geom"],
            "best_abund": best["abund"],
            "best_vturb": best["vturb"],

            "best_scale": best["scale"],
            "chi2": best["chi2"],
            "red_chi2": best["red_chi2"],
            "nfit": best["nfit"],

            "best_filename": best["filename"],
            "best_path": best["path"],

            "dTeff_model_minus_SP": best["teff"] - teff_sp,
            "dlogg_model_minus_SP": best["logg"] - logg_sp,
            "dFeH_model_minus_SP": best["feh"] - feh_sp,
            "daFe_model_minus_SP": best["alpha"] - afe_sp,

            "n_cache_before_priors": n_cache_before,
            "n_cache_after_priors": n_cache_after,
            "used_geom_preference": used_geom,
            "fallback_used": fallback_used,

            "teff_low": cmd_info["teff_low"],
            "teff_high": cmd_info["teff_high"],
            "logg_low": cmd_info["logg_low"],
            "logg_high": cmd_info["logg_high"],
            "feh_low": cmd_info["feh_low"],
            "feh_high": cmd_info["feh_high"],
            "alpha_low": cmd_info["alpha_low"],
            "alpha_high": cmd_info["alpha_high"],

            "teff_min_cmd": cmd_info["teff_min_cmd"],
            "teff_max_cmd": cmd_info["teff_max_cmd"],
            "logg_min_cmd": cmd_info["logg_min_cmd"],
            "logg_max_cmd": cmd_info["logg_max_cmd"],
            "teff_half_width": cmd_info["teff_half_width"],
            "logg_half_width": cmd_info["logg_half_width"],
            "feh_half_width": cmd_info["feh_half_width"],
            "alpha_half_width": cmd_info["alpha_half_width"],
        }

        master_rows.append(result_row)

    except Exception as e:
        failed_result["error"] = str(e)

        for col in [
            "RV", "e_RV", "S/N",
            "Teff_avg_SP", "logg_avg_SP", "FeH_avg_SP", "aFe_avg_SP",
            "umag", "Au", "imag", "Ai"
        ]:
            if col in row.index:
                failed_result[col] = row[col]

        master_rows.append(failed_result)


# ------------------------------------------------------------
# SAVE MASTER SUMMARY TABLE
# ------------------------------------------------------------

master = pd.DataFrame(master_rows)

master.to_csv(master_output_csv, index=False)

print("\nBatch restricted cached MARCS fitting complete.")
print(f"Modeled successfully: {(master['status'] == 'modeled').sum()}")
print(f"Failed/skipped:       {(master['status'] != 'modeled').sum()}")

print("\nMaster summary written to:")
print(master_output_csv)

print("\nPNG directory:")
print(output_dir)

print("\nFirst few rows of master table:")
print(master.head())

In [ ]:
# ------------------------------------------------------------
# FULL RESTRICTED BATCH MARCS SED FITTING CELL
# with SP_Ace / GCOG best-file model overlay
#
# Runs all fiberid values in the photometry file.
#
# Outputs:
#   1. Master table:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_batch_restricted_fit_summary_with_SPmodel.csv
#   2. One PNG per modeled fiber in:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_SED_PLOTS_RESTRICTED_SP_OVERLAY
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))
cache_file = os.path.join(MARCS_DIR, "MARCS_synthetic_ugrizy_cache.csv")

# SP_Ace / GCOG model directory
SP_MODEL_DIR = r"/Users/kerrycheon/repos/SpectraReductions2026/space\GCOGlibrarySPACE1"

output_dir = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_SED_PLOTS_RESTRICTED_SP_OVERLAY"
os.makedirs(output_dir, exist_ok=True)

master_output_csv = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_batch_restricted_fit_summary_with_SPmodel.csv"

# For testing, set this to e.g. 10.
# For the full run, set to None.
max_fibers_to_run = None

make_plots = True
skip_existing_png = False

exclude_bands_global = []

phot_error_floor_mag = 0.05

# Higher value keeps more spectral structure visible.
# 120 = smooth SED shape.
# 1800 = much more structure, including CaT-region features.
model_smoothing_bins = 1800

# SP_Ace/GCOG overlay flux mode:
#   "raw"       = use model flux as-is and scale to photometry
#   "flambda_A" = treat model flux as F_lambda per Angstrom and convert to nuFnu
sp_model_flux_mode = "raw"

# Strict CMD + SP_Ace prior widths
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Low-S/N widening
low_snr_threshold = 25
teff_half_width_lowsnr = 800
logg_half_width_lowsnr = 1.25
feh_half_width_lowsnr = 0.60
alpha_half_width_lowsnr = 0.50

# Optional cache filters
abundance_filter = None
geometry_filter = None

allow_fallback_wider_box = False

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not numeric data:\n{path}")

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


def resolve_model_path(path_from_cache, filename, marcs_dir):
    if isinstance(path_from_cache, str) and os.path.exists(path_from_cache):
        return path_from_cache

    matches = glob.glob(os.path.join(marcs_dir, "**", filename), recursive=True)

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find MARCS model file:\n{filename}\nCached path was:\n{path_from_cache}"
    )


# ------------------------------------------------------------
# MARCS MODEL HELPERS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def bin_model_in_log_lambda(lambda_um, flux, n_bins=1800):
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    if len(lambda_um) < 5:
        return lambda_um, flux

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


# ------------------------------------------------------------
# OBSERVED SED HELPERS
# ------------------------------------------------------------

def build_dereddened_sed(row, exclude_bands=None):
    if exclude_bands is None:
        exclude_bands = []

    sed_rows = []

    for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

        if band in exclude_bands:
            continue

        if mag_col not in row.index:
            continue

        mag = pd.to_numeric(row[mag_col], errors="coerce")
        emag_formal = pd.to_numeric(row[err_col], errors="coerce") if err_col in row.index else np.nan
        A_lambda = pd.to_numeric(row[A_col], errors="coerce") if A_col in row.index else 0.0

        if not np.isfinite(mag):
            continue
        if mag > 90 or mag < -90:
            continue
        if not np.isfinite(A_lambda):
            A_lambda = 0.0

        mag0 = mag - A_lambda

        if np.isfinite(emag_formal) and emag_formal > 0:
            emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
        else:
            emag_fit = phot_error_floor_mag

        fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
        fnu_cgs = fnu_jy * 1.0e-23

        lam_cm = lam_eff * 1.0e-4
        nu_hz = c_cgs / lam_cm
        nu_fnu = nu_hz * fnu_cgs

        frac_err = 0.4 * np.log(10.0) * emag_fit
        nu_fnu_err = nu_fnu * frac_err

        sed_rows.append({
            "band": band,
            "lambda_um": lam_eff,
            "mag_obs": mag,
            "A_lambda": A_lambda,
            "mag0": mag0,
            "mag_err_fit": emag_fit,
            "nu_fnu": nu_fnu,
            "nu_fnu_err": nu_fnu_err,
            "model_col": f"model_{band}",
        })

    sed = pd.DataFrame(sed_rows)

    if len(sed) > 0:
        sed = sed.sort_values("lambda_um").reset_index(drop=True)

    return sed


def get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp):
    u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
    i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
    ui0 = u0 - i0

    if np.isfinite(ui0) and np.isfinite(i0) and (ui0 >= 3.8) and (i0 <= 14.5):
        cmd_region = "bright RGB/AGB"
        teff_min, teff_max = 3300, 4800
        logg_min, logg_max = -0.5, 2.5

    elif np.isfinite(ui0) and (ui0 >= 3.0):
        cmd_region = "RGB"
        teff_min, teff_max = 3500, 5200
        logg_min, logg_max = 0.0, 3.2

    elif np.isfinite(ui0) and np.isfinite(i0) and (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
        cmd_region = "red clump / red HB / lower RGB"
        teff_min, teff_max = 4000, 6000
        logg_min, logg_max = 1.0, 3.8

    elif np.isfinite(ui0) and np.isfinite(i0) and (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
        cmd_region = "blue HB / warmer star"
        teff_min, teff_max = 5500, 8500
        logg_min, logg_max = 2.0, 4.8

    else:
        cmd_region = "broad fallback"
        teff_min, teff_max = 3300, 7000
        logg_min, logg_max = -0.5, 4.5

    teff_half_width = teff_half_width_default
    logg_half_width = logg_half_width_default
    feh_half_width = feh_half_width_default
    alpha_half_width = alpha_half_width_default

    snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

    if np.isfinite(snr_here) and snr_here < low_snr_threshold:
        teff_half_width = teff_half_width_lowsnr
        logg_half_width = logg_half_width_lowsnr
        feh_half_width = feh_half_width_lowsnr
        alpha_half_width = alpha_half_width_lowsnr

    teff_low = max(teff_min, teff_sp - teff_half_width)
    teff_high = min(teff_max, teff_sp + teff_half_width)

    logg_low = max(logg_min, logg_sp - logg_half_width)
    logg_high = min(logg_max, logg_sp + logg_half_width)

    feh_low = feh_sp - feh_half_width
    feh_high = feh_sp + feh_half_width

    alpha_low = afe_sp - alpha_half_width
    alpha_high = afe_sp + alpha_half_width

    return {
        "u0": u0,
        "i0": i0,
        "ui0": ui0,
        "cmd_region": cmd_region,
        "teff_low": teff_low,
        "teff_high": teff_high,
        "logg_low": logg_low,
        "logg_high": logg_high,
        "feh_low": feh_low,
        "feh_high": feh_high,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,
        "teff_min_cmd": teff_min,
        "teff_max_cmd": teff_max,
        "logg_min_cmd": logg_min,
        "logg_max_cmd": logg_max,
        "teff_half_width": teff_half_width,
        "logg_half_width": logg_half_width,
        "feh_half_width": feh_half_width,
        "alpha_half_width": alpha_half_width,
    }


def restrict_grid_for_fiber(grid_base, teff_sp, logg_sp, feh_sp, afe_sp, cmd_info):
    grid = grid_base.copy()

    grid = grid[
        (grid["teff"] >= cmd_info["teff_low"])
        & (grid["teff"] <= cmd_info["teff_high"])
        & (grid["logg"] >= cmd_info["logg_low"])
        & (grid["logg"] <= cmd_info["logg_high"])
        & (grid["feh"] >= cmd_info["feh_low"])
        & (grid["feh"] <= cmd_info["feh_high"])
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= cmd_info["alpha_low"])
            & (grid["alpha"] <= cmd_info["alpha_high"])
        ].copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    return grid.reset_index(drop=True), used_geom


def restrict_grid_fallback(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    grid = grid_base[
        (grid_base["teff"] >= teff_sp - 1000)
        & (grid_base["teff"] <= teff_sp + 1000)
        & (grid_base["logg"] >= logg_sp - 1.5)
        & (grid_base["logg"] <= logg_sp + 1.5)
        & (grid_base["feh"] >= feh_sp - 0.75)
        & (grid_base["feh"] <= feh_sp + 0.75)
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= afe_sp - 0.6)
            & (grid["alpha"] <= afe_sp + 0.6)
        ].copy()

    return grid.reset_index(drop=True)


def vectorized_cached_fit(grid, sed):
    obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)
    model_cols = sed["model_col"].tolist()

    missing_cols = [c for c in model_cols if c not in grid.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Cache is missing model columns: {missing_cols}")

    M = grid[model_cols].to_numpy(dtype=float)

    y = obs_flux.astype(float)
    s = obs_err.astype(float)

    good_band = np.isfinite(y) & np.isfinite(s) & (y > 0) & (s > 0)

    if np.sum(good_band) < 2:
        raise ValueError("Not enough valid observed bands for SED fit.")

    M = M[:, good_band]
    y = y[good_band]
    s = s[good_band]

    good_model = np.all(np.isfinite(M) & (M > 0), axis=1)

    grid_fit = grid.loc[good_model].copy().reset_index(drop=True)
    M = M[good_model, :]

    if len(grid_fit) == 0:
        raise ValueError("No valid model rows after checking cached model fluxes.")

    w = 1.0 / s**2

    numerator = np.sum(M * y * w, axis=1)
    denominator = np.sum(M**2 * w, axis=1)

    scale = numerator / denominator

    resid = y[None, :] - scale[:, None] * M
    chi2 = np.sum((resid / s[None, :])**2, axis=1)

    nfit = int(np.sum(good_band))
    dof = nfit - 1

    red_chi2 = chi2 / dof if dof > 0 else np.full_like(chi2, np.nan)

    results_df = grid_fit.copy()
    results_df["scale"] = scale
    results_df["chi2"] = chi2
    results_df["red_chi2"] = red_chi2
    results_df["nfit"] = nfit

    results_df = results_df[np.isfinite(results_df["chi2"])].copy()
    results_df = results_df.sort_values("chi2").reset_index(drop=True)

    if len(results_df) == 0:
        raise ValueError("No valid cached fits produced.")

    return results_df


# ------------------------------------------------------------
# SP_Ace / GCOG MODEL HELPERS
# ------------------------------------------------------------

def collect_sp_model_files(sp_model_dir):
    if not os.path.isdir(sp_model_dir):
        print(f"SP model directory not found:\n{sp_model_dir}")
        return []

    patterns = [
        "*.txt", "*.dat", "*.csv", "*.model", "*.mod",
        "*.spec", "*.out", "*.asc", "*.gz"
    ]

    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(sp_model_dir, "**", pat), recursive=True))

    files = sorted(list(set(files)))

    print(f"Found {len(files)} possible SP_Ace/GCOG model files.")
    return files


def parse_params_from_filename(path):
    name = os.path.basename(path)

    teff = np.nan
    logg = np.nan
    feh = np.nan
    alpha = np.nan

    m = re.search(r"(?:Teff|TEFF|teff|T|t)[_\-=]?(\d{4,5})", name)
    if m:
        teff = float(m.group(1))
    else:
        m = re.search(r"(^|_)(\d{4,5})(_|$)", name)
        if m:
            teff = float(m.group(2))

    m = re.search(r"(?:logg|LOGG|Logg|g)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        logg = float(m.group(1))

    m = re.search(r"(?:FeH|FEH|feh|MH|mh|z)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        feh = float(m.group(1))

    m = re.search(r"(?:alpha|Alpha|ALPHA|aFe|afe|a)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        alpha = float(m.group(1))

    return teff, logg, feh, alpha


def build_sp_param_table(sp_files):
    rows = []

    for path in sp_files:
        t, g, z, a = parse_params_from_filename(path)
        rows.append({
            "path": path,
            "filename": os.path.basename(path),
            "teff": t,
            "logg": g,
            "feh": z,
            "alpha": a,
        })

    return pd.DataFrame(rows)


def find_sp_model_file(row, fiberid, teff_sp, logg_sp, feh_sp, afe_sp, sp_files, sp_param_table):
    """
    Find SP_Ace/GCOG model file to overplot.

    Priority:
      1. A model filename column in the table
      2. A file containing this fiber number
      3. A nearest file by parameters encoded in filename
    """

    possible_file_cols = [
        "best_file", "best_model_file", "SP_Ace_best_file",
        "SP_model_file", "model_file", "best_filename",
        "filename", "space_model_file", "bestfit_file", "best_fit_file"
    ]

    for col in possible_file_cols:
        if col in row.index:
            val = row[col]
            if isinstance(val, str) and len(val.strip()) > 0 and val.strip().lower() != "nan":
                candidate = val.strip()

                if os.path.exists(candidate):
                    return candidate, f"from column {col}"

                candidate2 = os.path.join(SP_MODEL_DIR, candidate)

                if os.path.exists(candidate2):
                    return candidate2, f"from column {col}"

    # Search by fiber number
    fiber_strings = [
        str(fiberid),
        f"{fiberid:03d}",
        f"{fiberid:04d}",
        f"fiber{fiberid}",
        f"Fiber{fiberid}",
        f"F{fiberid}",
        f"f{fiberid}",
    ]

    matches = []

    for path in sp_files:
        name = os.path.basename(path)

        for fs in fiber_strings:
            if fs in name:
                matches.append(path)
                break

    if len(matches) > 0:
        matches = sorted(matches, key=lambda p: len(os.path.basename(p)))
        return matches[0], "matched fiberid in filename"

    # Search nearest by filename parameters
    if sp_param_table is not None and len(sp_param_table) > 0:
        tab = sp_param_table.copy()

        good = (
            np.isfinite(tab["teff"])
            & np.isfinite(tab["logg"])
            & np.isfinite(tab["feh"])
        )

        tab = tab[good].copy()

        if len(tab) > 0:
            tab["distance"] = (
                ((tab["teff"] - teff_sp) / 100.0) ** 2
                + ((tab["logg"] - logg_sp) / 0.5) ** 2
                + ((tab["feh"] - feh_sp) / 0.25) ** 2
            )

            if np.isfinite(afe_sp):
                alpha_good = np.isfinite(tab["alpha"])
                tab.loc[alpha_good, "distance"] += (
                    (tab.loc[alpha_good, "alpha"] - afe_sp) / 0.20
                ) ** 2

            tab = tab.sort_values("distance").reset_index(drop=True)

            return tab.loc[0, "path"], "nearest filename parameters"

    return None, "no SP_Ace/GCOG model found"


def read_two_column_spectrum(path):
    """
    Read a two-column model spectrum.
    Handles whitespace or comma-separated files, with or without headers.
    """

    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not a spectrum:\n{path}")

    opener = gzip.open if is_gzip_file(path) else open

    try:
        with opener(path, "rt") as f:
            dfm = pd.read_csv(
                f,
                comment="#",
                sep=r"\s+|,",
                engine="python",
                header=None
            )

        numeric = dfm.apply(pd.to_numeric, errors="coerce")
        good_cols = [c for c in numeric.columns if numeric[c].notna().sum() > 10]

        if len(good_cols) < 2:
            raise ValueError("Could not identify two numeric columns.")

        wave = numeric[good_cols[0]].to_numpy(dtype=float)
        flux = numeric[good_cols[1]].to_numpy(dtype=float)

    except Exception:
        with opener(path, "rt") as f:
            arr = np.loadtxt(f, comments="#")

        if arr.ndim != 2 or arr.shape[1] < 2:
            raise ValueError(f"Could not read {path} as a two-column spectrum.")

        wave = arr[:, 0]
        flux = arr[:, 1]

    good = np.isfinite(wave) & np.isfinite(flux)
    wave = wave[good]
    flux = flux[good]

    med_wave = np.nanmedian(wave)

    if med_wave > 1000:
        lam_um = wave * 1.0e-4        # Angstrom to micron
    elif med_wave > 100:
        lam_um = wave * 1.0e-3        # nm to micron
    elif med_wave < 1.0e-3:
        lam_um = wave * 1.0e6         # meters to micron
    else:
        lam_um = wave                 # already micron

    idx = np.argsort(lam_um)

    return lam_um[idx], flux[idx]


def prepare_sp_flux(lam_um, flux):
    """
    Prepare SP_Ace/GCOG model flux for SED overlay.
    """

    lam_um = np.asarray(lam_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = np.isfinite(lam_um) & np.isfinite(flux) & (lam_um > 0)
    lam_um = lam_um[good]
    flux = flux[good]

    if sp_model_flux_mode.lower() == "flambda_a":
        wave_cm = lam_um * 1.0e-4
        nu_hz = c_cgs / wave_cm

        flux_lambda_per_cm = flux * 1.0e8
        flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
        flux_out = nu_hz * flux_nu
    else:
        flux_out = flux

    good2 = np.isfinite(flux_out) & (flux_out > 0)

    return lam_um[good2], flux_out[good2]


def scale_model_to_photometry(lam_um, model_flux, sed):
    obs_lam = sed["lambda_um"].to_numpy(dtype=float)
    obs_y = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)

    good_model = np.isfinite(lam_um) & np.isfinite(model_flux) & (lam_um > 0) & (model_flux > 0)

    lam_um = lam_um[good_model]
    model_flux = model_flux[good_model]

    if len(lam_um) < 5:
        raise ValueError("SP_Ace/GCOG model has too few positive points.")

    idx = np.argsort(lam_um)
    lam_um = lam_um[idx]
    model_flux = model_flux[idx]

    interp_model = np.interp(obs_lam, lam_um, model_flux)

    good = (
        np.isfinite(obs_y)
        & np.isfinite(obs_err)
        & np.isfinite(interp_model)
        & (obs_y > 0)
        & (obs_err > 0)
        & (interp_model > 0)
    )

    if np.sum(good) < 2:
        raise ValueError("Not enough overlap between SP_Ace/GCOG model and photometry.")

    w = 1.0 / obs_err[good] ** 2

    scale = np.sum(interp_model[good] * obs_y[good] * w) / np.sum(interp_model[good] ** 2 * w)

    return scale


def get_sp_overlay_curve(sp_model_path, sed, sp_curve_cache):
    """
    Return smoothed, scaled SP_Ace/GCOG overlay curve.
    """

    if sp_model_path in sp_curve_cache:
        lam_smooth_raw, flux_smooth_raw = sp_curve_cache[sp_model_path]
    else:
        lam_um, flux = read_two_column_spectrum(sp_model_path)
        lam_um, flux = prepare_sp_flux(lam_um, flux)

        use = (
            (lam_um >= 0.30)
            & (lam_um <= 1.20)
            & np.isfinite(flux)
            & (flux > 0)
        )

        lam_um = lam_um[use]
        flux = flux[use]

        lam_smooth_raw, flux_smooth_raw = bin_model_in_log_lambda(
            lam_um,
            flux,
            n_bins=model_smoothing_bins
        )

        sp_curve_cache[sp_model_path] = (lam_smooth_raw, flux_smooth_raw)

    sp_scale = scale_model_to_photometry(lam_smooth_raw, flux_smooth_raw, sed)
    flux_smooth_scaled = sp_scale * flux_smooth_raw

    return lam_smooth_raw, flux_smooth_scaled, sp_scale


# ------------------------------------------------------------
# PLOTTER
# ------------------------------------------------------------

def plot_best_model_for_fiber(
    fiberid,
    row,
    sed,
    best,
    cmd_info,
    wave_A,
    model_curve_cache,
    sp_curve_cache,
    sp_model_path,
    sp_model_note,
    output_png
):
    if skip_existing_png and os.path.exists(output_png):
        return output_png, sp_model_path, sp_model_note

    best_path = resolve_model_path(best["path"], best["filename"], MARCS_DIR)

    if best_path in model_curve_cache:
        lam_smooth_unscaled, smooth_unscaled = model_curve_cache[best_path]
    else:
        flux_lambda_best = read_numeric_file(best_path)

        n = min(len(wave_A), len(flux_lambda_best))

        lam_um_best, nu_fnu_surface_best = marcs_flux_to_nuFnu(
            wave_A[:n],
            flux_lambda_best[:n]
        )

        mrange = (lam_um_best >= 0.30) & (lam_um_best <= 1.20)
        lam_um_best = lam_um_best[mrange]
        nu_fnu_surface_best = nu_fnu_surface_best[mrange]

        lam_smooth_unscaled, smooth_unscaled = bin_model_in_log_lambda(
            lam_um_best,
            nu_fnu_surface_best,
            n_bins=model_smoothing_bins
        )

        model_curve_cache[best_path] = (lam_smooth_unscaled, smooth_unscaled)

    scaled_smooth = best["scale"] * smooth_unscaled

    sp_lam = None
    sp_flux = None
    sp_scale = np.nan
    sp_plot_status = "not plotted"

    if sp_model_path is not None:
        try:
            sp_lam, sp_flux, sp_scale = get_sp_overlay_curve(sp_model_path, sed, sp_curve_cache)
            sp_plot_status = "plotted"
        except Exception as e:
            sp_plot_status = f"failed: {e}"
            sp_lam = None
            sp_flux = None

    y = sed["nu_fnu"].to_numpy(dtype=float)
    yerr = sed["nu_fnu_err"].to_numpy(dtype=float)

    y_low = y - yerr
    y_high = y + yerr

    positive_data = np.concatenate([
        y_low[np.isfinite(y_low) & (y_low > 0)],
        y_high[np.isfinite(y_high) & (y_high > 0)]
    ])

    positive_model = scaled_smooth[np.isfinite(scaled_smooth) & (scaled_smooth > 0)]

    all_positive = [positive_data, positive_model]

    if sp_flux is not None:
        positive_sp = sp_flux[np.isfinite(sp_flux) & (sp_flux > 0)]
        if len(positive_sp) > 0:
            all_positive.append(positive_sp)

    ymin = np.nanmin(np.concatenate(all_positive))
    ymax = np.nanmax(np.concatenate(all_positive))

    log_ymin = np.log10(ymin)
    log_ymax = np.log10(ymax)

    ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

    ylim_low = 10.0 ** (log_ymin - ypad)
    ylim_high = 10.0 ** (log_ymax + ypad)

    teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
    logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
    feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
    afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

    fig, ax = plt.subplots(figsize=(8.2, 5.8))

    # Smaller filled photometry points
    ax.errorbar(
        sed["lambda_um"],
        sed["nu_fnu"],
        yerr=sed["nu_fnu_err"],
        fmt="o",
        markersize=4.2,
        capsize=2.5,
        elinewidth=0.9,
        markeredgewidth=0.6,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
        label=f"Fiber {fiberid} dereddened photometry"
    )

    for _, r in sed.iterrows():
        ax.annotate(
            r["band"],
            (r["lambda_um"], r["nu_fnu"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8
        )

    marcs_label = (
        "Restricted cached MARCS "
        f"T={best['teff']:.0f} K, "
        f"logg={best['logg']:.1f}, "
        f"[Fe/H]={best['feh']:.2f}, "
        f"[α/Fe]={best['alpha']:.2f}, "
        f"χ²ν={best['red_chi2']:.2f}"
    )

    ax.plot(
        lam_smooth_unscaled,
        scaled_smooth,
        linewidth=1.1,
        alpha=0.9,
        color="tab:blue",
        label=marcs_label
    )

    if sp_lam is not None and sp_flux is not None:
        ax.plot(
            sp_lam,
            sp_flux,
            linewidth=0.9,
            alpha=0.9,
            color="tab:orange",
            label="SP_Ace/GCOG model overlay"
        )

    CaT_lines_um = [0.849802, 0.854209, 0.866214]
    for line_um in CaT_lines_um:
        ax.axvline(line_um, linestyle=":", linewidth=0.8, color="gray", alpha=0.8)

    textstr = (
        f"CMD region: {cmd_info['cmd_region']}\n"
        f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
        f"MARCS best: T={best['teff']:.0f} K, logg={best['logg']:.1f}, [Fe/H]={best['feh']:.2f}, [α/Fe]={best['alpha']:.2f}"
    )

    ax.text(
        0.03,
        0.04,
        textstr,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(0.32, 1.10)
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_xlabel("Wavelength [micron]")
    ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_title(f"Fiber {fiberid}: restricted cached MARCS SED fit + SP_Ace/GCOG overlay")

    ax.grid(alpha=0.25, which="both")
    ax.legend(loc="best", fontsize=7.3)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)

    return output_png, sp_model_path, sp_plot_status


# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "Cache file does not exist. Run the MARCS synthetic cache-building cell first:\n"
        f"{cache_file}"
    )

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)
cache = pd.read_csv(cache_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

df = df[np.isfinite(df["fiberid"])].copy()
df["fiberid"] = df["fiberid"].astype(int)

df = df.drop_duplicates(subset=["fiberid"]).sort_values("fiberid").reset_index(drop=True)

if max_fibers_to_run is not None:
    df = df.iloc[:max_fibers_to_run].copy()

grid_base = cache.copy()

if abundance_filter is not None:
    grid_base = grid_base[grid_base["abund"] == abundance_filter].copy()

if geometry_filter is not None:
    grid_base = grid_base[grid_base["geom"] == geometry_filter].copy()

if len(grid_base) == 0:
    raise ValueError("Cached MARCS grid is empty after abundance/geometry filters.")

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

sp_files = collect_sp_model_files(SP_MODEL_DIR)
sp_param_table = build_sp_param_table(sp_files)

print(f"Number of fibers to process: {len(df)}")
print(f"Number of cached MARCS models before per-fiber restrictions: {len(grid_base)}")
print(f"Master output table:\n{master_output_csv}")
print(f"PNG output directory:\n{output_dir}")


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

master_rows = []
model_curve_cache = {}
sp_curve_cache = {}

for idx, row in df.iterrows():

    fiberid = int(row["fiberid"])

    if (idx % 10) == 0:
        print(f"\nProcessing fiber {idx+1} / {len(df)}: fiberid={fiberid}", flush=True)

    output_png = os.path.join(output_dir, f"SED_Fiber_{fiberid:04d}_MARCS_restricted_SPoverlay.png")

    failed_result = {
        "fiberid": fiberid,
        "status": "failed",
        "error": "",
        "png_file": output_png,
    }

    try:
        teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
        logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
        feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
        afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

        if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
            raise ValueError(
                f"Missing SP_Ace parameters: Teff={teff_sp}, logg={logg_sp}, FeH={feh_sp}"
            )

        sed = build_dereddened_sed(row, exclude_bands=exclude_bands_global)

        if len(sed) < 2:
            raise ValueError("Fewer than two usable photometric bands.")

        cmd_info = get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp)

        n_cache_before = len(grid_base)

        grid_restricted, used_geom = restrict_grid_for_fiber(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            cmd_info
        )

        fallback_used = False

        if len(grid_restricted) == 0 and allow_fallback_wider_box:
            grid_restricted = restrict_grid_fallback(
                grid_base,
                teff_sp,
                logg_sp,
                feh_sp,
                afe_sp
            )
            used_geom = "fallback"
            fallback_used = True

        n_cache_after = len(grid_restricted)

        if n_cache_after == 0:
            raise ValueError("No cached MARCS models left after restricted CMD/SP_Ace cuts.")

        if (idx % 10) == 0:
            print(f"  restricted models: {n_cache_after} / {n_cache_before}", flush=True)

        results_df = vectorized_cached_fit(grid_restricted, sed)

        best = results_df.iloc[0]

        sp_model_path, sp_model_note = find_sp_model_file(
            row,
            fiberid,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            sp_files,
            sp_param_table
        )

        if make_plots:
            png_file, sp_model_path_used, sp_plot_status = plot_best_model_for_fiber(
                fiberid,
                row,
                sed,
                best,
                cmd_info,
                wave_A,
                model_curve_cache,
                sp_curve_cache,
                sp_model_path,
                sp_model_note,
                output_png
            )
        else:
            png_file = ""
            sp_model_path_used = sp_model_path
            sp_plot_status = "not plotted because make_plots=False"

        used_bands = ",".join(sed["band"].tolist())

        result_row = {
            "fiberid": fiberid,
            "status": "modeled",
            "error": "",
            "png_file": png_file,

            "RV": row["RV"] if "RV" in row.index else np.nan,
            "e_RV": row["e_RV"] if "e_RV" in row.index else np.nan,
            "S/N": row["S/N"] if "S/N" in row.index else np.nan,

            "u0": cmd_info["u0"],
            "i0": cmd_info["i0"],
            "ui0": cmd_info["ui0"],
            "cmd_region": cmd_info["cmd_region"],
            "used_bands": used_bands,
            "n_bands": len(sed),

            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,

            "best_Teff": best["teff"],
            "best_logg": best["logg"],
            "best_FeH": best["feh"],
            "best_aFe": best["alpha"],
            "best_geom": best["geom"],
            "best_abund": best["abund"],
            "best_vturb": best["vturb"],

            "best_scale": best["scale"],
            "chi2": best["chi2"],
            "red_chi2": best["red_chi2"],
            "nfit": best["nfit"],

            "best_filename": best["filename"],
            "best_path": best["path"],

            "SP_model_file": sp_model_path_used,
            "SP_model_match_note": sp_model_note,
            "SP_model_plot_status": sp_plot_status,

            "dTeff_model_minus_SP": best["teff"] - teff_sp,
            "dlogg_model_minus_SP": best["logg"] - logg_sp,
            "dFeH_model_minus_SP": best["feh"] - feh_sp,
            "daFe_model_minus_SP": best["alpha"] - afe_sp,

            "n_cache_before_priors": n_cache_before,
            "n_cache_after_priors": n_cache_after,
            "used_geom_preference": used_geom,
            "fallback_used": fallback_used,

            "teff_low": cmd_info["teff_low"],
            "teff_high": cmd_info["teff_high"],
            "logg_low": cmd_info["logg_low"],
            "logg_high": cmd_info["logg_high"],
            "feh_low": cmd_info["feh_low"],
            "feh_high": cmd_info["feh_high"],
            "alpha_low": cmd_info["alpha_low"],
            "alpha_high": cmd_info["alpha_high"],

            "teff_min_cmd": cmd_info["teff_min_cmd"],
            "teff_max_cmd": cmd_info["teff_max_cmd"],
            "logg_min_cmd": cmd_info["logg_min_cmd"],
            "logg_max_cmd": cmd_info["logg_max_cmd"],
            "teff_half_width": cmd_info["teff_half_width"],
            "logg_half_width": cmd_info["logg_half_width"],
            "feh_half_width": cmd_info["feh_half_width"],
            "alpha_half_width": cmd_info["alpha_half_width"],
        }

        master_rows.append(result_row)

    except Exception as e:
        failed_result["error"] = str(e)

        for col in [
            "RV", "e_RV", "S/N",
            "Teff_avg_SP", "logg_avg_SP", "FeH_avg_SP", "aFe_avg_SP",
            "umag", "Au", "imag", "Ai"
        ]:
            if col in row.index:
                failed_result[col] = row[col]

        master_rows.append(failed_result)


# ------------------------------------------------------------
# SAVE MASTER SUMMARY TABLE
# ------------------------------------------------------------

master = pd.DataFrame(master_rows)

master.to_csv(master_output_csv, index=False)

print("\nBatch restricted cached MARCS fitting complete.")
print(f"Modeled successfully: {(master['status'] == 'modeled').sum()}")
print(f"Failed/skipped:       {(master['status'] != 'modeled').sum()}")

print("\nMaster summary written to:")
print(master_output_csv)

print("\nPNG directory:")
print(output_dir)

print("\nFirst few rows of master table:")
print(master.head())

In [ ]:
# ------------------------------------------------------------
# FULL RESTRICTED BATCH MARCS SED FITTING CELL
# with automatic photometry outlier rejection
# and UNSMOOTHED / THINNED SP_Ace-GCOG model overlay
#
# Runs all fiberid values in the photometry file.
#
# Requires:
#   1. /Users/kerrycheon/repos/SpectraReductions2026/space/CaT_summary_BDBS_XCSAO.csv
#   2. /Users/kerrycheon/repos/SpectraReductions2026/space/SP_Ace_averages_with_FeHDP_avg_std.csv
#   3. /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS/MARCS_synthetic_ugrizy_cache.csv
#   4. MARCS .flx/.flx.gz files for plotting best MARCS model curves
#   5. SP_Ace/GCOG model directory:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/GCOGlibrarySPACE1
#
# Outputs:
#   1. Master table:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_batch_restricted_fit_summary_with_outlier_rejection.csv
#   2. One PNG per modeled fiber in:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_SED_PLOTS_RESTRICTED_OUTLIER_REJECT
# ------------------------------------------------------------

import os
import re
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))
cache_file = os.path.join(MARCS_DIR, "MARCS_synthetic_ugrizy_cache.csv")

# SP_Ace / GCOG model directory
SP_MODEL_DIR = r"/Users/kerrycheon/repos/SpectraReductions2026/space\GCOGlibrarySPACE1"

output_dir = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_SED_PLOTS_RESTRICTED_OUTLIER_REJECT"
os.makedirs(output_dir, exist_ok=True)

master_output_csv = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_batch_restricted_fit_summary_with_outlier_rejection.csv"

# For testing, set this to e.g. 10.
# For the full run, set to None.
max_fibers_to_run = None

make_plots = True
skip_existing_png = False

# Global manual bad-band exclusion, if needed.
# Example: exclude_bands_global = ["r"]
exclude_bands_global = []

# Optional manual per-fiber exclusions.
# Example:
# per_fiber_exclude_bands = {
#     69: ["r"],
#     215: ["u"]
# }
per_fiber_exclude_bands = {}

# Formal BDBS errors are often too small for model/photometry comparison.
phot_error_floor_mag = 0.05

# Automatic photometry outlier rejection.
# The code first fits all bands, finds the largest residual in sigma,
# removes it if it exceeds phot_outlier_sigma, and refits.
enable_photometry_outlier_rejection = True
phot_outlier_sigma = 3.0
phot_outlier_max_reject = 1

# Do not reject if fewer than this many bands would remain.
# With ugrizy, 4 is a good default.
min_bands_after_rejection = 4

# Blue MARCS curve smoothing.
# 120 = smooth SED shape.
# 1800 = keeps much more spectral structure visible.
model_smoothing_bins = 1800

# Orange SP_Ace/GCOG overlay:
# "unsmoothed" preserves line structure and only thins points for plotting.
# "smoothed" uses the same binning function as the MARCS curve.
sp_overlay_mode = "unsmoothed"

# Maximum plotted points for the orange SP_Ace/GCOG curve.
# This keeps PNG files manageable without smoothing the spectrum.
sp_max_points_to_plot = 12000

# SP_Ace/GCOG overlay flux mode:
#   "raw"       = use model flux as-is and scale to photometry
#   "flambda_A" = treat model flux as F_lambda per Angstrom and convert to nuFnu
sp_model_flux_mode = "raw"

# Strict CMD + SP_Ace prior widths
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Low-S/N widening
low_snr_threshold = 25
teff_half_width_lowsnr = 800
logg_half_width_lowsnr = 1.25
feh_half_width_lowsnr = 0.60
alpha_half_width_lowsnr = 0.50

# Optional cache filters before CMD/SP_Ace restrictions.
# Usually keep both as None.
abundance_filter = None     # None, "st", or "ae"
geometry_filter = None      # None, "s", or "p"

# If a fiber has zero models after cuts, set this True to retry with wider cuts.
# For a clean restricted run, keep False.
allow_fallback_wider_box = False

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not numeric data:\n{path}")

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


def resolve_model_path(path_from_cache, filename, marcs_dir):
    """
    Use the cached full path if it exists.
    If not, search by filename in MARCS_DIR.
    """
    if isinstance(path_from_cache, str) and os.path.exists(path_from_cache):
        return path_from_cache

    matches = glob.glob(os.path.join(marcs_dir, "**", filename), recursive=True)

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find MARCS model file:\n{filename}\nCached path was:\n{path_from_cache}"
    )


# ------------------------------------------------------------
# MARCS MODEL HELPERS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    """
    Convert MARCS F_lambda to nu Fnu.

    MARCS .flx units:
        F_lambda = erg cm^-2 s^-1 Angstrom^-1 at stellar surface
    """
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def bin_model_in_log_lambda(lambda_um, flux, n_bins=1800):
    """
    Smooth/bin high-resolution MARCS model for plotting.
    Used for the blue MARCS curve.
    """
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    if len(lambda_um) < 5:
        return lambda_um, flux

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


def thin_curve_for_plot(lam_um, flux, max_points=12000):
    """
    Thin a curve for plotting without smoothing it.
    This preserves narrow spectral features much better than median binning.
    """
    lam_um = np.asarray(lam_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lam_um)
        & np.isfinite(flux)
        & (lam_um > 0)
        & (flux > 0)
    )

    lam_um = lam_um[good]
    flux = flux[good]

    if len(lam_um) <= max_points:
        return lam_um, flux

    step = int(np.ceil(len(lam_um) / max_points))

    return lam_um[::step], flux[::step]


# ------------------------------------------------------------
# OBSERVED SED HELPERS
# ------------------------------------------------------------

def build_dereddened_sed(row, exclude_bands=None):
    """
    Build dereddened observed SED from BDBS ugrizy photometry.
    """
    if exclude_bands is None:
        exclude_bands = []

    sed_rows = []

    for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

        if band in exclude_bands:
            continue

        if mag_col not in row.index:
            continue

        mag = pd.to_numeric(row[mag_col], errors="coerce")
        emag_formal = pd.to_numeric(row[err_col], errors="coerce") if err_col in row.index else np.nan
        A_lambda = pd.to_numeric(row[A_col], errors="coerce") if A_col in row.index else 0.0

        if not np.isfinite(mag):
            continue
        if mag > 90 or mag < -90:
            continue
        if not np.isfinite(A_lambda):
            A_lambda = 0.0

        mag0 = mag - A_lambda

        if np.isfinite(emag_formal) and emag_formal > 0:
            emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
        else:
            emag_fit = phot_error_floor_mag

        fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
        fnu_cgs = fnu_jy * 1.0e-23

        lam_cm = lam_eff * 1.0e-4
        nu_hz = c_cgs / lam_cm
        nu_fnu = nu_hz * fnu_cgs

        frac_err = 0.4 * np.log(10.0) * emag_fit
        nu_fnu_err = nu_fnu * frac_err

        sed_rows.append({
            "band": band,
            "lambda_um": lam_eff,
            "mag_obs": mag,
            "A_lambda": A_lambda,
            "mag0": mag0,
            "mag_err_fit": emag_fit,
            "nu_fnu": nu_fnu,
            "nu_fnu_err": nu_fnu_err,
            "model_col": f"model_{band}",
        })

    sed = pd.DataFrame(sed_rows)

    if len(sed) > 0:
        sed = sed.sort_values("lambda_um").reset_index(drop=True)

    return sed


def get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Assign CMD region and combine CMD priors with SP_Ace local prior.
    """

    u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
    i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
    ui0 = u0 - i0

    if np.isfinite(ui0) and np.isfinite(i0) and (ui0 >= 3.8) and (i0 <= 14.5):
        cmd_region = "bright RGB/AGB"
        teff_min, teff_max = 3300, 4800
        logg_min, logg_max = -0.5, 2.5

    elif np.isfinite(ui0) and (ui0 >= 3.0):
        cmd_region = "RGB"
        teff_min, teff_max = 3500, 5200
        logg_min, logg_max = 0.0, 3.2

    elif np.isfinite(ui0) and np.isfinite(i0) and (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
        cmd_region = "red clump / red HB / lower RGB"
        teff_min, teff_max = 4000, 6000
        logg_min, logg_max = 1.0, 3.8

    elif np.isfinite(ui0) and np.isfinite(i0) and (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
        cmd_region = "blue HB / warmer star"
        teff_min, teff_max = 5500, 8500
        logg_min, logg_max = 2.0, 4.8

    else:
        cmd_region = "broad fallback"
        teff_min, teff_max = 3300, 7000
        logg_min, logg_max = -0.5, 4.5

    teff_half_width = teff_half_width_default
    logg_half_width = logg_half_width_default
    feh_half_width = feh_half_width_default
    alpha_half_width = alpha_half_width_default

    snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

    if np.isfinite(snr_here) and snr_here < low_snr_threshold:
        teff_half_width = teff_half_width_lowsnr
        logg_half_width = logg_half_width_lowsnr
        feh_half_width = feh_half_width_lowsnr
        alpha_half_width = alpha_half_width_lowsnr

    teff_low = max(teff_min, teff_sp - teff_half_width)
    teff_high = min(teff_max, teff_sp + teff_half_width)

    logg_low = max(logg_min, logg_sp - logg_half_width)
    logg_high = min(logg_max, logg_sp + logg_half_width)

    feh_low = feh_sp - feh_half_width
    feh_high = feh_sp + feh_half_width

    alpha_low = afe_sp - alpha_half_width
    alpha_high = afe_sp + alpha_half_width

    return {
        "u0": u0,
        "i0": i0,
        "ui0": ui0,
        "cmd_region": cmd_region,

        "teff_low": teff_low,
        "teff_high": teff_high,
        "logg_low": logg_low,
        "logg_high": logg_high,
        "feh_low": feh_low,
        "feh_high": feh_high,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,

        "teff_min_cmd": teff_min,
        "teff_max_cmd": teff_max,
        "logg_min_cmd": logg_min,
        "logg_max_cmd": logg_max,

        "teff_half_width": teff_half_width,
        "logg_half_width": logg_half_width,
        "feh_half_width": feh_half_width,
        "alpha_half_width": alpha_half_width,
    }


def restrict_grid_for_fiber(grid_base, teff_sp, logg_sp, feh_sp, afe_sp, cmd_info):
    """
    Apply strict CMD + SP_Ace restrictions to cached MARCS grid.
    """

    grid = grid_base.copy()

    grid = grid[
        (grid["teff"] >= cmd_info["teff_low"])
        & (grid["teff"] <= cmd_info["teff_high"])
        & (grid["logg"] >= cmd_info["logg_low"])
        & (grid["logg"] <= cmd_info["logg_high"])
        & (grid["feh"] >= cmd_info["feh_low"])
        & (grid["feh"] <= cmd_info["feh_high"])
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= cmd_info["alpha_low"])
            & (grid["alpha"] <= cmd_info["alpha_high"])
        ].copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    return grid.reset_index(drop=True), used_geom


def restrict_grid_fallback(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Optional wider fallback if the strict restricted grid is empty.
    """

    grid = grid_base[
        (grid_base["teff"] >= teff_sp - 1000)
        & (grid_base["teff"] <= teff_sp + 1000)
        & (grid_base["logg"] >= logg_sp - 1.5)
        & (grid_base["logg"] <= logg_sp + 1.5)
        & (grid_base["feh"] >= feh_sp - 0.75)
        & (grid_base["feh"] <= feh_sp + 0.75)
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= afe_sp - 0.6)
            & (grid["alpha"] <= afe_sp + 0.6)
        ].copy()

    return grid.reset_index(drop=True)


# ------------------------------------------------------------
# FITTING HELPERS INCLUDING PHOTOMETRY OUTLIER REJECTION
# ------------------------------------------------------------

def vectorized_cached_fit(grid, sed):
    """
    Fit all models in restricted cached grid at once.
    """

    obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)
    model_cols = sed["model_col"].tolist()

    missing_cols = [c for c in model_cols if c not in grid.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Cache is missing model columns: {missing_cols}")

    M = grid[model_cols].to_numpy(dtype=float)

    y = obs_flux.astype(float)
    s = obs_err.astype(float)

    good_band = np.isfinite(y) & np.isfinite(s) & (y > 0) & (s > 0)

    if np.sum(good_band) < 2:
        raise ValueError("Not enough valid observed bands for SED fit.")

    M = M[:, good_band]
    y = y[good_band]
    s = s[good_band]

    good_model = np.all(np.isfinite(M) & (M > 0), axis=1)

    grid_fit = grid.loc[good_model].copy().reset_index(drop=True)
    M = M[good_model, :]

    if len(grid_fit) == 0:
        raise ValueError("No valid model rows after checking cached model fluxes.")

    w = 1.0 / s**2

    numerator = np.sum(M * y * w, axis=1)
    denominator = np.sum(M**2 * w, axis=1)

    scale = numerator / denominator

    resid = y[None, :] - scale[:, None] * M
    chi2 = np.sum((resid / s[None, :])**2, axis=1)

    nfit = int(np.sum(good_band))
    dof = nfit - 1

    if dof > 0:
        red_chi2 = chi2 / dof
    else:
        red_chi2 = np.full_like(chi2, np.nan)

    results_df = grid_fit.copy()
    results_df["scale"] = scale
    results_df["chi2"] = chi2
    results_df["red_chi2"] = red_chi2
    results_df["nfit"] = nfit

    results_df = results_df[np.isfinite(results_df["chi2"])].copy()
    results_df = results_df.sort_values("chi2").reset_index(drop=True)

    if len(results_df) == 0:
        raise ValueError("No valid cached fits produced.")

    return results_df


def compute_best_band_residuals(best, sed):
    """
    Compute residuals for the best model in each photometric band.

    residual_sigma = (observed - model) / uncertainty
    positive residual means photometry lies above the model.
    negative residual means photometry lies below the model.
    """

    model_cols = sed["model_col"].tolist()

    obs = sed["nu_fnu"].to_numpy(dtype=float)
    err = sed["nu_fnu_err"].to_numpy(dtype=float)

    model_raw = best[model_cols].to_numpy(dtype=float)
    model_scaled = best["scale"] * model_raw

    residual = obs - model_scaled
    residual_sigma = residual / err

    frac_residual = residual / model_scaled

    mag_residual = np.full_like(residual, np.nan, dtype=float)
    good = (obs > 0) & (model_scaled > 0)

    # Positive mag_residual means observed point is fainter/lower than model.
    mag_residual[good] = -2.5 * np.log10(obs[good] / model_scaled[good])

    resid_df = sed[["band", "lambda_um", "mag_obs", "mag0", "mag_err_fit"]].copy()
    resid_df["obs_nu_fnu"] = obs
    resid_df["model_nu_fnu"] = model_scaled
    resid_df["residual_nu_fnu"] = residual
    resid_df["residual_sigma"] = residual_sigma
    resid_df["frac_residual"] = frac_residual
    resid_df["mag_residual_obs_minus_model"] = mag_residual
    resid_df["abs_residual_sigma"] = np.abs(residual_sigma)

    resid_df = resid_df.sort_values("abs_residual_sigma", ascending=False).reset_index(drop=True)

    return resid_df


def fit_with_photometry_outlier_rejection(grid, sed_initial):
    """
    Fit the cached MARCS grid while optionally rejecting bad photometry bands.

    Procedure:
      1. Fit all usable bands.
      2. Find the band with the largest absolute residual in sigma.
      3. If that residual exceeds phot_outlier_sigma, reject that band.
      4. Refit.
      5. Repeat up to phot_outlier_max_reject times.

    The model grid is not changed during the rejection loop.
    """

    sed_work = sed_initial.copy().reset_index(drop=True)

    rejected_bands = []
    rejection_reasons = []
    residual_tables = []

    for rejection_pass in range(phot_outlier_max_reject + 1):

        results_df = vectorized_cached_fit(grid, sed_work)
        best = results_df.iloc[0]

        resid_df = compute_best_band_residuals(best, sed_work)
        residual_tables.append(resid_df.copy())

        if len(resid_df) == 0:
            break

        worst = resid_df.iloc[0]
        worst_band = str(worst["band"])
        worst_sigma = float(worst["abs_residual_sigma"])
        worst_signed_sigma = float(worst["residual_sigma"])

        can_reject_more = len(rejected_bands) < phot_outlier_max_reject
        enough_bands_after = (len(sed_work) - 1) >= min_bands_after_rejection

        if (
            enable_photometry_outlier_rejection
            and can_reject_more
            and enough_bands_after
            and np.isfinite(worst_sigma)
            and worst_sigma >= phot_outlier_sigma
        ):
            rejected_bands.append(worst_band)

            if worst_signed_sigma > 0:
                direction = "high"
            else:
                direction = "low"

            rejection_reasons.append(
                f"{worst_band}: {direction}, residual_sigma={worst_signed_sigma:.2f}"
            )

            sed_work = sed_work[sed_work["band"] != worst_band].copy().reset_index(drop=True)

        else:
            break

    # Final fit after any rejection
    final_results_df = vectorized_cached_fit(grid, sed_work)
    final_best = final_results_df.iloc[0]
    final_resid_df = compute_best_band_residuals(final_best, sed_work)

    return {
        "results_df": final_results_df,
        "best": final_best,
        "sed_used": sed_work,
        "sed_initial": sed_initial,
        "rejected_bands": rejected_bands,
        "rejection_reasons": rejection_reasons,
        "final_residuals": final_resid_df,
    }


# ------------------------------------------------------------
# SP_Ace / GCOG MODEL HELPERS
# ------------------------------------------------------------

def collect_sp_model_files(sp_model_dir):
    if not os.path.isdir(sp_model_dir):
        print(f"SP model directory not found:\n{sp_model_dir}")
        return []

    patterns = [
        "*.txt", "*.dat", "*.csv", "*.model", "*.mod",
        "*.spec", "*.out", "*.asc", "*.gz"
    ]

    files = []

    for pat in patterns:
        files.extend(glob.glob(os.path.join(sp_model_dir, "**", pat), recursive=True))

    files = sorted(list(set(files)))

    print(f"Found {len(files)} possible SP_Ace/GCOG model files.")

    return files


def parse_params_from_filename(path):
    name = os.path.basename(path)

    teff = np.nan
    logg = np.nan
    feh = np.nan
    alpha = np.nan

    m = re.search(r"(?:Teff|TEFF|teff|T|t)[_\-=]?(\d{4,5})", name)
    if m:
        teff = float(m.group(1))
    else:
        m = re.search(r"(^|_)(\d{4,5})(_|$)", name)
        if m:
            teff = float(m.group(2))

    m = re.search(r"(?:logg|LOGG|Logg|g)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        logg = float(m.group(1))

    m = re.search(r"(?:FeH|FEH|feh|MH|mh|z)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        feh = float(m.group(1))

    m = re.search(r"(?:alpha|Alpha|ALPHA|aFe|afe|a)[_\-=]?([+-]?\d+\.\d+)", name)
    if m:
        alpha = float(m.group(1))

    return teff, logg, feh, alpha


def build_sp_param_table(sp_files):
    rows = []

    for path in sp_files:
        t, g, z, a = parse_params_from_filename(path)

        rows.append({
            "path": path,
            "filename": os.path.basename(path),
            "teff": t,
            "logg": g,
            "feh": z,
            "alpha": a,
        })

    return pd.DataFrame(rows)


def find_sp_model_file(row, fiberid, teff_sp, logg_sp, feh_sp, afe_sp, sp_files, sp_param_table):
    """
    Find SP_Ace/GCOG model file to overplot.

    Priority:
      1. A model filename column in the table
      2. A file containing this fiber number
      3. A nearest file by parameters encoded in filename
    """

    possible_file_cols = [
        "best_file", "best_model_file", "SP_Ace_best_file",
        "SP_model_file", "model_file", "best_filename",
        "filename", "space_model_file", "bestfit_file", "best_fit_file"
    ]

    for col in possible_file_cols:
        if col in row.index:
            val = row[col]

            if isinstance(val, str) and len(val.strip()) > 0 and val.strip().lower() != "nan":
                candidate = val.strip()

                if os.path.exists(candidate):
                    return candidate, f"from column {col}"

                candidate2 = os.path.join(SP_MODEL_DIR, candidate)

                if os.path.exists(candidate2):
                    return candidate2, f"from column {col}"

    # Search by fiber number
    fiber_strings = [
        str(fiberid),
        f"{fiberid:03d}",
        f"{fiberid:04d}",
        f"fiber{fiberid}",
        f"Fiber{fiberid}",
        f"F{fiberid}",
        f"f{fiberid}",
    ]

    matches = []

    for path in sp_files:
        name = os.path.basename(path)

        for fs in fiber_strings:
            if fs in name:
                matches.append(path)
                break

    if len(matches) > 0:
        matches = sorted(matches, key=lambda p: len(os.path.basename(p)))
        return matches[0], "matched fiberid in filename"

    # Search nearest by filename parameters
    if sp_param_table is not None and len(sp_param_table) > 0:
        tab = sp_param_table.copy()

        good = (
            np.isfinite(tab["teff"])
            & np.isfinite(tab["logg"])
            & np.isfinite(tab["feh"])
        )

        tab = tab[good].copy()

        if len(tab) > 0:
            tab["distance"] = (
                ((tab["teff"] - teff_sp) / 100.0) ** 2
                + ((tab["logg"] - logg_sp) / 0.5) ** 2
                + ((tab["feh"] - feh_sp) / 0.25) ** 2
            )

            if np.isfinite(afe_sp):
                alpha_good = np.isfinite(tab["alpha"])

                tab.loc[alpha_good, "distance"] += (
                    (tab.loc[alpha_good, "alpha"] - afe_sp) / 0.20
                ) ** 2

            tab = tab.sort_values("distance").reset_index(drop=True)

            return tab.loc[0, "path"], "nearest filename parameters"

    return None, "no SP_Ace/GCOG model found"


def read_two_column_spectrum(path):
    """
    Read a two-column model spectrum.
    Handles whitespace or comma-separated files, with or without headers.
    """

    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not a spectrum:\n{path}")

    opener = gzip.open if is_gzip_file(path) else open

    try:
        with opener(path, "rt") as f:
            dfm = pd.read_csv(
                f,
                comment="#",
                sep=r"\s+|,",
                engine="python",
                header=None
            )

        numeric = dfm.apply(pd.to_numeric, errors="coerce")
        good_cols = [c for c in numeric.columns if numeric[c].notna().sum() > 10]

        if len(good_cols) < 2:
            raise ValueError("Could not identify two numeric columns.")

        wave = numeric[good_cols[0]].to_numpy(dtype=float)
        flux = numeric[good_cols[1]].to_numpy(dtype=float)

    except Exception:
        with opener(path, "rt") as f:
            arr = np.loadtxt(f, comments="#")

        if arr.ndim != 2 or arr.shape[1] < 2:
            raise ValueError(f"Could not read {path} as a two-column spectrum.")

        wave = arr[:, 0]
        flux = arr[:, 1]

    good = np.isfinite(wave) & np.isfinite(flux)

    wave = wave[good]
    flux = flux[good]

    med_wave = np.nanmedian(wave)

    if med_wave > 1000:
        lam_um = wave * 1.0e-4        # Angstrom to micron
    elif med_wave > 100:
        lam_um = wave * 1.0e-3        # nm to micron
    elif med_wave < 1.0e-3:
        lam_um = wave * 1.0e6         # meters to micron
    else:
        lam_um = wave                 # already micron

    idx = np.argsort(lam_um)

    return lam_um[idx], flux[idx]


def prepare_sp_flux(lam_um, flux):
    """
    Prepare SP_Ace/GCOG model flux for SED overlay.
    """

    lam_um = np.asarray(lam_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = np.isfinite(lam_um) & np.isfinite(flux) & (lam_um > 0)

    lam_um = lam_um[good]
    flux = flux[good]

    if sp_model_flux_mode.lower() == "flambda_a":
        wave_cm = lam_um * 1.0e-4
        nu_hz = c_cgs / wave_cm

        flux_lambda_per_cm = flux * 1.0e8
        flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
        flux_out = nu_hz * flux_nu
    else:
        flux_out = flux

    good2 = np.isfinite(flux_out) & (flux_out > 0)

    return lam_um[good2], flux_out[good2]


def scale_model_to_photometry(lam_um, model_flux, sed):
    """
    Fit arbitrary model curve to dereddened photometry by scale only.
    """

    obs_lam = sed["lambda_um"].to_numpy(dtype=float)
    obs_y = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)

    good_model = np.isfinite(lam_um) & np.isfinite(model_flux) & (lam_um > 0) & (model_flux > 0)

    lam_um = lam_um[good_model]
    model_flux = model_flux[good_model]

    if len(lam_um) < 5:
        raise ValueError("SP_Ace/GCOG model has too few positive points.")

    idx = np.argsort(lam_um)

    lam_um = lam_um[idx]
    model_flux = model_flux[idx]

    interp_model = np.interp(obs_lam, lam_um, model_flux)

    good = (
        np.isfinite(obs_y)
        & np.isfinite(obs_err)
        & np.isfinite(interp_model)
        & (obs_y > 0)
        & (obs_err > 0)
        & (interp_model > 0)
    )

    if np.sum(good) < 2:
        raise ValueError("Not enough overlap between SP_Ace/GCOG model and photometry.")

    w = 1.0 / obs_err[good] ** 2

    scale = np.sum(interp_model[good] * obs_y[good] * w) / np.sum(interp_model[good] ** 2 * w)

    return scale


def get_sp_overlay_curve(sp_model_path, sed, sp_curve_cache):
    """
    Return scaled SP_Ace/GCOG overlay curve.

    This version does NOT median-bin the SP_Ace/GCOG model by default.
    It scales the full model to the fitted photometry, then thins for plotting.
    """

    if sp_model_path in sp_curve_cache:
        return sp_curve_cache[sp_model_path]

    lam_um, flux = read_two_column_spectrum(sp_model_path)
    lam_um, flux = prepare_sp_flux(lam_um, flux)

    use = (
        (lam_um >= 0.30)
        & (lam_um <= 1.20)
        & np.isfinite(flux)
        & (flux > 0)
    )

    lam_um = lam_um[use]
    flux = flux[use]

    if len(lam_um) < 5:
        raise ValueError("SP_Ace/GCOG model has too few usable points.")

    idx = np.argsort(lam_um)

    lam_um = lam_um[idx]
    flux = flux[idx]

    sp_scale = scale_model_to_photometry(lam_um, flux, sed)
    flux_scaled = sp_scale * flux

    if sp_overlay_mode == "smoothed":
        lam_plot, flux_plot = bin_model_in_log_lambda(
            lam_um,
            flux_scaled,
            n_bins=model_smoothing_bins
        )
    else:
        lam_plot, flux_plot = thin_curve_for_plot(
            lam_um,
            flux_scaled,
            max_points=sp_max_points_to_plot
        )

    sp_curve_cache[sp_model_path] = (lam_plot, flux_plot, sp_scale)

    return lam_plot, flux_plot, sp_scale


# ------------------------------------------------------------
# PLOTTER
# ------------------------------------------------------------

def plot_best_model_for_fiber(
    fiberid,
    row,
    sed_initial,
    sed_used,
    rejected_bands,
    best,
    cmd_info,
    wave_A,
    model_curve_cache,
    sp_curve_cache,
    sp_model_path,
    sp_model_note,
    output_png
):
    if skip_existing_png and os.path.exists(output_png):
        return output_png, sp_model_path, sp_model_note

    best_path = resolve_model_path(best["path"], best["filename"], MARCS_DIR)

    if best_path in model_curve_cache:
        lam_smooth_unscaled, smooth_unscaled = model_curve_cache[best_path]
    else:
        flux_lambda_best = read_numeric_file(best_path)

        n = min(len(wave_A), len(flux_lambda_best))

        lam_um_best, nu_fnu_surface_best = marcs_flux_to_nuFnu(
            wave_A[:n],
            flux_lambda_best[:n]
        )

        mrange = (lam_um_best >= 0.30) & (lam_um_best <= 1.20)

        lam_um_best = lam_um_best[mrange]
        nu_fnu_surface_best = nu_fnu_surface_best[mrange]

        lam_smooth_unscaled, smooth_unscaled = bin_model_in_log_lambda(
            lam_um_best,
            nu_fnu_surface_best,
            n_bins=model_smoothing_bins
        )

        model_curve_cache[best_path] = (lam_smooth_unscaled, smooth_unscaled)

    scaled_smooth = best["scale"] * smooth_unscaled

    sp_lam = None
    sp_flux = None
    sp_scale = np.nan
    sp_plot_status = "not plotted"

    if sp_model_path is not None:
        try:
            # Scale SP_Ace/GCOG model to the final fitted photometry, after rejection.
            sp_lam, sp_flux, sp_scale = get_sp_overlay_curve(sp_model_path, sed_used, sp_curve_cache)
            sp_plot_status = "plotted"
        except Exception as e:
            sp_plot_status = f"failed: {e}"
            sp_lam = None
            sp_flux = None

    # Axis limits include all original points, including rejected ones.
    y_all = sed_initial["nu_fnu"].to_numpy(dtype=float)
    yerr_all = sed_initial["nu_fnu_err"].to_numpy(dtype=float)

    y_low = y_all - yerr_all
    y_high = y_all + yerr_all

    positive_data = np.concatenate([
        y_low[np.isfinite(y_low) & (y_low > 0)],
        y_high[np.isfinite(y_high) & (y_high > 0)]
    ])

    positive_model = scaled_smooth[np.isfinite(scaled_smooth) & (scaled_smooth > 0)]

    all_positive = [positive_data, positive_model]

    if sp_flux is not None:
        positive_sp = sp_flux[np.isfinite(sp_flux) & (sp_flux > 0)]
        if len(positive_sp) > 0:
            all_positive.append(positive_sp)

    ymin = np.nanmin(np.concatenate(all_positive))
    ymax = np.nanmax(np.concatenate(all_positive))

    log_ymin = np.log10(ymin)
    log_ymax = np.log10(ymax)

    ypad = 0.12 * (log_ymax - log_ymin) if log_ymax > log_ymin else 0.25

    ylim_low = 10.0 ** (log_ymin - ypad)
    ylim_high = 10.0 ** (log_ymax + ypad)

    teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
    logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
    feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
    afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

    fig, ax = plt.subplots(figsize=(8.2, 5.8))

    # Plot used photometry as smaller filled black circles.
    used = sed_initial[~sed_initial["band"].isin(rejected_bands)].copy()

    ax.errorbar(
        used["lambda_um"],
        used["nu_fnu"],
        yerr=used["nu_fnu_err"],
        fmt="o",
        markersize=4.2,
        capsize=2.5,
        elinewidth=0.9,
        markeredgewidth=0.6,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
        label=f"Fiber {fiberid} fitted photometry"
    )

    # Plot rejected photometry as open red squares.
    rejected = sed_initial[sed_initial["band"].isin(rejected_bands)].copy()

    if len(rejected) > 0:
        ax.errorbar(
            rejected["lambda_um"],
            rejected["nu_fnu"],
            yerr=rejected["nu_fnu_err"],
            fmt="s",
            markersize=5.0,
            capsize=2.5,
            elinewidth=0.9,
            markeredgewidth=1.0,
            markerfacecolor="none",
            markeredgecolor="red",
            ecolor="red",
            linestyle="none",
            label="Rejected photometry"
        )

    for _, r in sed_initial.iterrows():
        label_color = "red" if r["band"] in rejected_bands else "black"
        ax.annotate(
            r["band"],
            (r["lambda_um"], r["nu_fnu"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
            color=label_color
        )

    marcs_label = (
        "Restricted cached MARCS "
        f"T={best['teff']:.0f} K, "
        f"logg={best['logg']:.1f}, "
        f"[Fe/H]={best['feh']:.2f}, "
        f"[α/Fe]={best['alpha']:.2f}, "
        f"χ²ν={best['red_chi2']:.2f}"
    )

    ax.plot(
        lam_smooth_unscaled,
        scaled_smooth,
        linewidth=1.1,
        alpha=0.9,
        color="tab:blue",
        label=marcs_label
    )

    # Orange SP_Ace/GCOG overlay is unsmoothed/thinned.
    if sp_lam is not None and sp_flux is not None:
        ax.plot(
            sp_lam,
            sp_flux,
            linewidth=0.45,
            alpha=0.85,
            color="tab:orange",
            label="SP_Ace/GCOG model overlay"
        )

    CaT_lines_um = [0.849802, 0.854209, 0.866214]

    for line_um in CaT_lines_um:
        ax.axvline(line_um, linestyle=":", linewidth=0.8, color="gray", alpha=0.8)

    rejected_text = ",".join(rejected_bands) if len(rejected_bands) > 0 else "none"

    textstr = (
        f"CMD region: {cmd_info['cmd_region']}\n"
        f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
        f"MARCS best: T={best['teff']:.0f} K, logg={best['logg']:.1f}, [Fe/H]={best['feh']:.2f}, [α/Fe]={best['alpha']:.2f}\n"
        f"Rejected bands: {rejected_text}"
    )

    ax.text(
        0.03,
        0.04,
        textstr,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(0.32, 1.10)
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_xlabel("Wavelength [micron]")
    ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_title(f"Fiber {fiberid}: restricted cached MARCS SED fit + outlier rejection")

    ax.grid(alpha=0.25, which="both")
    ax.legend(loc="best", fontsize=7.3)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)

    return output_png, sp_model_path, sp_plot_status


# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "Cache file does not exist. Run the MARCS synthetic cache-building cell first:\n"
        f"{cache_file}"
    )

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)
cache = pd.read_csv(cache_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

df = df[np.isfinite(df["fiberid"])].copy()
df["fiberid"] = df["fiberid"].astype(int)

df = df.drop_duplicates(subset=["fiberid"]).sort_values("fiberid").reset_index(drop=True)

if max_fibers_to_run is not None:
    df = df.iloc[:max_fibers_to_run].copy()

grid_base = cache.copy()

if abundance_filter is not None:
    grid_base = grid_base[grid_base["abund"] == abundance_filter].copy()

if geometry_filter is not None:
    grid_base = grid_base[grid_base["geom"] == geometry_filter].copy()

if len(grid_base) == 0:
    raise ValueError("Cached MARCS grid is empty after abundance/geometry filters.")

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

sp_files = collect_sp_model_files(SP_MODEL_DIR)
sp_param_table = build_sp_param_table(sp_files)

print(f"Number of fibers to process: {len(df)}")
print(f"Number of cached MARCS models before per-fiber restrictions: {len(grid_base)}")
print(f"Automatic photometry outlier rejection: {enable_photometry_outlier_rejection}")
print(f"Outlier threshold: {phot_outlier_sigma} sigma")
print(f"Maximum rejected bands per star: {phot_outlier_max_reject}")
print(f"Master output table:\n{master_output_csv}")
print(f"PNG output directory:\n{output_dir}")


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

master_rows = []
model_curve_cache = {}
sp_curve_cache = {}

for idx, row in df.iterrows():

    fiberid = int(row["fiberid"])

    if (idx % 10) == 0:
        print(f"\nProcessing fiber {idx+1} / {len(df)}: fiberid={fiberid}", flush=True)

    output_png = os.path.join(output_dir, f"SED_Fiber_{fiberid:04d}_MARCS_restricted_outlier_reject.png")

    failed_result = {
        "fiberid": fiberid,
        "status": "failed",
        "error": "",
        "png_file": output_png,
    }

    try:
        teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
        logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
        feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
        afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

        if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
            raise ValueError(
                f"Missing SP_Ace parameters: Teff={teff_sp}, logg={logg_sp}, FeH={feh_sp}"
            )

        manual_exclude = list(exclude_bands_global)

        if fiberid in per_fiber_exclude_bands:
            manual_exclude += list(per_fiber_exclude_bands[fiberid])

        manual_exclude = sorted(list(set(manual_exclude)))

        sed_initial = build_dereddened_sed(row, exclude_bands=manual_exclude)

        if len(sed_initial) < 2:
            raise ValueError("Fewer than two usable photometric bands.")

        cmd_info = get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp)

        n_cache_before = len(grid_base)

        grid_restricted, used_geom = restrict_grid_for_fiber(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            cmd_info
        )

        fallback_used = False

        if len(grid_restricted) == 0 and allow_fallback_wider_box:
            grid_restricted = restrict_grid_fallback(
                grid_base,
                teff_sp,
                logg_sp,
                feh_sp,
                afe_sp
            )
            used_geom = "fallback"
            fallback_used = True

        n_cache_after = len(grid_restricted)

        if n_cache_after == 0:
            raise ValueError("No cached MARCS models left after restricted CMD/SP_Ace cuts.")

        if (idx % 10) == 0:
            print(f"  restricted models: {n_cache_after} / {n_cache_before}", flush=True)

        fit_info = fit_with_photometry_outlier_rejection(
            grid_restricted,
            sed_initial
        )

        results_df = fit_info["results_df"]
        best = fit_info["best"]
        sed_used = fit_info["sed_used"]
        rejected_bands = fit_info["rejected_bands"]
        rejection_reasons = fit_info["rejection_reasons"]
        final_residuals = fit_info["final_residuals"]

        sp_model_path, sp_model_note = find_sp_model_file(
            row,
            fiberid,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            sp_files,
            sp_param_table
        )

        if make_plots:
            png_file, sp_model_path_used, sp_plot_status = plot_best_model_for_fiber(
                fiberid,
                row,
                sed_initial,
                sed_used,
                rejected_bands,
                best,
                cmd_info,
                wave_A,
                model_curve_cache,
                sp_curve_cache,
                sp_model_path,
                sp_model_note,
                output_png
            )
        else:
            png_file = ""
            sp_model_path_used = sp_model_path
            sp_plot_status = "not plotted because make_plots=False"

        initial_bands = ",".join(sed_initial["band"].tolist())
        used_bands = ",".join(sed_used["band"].tolist())
        rejected_bands_str = ",".join(rejected_bands)
        rejection_reasons_str = "; ".join(rejection_reasons)

        if len(final_residuals) > 0:
            max_resid_band = final_residuals.iloc[0]["band"]
            max_abs_resid_sigma = final_residuals.iloc[0]["abs_residual_sigma"]
            max_signed_resid_sigma = final_residuals.iloc[0]["residual_sigma"]

            residual_summary = "; ".join([
                f"{r['band']}:{r['residual_sigma']:.2f}"
                for _, r in final_residuals.iterrows()
            ])
        else:
            max_resid_band = ""
            max_abs_resid_sigma = np.nan
            max_signed_resid_sigma = np.nan
            residual_summary = ""

        result_row = {
            "fiberid": fiberid,
            "status": "modeled",
            "error": "",
            "png_file": png_file,

            "RV": row["RV"] if "RV" in row.index else np.nan,
            "e_RV": row["e_RV"] if "e_RV" in row.index else np.nan,
            "S/N": row["S/N"] if "S/N" in row.index else np.nan,

            "u0": cmd_info["u0"],
            "i0": cmd_info["i0"],
            "ui0": cmd_info["ui0"],
            "cmd_region": cmd_info["cmd_region"],

            "initial_bands": initial_bands,
            "used_bands": used_bands,
            "manual_excluded_bands": ",".join(manual_exclude),
            "auto_rejected_bands": rejected_bands_str,
            "auto_rejection_reasons": rejection_reasons_str,
            "n_bands_initial": len(sed_initial),
            "n_bands_used": len(sed_used),

            "max_resid_band_after_rejection": max_resid_band,
            "max_abs_resid_sigma_after_rejection": max_abs_resid_sigma,
            "max_signed_resid_sigma_after_rejection": max_signed_resid_sigma,
            "band_residuals_after_rejection": residual_summary,

            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,

            "best_Teff": best["teff"],
            "best_logg": best["logg"],
            "best_FeH": best["feh"],
            "best_aFe": best["alpha"],
            "best_geom": best["geom"],
            "best_abund": best["abund"],
            "best_vturb": best["vturb"],

            "best_scale": best["scale"],
            "chi2": best["chi2"],
            "red_chi2": best["red_chi2"],
            "nfit": best["nfit"],

            "best_filename": best["filename"],
            "best_path": best["path"],

            "SP_model_file": sp_model_path_used,
            "SP_model_match_note": sp_model_note,
            "SP_model_plot_status": sp_plot_status,

            "dTeff_model_minus_SP": best["teff"] - teff_sp,
            "dlogg_model_minus_SP": best["logg"] - logg_sp,
            "dFeH_model_minus_SP": best["feh"] - feh_sp,
            "daFe_model_minus_SP": best["alpha"] - afe_sp,

            "n_cache_before_priors": n_cache_before,
            "n_cache_after_priors": n_cache_after,
            "used_geom_preference": used_geom,
            "fallback_used": fallback_used,

            "teff_low": cmd_info["teff_low"],
            "teff_high": cmd_info["teff_high"],
            "logg_low": cmd_info["logg_low"],
            "logg_high": cmd_info["logg_high"],
            "feh_low": cmd_info["feh_low"],
            "feh_high": cmd_info["feh_high"],
            "alpha_low": cmd_info["alpha_low"],
            "alpha_high": cmd_info["alpha_high"],

            "teff_min_cmd": cmd_info["teff_min_cmd"],
            "teff_max_cmd": cmd_info["teff_max_cmd"],
            "logg_min_cmd": cmd_info["logg_min_cmd"],
            "logg_max_cmd": cmd_info["logg_max_cmd"],
            "teff_half_width": cmd_info["teff_half_width"],
            "logg_half_width": cmd_info["logg_half_width"],
            "feh_half_width": cmd_info["feh_half_width"],
            "alpha_half_width": cmd_info["alpha_half_width"],
        }

        master_rows.append(result_row)

    except Exception as e:
        failed_result["error"] = str(e)

        for col in [
            "RV", "e_RV", "S/N",
            "Teff_avg_SP", "logg_avg_SP", "FeH_avg_SP", "aFe_avg_SP",
            "umag", "Au", "imag", "Ai"
        ]:
            if col in row.index:
                failed_result[col] = row[col]

        master_rows.append(failed_result)


# ------------------------------------------------------------
# SAVE MASTER SUMMARY TABLE
# ------------------------------------------------------------

master = pd.DataFrame(master_rows)

master.to_csv(master_output_csv, index=False)

print("\nBatch restricted cached MARCS fitting complete.")
print(f"Modeled successfully: {(master['status'] == 'modeled').sum()}")
print(f"Failed/skipped:       {(master['status'] != 'modeled').sum()}")

print("\nMaster summary written to:")
print(master_output_csv)

print("\nPNG directory:")
print(output_dir)

print("\nFirst few rows of master table:")
print(master.head())


In [ ]:
# ------------------------------------------------------------
# FULL RESTRICTED BATCH MARCS SED FITTING CELL
# with leave-one-out photometry outlier rejection
# and MARCS model closest to the SP_Ace fit overplotted
#
# Plot layout:
#   legend fixed inside the plot at upper-left
#   information box fixed inside the plot at lower-left
#
# Outputs:
#   1. Master table:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_batch_restricted_fit_summary_LOO_reject_SPclosest_MARCS.csv
#   2. One PNG per modeled fiber in:
#        /Users/kerrycheon/repos/SpectraReductions2026/space/MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS
# ------------------------------------------------------------

import os
import glob
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

phot_file = r"/Users/kerrycheon/repos/SpectraReductions2026/space\CaT_summary_BDBS_XCSAO.csv"

summary_file_candidates = [
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std.csv",
    r"/Users/kerrycheon/repos/SpectraReductions2026/space\SP_Ace_averages_with_FeHDP_avg_std(1).csv",
]

MARCS_DIR = os.getenv("MARCS_DIR", str(PROJECT_ROOT / "data/raw/MARCS"))
cache_file = os.path.join(MARCS_DIR, "MARCS_synthetic_ugrizy_cache.csv")

output_dir = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_SED_PLOTS_RESTRICTED_LOO_REJECT_SPclosest_MARCS"
os.makedirs(output_dir, exist_ok=True)

master_output_csv = r"/Users/kerrycheon/repos/SpectraReductions2026/space\MARCS_batch_restricted_fit_summary_LOO_reject_SPclosest_MARCS.csv"

# For testing, set this to e.g. 10.
# For the full run, set to None.
max_fibers_to_run = None

make_plots = True
skip_existing_png = False

# Manual exclusions, if needed.
exclude_bands_global = []

per_fiber_exclude_bands = {
    # Example:
    # 69: ["r"],
    # 215: ["r"],
}

# Formal BDBS errors are often too small for model/photometry comparison.
phot_error_floor_mag = 0.05

# ------------------------------------------------------------
# Improved automatic photometry outlier rejection
# ------------------------------------------------------------

enable_photometry_outlier_rejection = True

# Candidate point must be this discrepant from the leave-one-out refit.
phot_outlier_sigma = 3.0

# Reject at most this many bands automatically.
phot_outlier_max_reject = 1

# Do not reject if too few bands would remain.
min_bands_after_rejection = 4

# Require the leave-one-out reduced chi2 to improve enough.
phot_outlier_min_redchi2_improvement = 1.0
phot_outlier_min_redchi2_ratio = 0.80

# Require a local SED kink if possible.
# This helps avoid rejecting a point that is part of a broad color mismatch.
phot_outlier_local_mag_threshold = 0.30

# Flag possible variables / broad photometric mismatch after final fit.
variable_flag_abs_resid_sigma = 3.0
variable_flag_min_bad_bands = 2
variable_flag_red_chi2 = 5.0

# MARCS display smoothing.
# 120 = smooth SED shape.
# 1800 = keeps more spectral structure visible.
model_smoothing_bins = 1800

# SP_Ace prior widths for the restricted photometric fit.
teff_half_width_default = 600
logg_half_width_default = 1.0
feh_half_width_default = 0.50
alpha_half_width_default = 0.40

# Low-S/N widening.
low_snr_threshold = 25
teff_half_width_lowsnr = 800
logg_half_width_lowsnr = 1.25
feh_half_width_lowsnr = 0.60
alpha_half_width_lowsnr = 0.50

# Optional cache filters before CMD/SP_Ace restrictions.
abundance_filter = None     # None, "st", or "ae"
geometry_filter = None      # None, "s", or "p"

# If a fiber has zero models after cuts, set this True to retry with wider cuts.
allow_fallback_wider_box = False

c_cgs = 2.99792458e10


# ------------------------------------------------------------
# BAND DEFINITIONS
# ------------------------------------------------------------

bands = [
    # band, mag_col, err_col, extinction_col, lambda_eff_um, band_min_um, band_max_um, zp_Jy
    ("u", "umag", "uerr", "Au", 0.36, 0.32, 0.40, 3631.0),
    ("g", "gmag", "gerr", "Ag", 0.48, 0.40, 0.55, 3631.0),
    ("r", "rmag", "rerr", "Ar", 0.625, 0.55, 0.70, 3631.0),
    ("i", "imag", "ierr", "Ai", 0.77, 0.70, 0.85, 3631.0),
    ("z", "zmag", "zerr", "Az", 0.91, 0.85, 0.96, 3631.0),
    ("y", "ymag", "yerr", "Ay", 1.00, 0.96, 1.08, 3631.0),
]


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def find_existing_file(candidates):
    for f in candidates:
        if os.path.exists(f):
            return f
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def is_gzip_file(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"


def looks_like_html(path):
    with open(path, "rb") as f:
        start = f.read(300).lstrip().lower()
    return start.startswith(b"<!doctype") or start.startswith(b"<html")


def read_numeric_file(path):
    if looks_like_html(path):
        raise ValueError(f"This file is HTML, not numeric data:\n{path}")

    if is_gzip_file(path):
        with gzip.open(path, "rt") as f:
            arr = np.loadtxt(f)
    else:
        arr = np.loadtxt(path)

    return np.ravel(arr)


def find_wavelength_file(marcs_dir):
    candidates = [
        os.path.join(marcs_dir, "flx_wavelengths.vac"),
        os.path.join(marcs_dir, "flx_wavelengths.vac.gz"),
    ]

    for f in candidates:
        if os.path.exists(f):
            return f

    matches = glob.glob(
        os.path.join(marcs_dir, "**", "flx_wavelengths.vac*"),
        recursive=True
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find flx_wavelengths.vac or flx_wavelengths.vac.gz in {marcs_dir}"
        )

    return matches[0]


def resolve_model_path(path_from_cache, filename, marcs_dir):
    if isinstance(path_from_cache, str) and os.path.exists(path_from_cache):
        return path_from_cache

    matches = glob.glob(os.path.join(marcs_dir, "**", filename), recursive=True)

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find MARCS model file:\n{filename}\nCached path was:\n{path_from_cache}"
    )


# ------------------------------------------------------------
# MARCS MODEL HELPERS
# ------------------------------------------------------------

def marcs_flux_to_nuFnu(wave_A, flux_lambda):
    good = (
        np.isfinite(wave_A)
        & np.isfinite(flux_lambda)
        & (wave_A > 0)
        & (flux_lambda > 0)
    )

    wave_A = wave_A[good]
    flux_lambda = flux_lambda[good]

    wave_cm = wave_A * 1.0e-8
    lambda_um = wave_A * 1.0e-4
    nu_hz = c_cgs / wave_cm

    flux_lambda_per_cm = flux_lambda * 1.0e8
    flux_nu = flux_lambda_per_cm * wave_cm**2 / c_cgs
    nu_fnu = nu_hz * flux_nu

    idx = np.argsort(lambda_um)

    return lambda_um[idx], nu_fnu[idx]


def bin_model_in_log_lambda(lambda_um, flux, n_bins=1800):
    lambda_um = np.asarray(lambda_um, dtype=float)
    flux = np.asarray(flux, dtype=float)

    good = (
        np.isfinite(lambda_um)
        & np.isfinite(flux)
        & (lambda_um > 0)
        & (flux > 0)
    )

    lambda_um = lambda_um[good]
    flux = flux[good]

    if len(lambda_um) < 5:
        return lambda_um, flux

    loglam = np.log10(lambda_um)
    bins = np.linspace(loglam.min(), loglam.max(), n_bins + 1)

    lam_b = []
    flux_b = []

    for i in range(n_bins):
        m = (loglam >= bins[i]) & (loglam < bins[i + 1])

        if np.sum(m) < 2:
            continue

        lam_b.append(10 ** np.nanmean(loglam[m]))
        flux_b.append(np.nanmedian(flux[m]))

    return np.array(lam_b), np.array(flux_b)


# ------------------------------------------------------------
# OBSERVED SED HELPERS
# ------------------------------------------------------------

def build_dereddened_sed(row, exclude_bands=None):
    if exclude_bands is None:
        exclude_bands = []

    sed_rows = []

    for band, mag_col, err_col, A_col, lam_eff, band_min, band_max, zp_jy in bands:

        if band in exclude_bands:
            continue

        if mag_col not in row.index:
            continue

        mag = pd.to_numeric(row[mag_col], errors="coerce")
        emag_formal = pd.to_numeric(row[err_col], errors="coerce") if err_col in row.index else np.nan
        A_lambda = pd.to_numeric(row[A_col], errors="coerce") if A_col in row.index else 0.0

        if not np.isfinite(mag):
            continue
        if mag > 90 or mag < -90:
            continue
        if not np.isfinite(A_lambda):
            A_lambda = 0.0

        mag0 = mag - A_lambda

        if np.isfinite(emag_formal) and emag_formal > 0:
            emag_fit = np.sqrt(emag_formal**2 + phot_error_floor_mag**2)
        else:
            emag_fit = phot_error_floor_mag

        fnu_jy = zp_jy * 10.0 ** (-0.4 * mag0)
        fnu_cgs = fnu_jy * 1.0e-23

        lam_cm = lam_eff * 1.0e-4
        nu_hz = c_cgs / lam_cm
        nu_fnu = nu_hz * fnu_cgs

        frac_err = 0.4 * np.log(10.0) * emag_fit
        nu_fnu_err = nu_fnu * frac_err

        sed_rows.append({
            "band": band,
            "lambda_um": lam_eff,
            "mag_obs": mag,
            "A_lambda": A_lambda,
            "mag0": mag0,
            "mag_err_fit": emag_fit,
            "nu_fnu": nu_fnu,
            "nu_fnu_err": nu_fnu_err,
            "model_col": f"model_{band}",
        })

    sed = pd.DataFrame(sed_rows)

    if len(sed) > 0:
        sed = sed.sort_values("lambda_um").reset_index(drop=True)

    return sed


def get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp):
    u0 = pd.to_numeric(row["umag"], errors="coerce") - pd.to_numeric(row["Au"], errors="coerce")
    i0 = pd.to_numeric(row["imag"], errors="coerce") - pd.to_numeric(row["Ai"], errors="coerce")
    ui0 = u0 - i0

    if np.isfinite(ui0) and np.isfinite(i0) and (ui0 >= 3.8) and (i0 <= 14.5):
        cmd_region = "bright RGB/AGB"
        teff_min, teff_max = 3300, 4800
        logg_min, logg_max = -0.5, 2.5

    elif np.isfinite(ui0) and (ui0 >= 3.0):
        cmd_region = "RGB"
        teff_min, teff_max = 3500, 5200
        logg_min, logg_max = 0.0, 3.2

    elif np.isfinite(ui0) and np.isfinite(i0) and (2.0 <= ui0 < 3.8) and (14.0 <= i0 <= 16.8):
        cmd_region = "red clump / red HB / lower RGB"
        teff_min, teff_max = 4000, 6000
        logg_min, logg_max = 1.0, 3.8

    elif np.isfinite(ui0) and np.isfinite(i0) and (ui0 < 2.0) and (14.5 <= i0 <= 17.0):
        cmd_region = "blue HB / warmer star"
        teff_min, teff_max = 5500, 8500
        logg_min, logg_max = 2.0, 4.8

    else:
        cmd_region = "broad fallback"
        teff_min, teff_max = 3300, 7000
        logg_min, logg_max = -0.5, 4.5

    teff_half_width = teff_half_width_default
    logg_half_width = logg_half_width_default
    feh_half_width = feh_half_width_default
    alpha_half_width = alpha_half_width_default

    snr_here = pd.to_numeric(row["S/N"], errors="coerce") if "S/N" in row.index else np.nan

    if np.isfinite(snr_here) and snr_here < low_snr_threshold:
        teff_half_width = teff_half_width_lowsnr
        logg_half_width = logg_half_width_lowsnr
        feh_half_width = feh_half_width_lowsnr
        alpha_half_width = alpha_half_width_lowsnr

    teff_low = max(teff_min, teff_sp - teff_half_width)
    teff_high = min(teff_max, teff_sp + teff_half_width)

    logg_low = max(logg_min, logg_sp - logg_half_width)
    logg_high = min(logg_max, logg_sp + logg_half_width)

    feh_low = feh_sp - feh_half_width
    feh_high = feh_sp + feh_half_width

    alpha_low = afe_sp - alpha_half_width
    alpha_high = afe_sp + alpha_half_width

    return {
        "u0": u0,
        "i0": i0,
        "ui0": ui0,
        "cmd_region": cmd_region,

        "teff_low": teff_low,
        "teff_high": teff_high,
        "logg_low": logg_low,
        "logg_high": logg_high,
        "feh_low": feh_low,
        "feh_high": feh_high,
        "alpha_low": alpha_low,
        "alpha_high": alpha_high,

        "teff_min_cmd": teff_min,
        "teff_max_cmd": teff_max,
        "logg_min_cmd": logg_min,
        "logg_max_cmd": logg_max,

        "teff_half_width": teff_half_width,
        "logg_half_width": logg_half_width,
        "feh_half_width": feh_half_width,
        "alpha_half_width": alpha_half_width,
    }


def restrict_grid_for_fiber(grid_base, teff_sp, logg_sp, feh_sp, afe_sp, cmd_info):
    grid = grid_base.copy()

    grid = grid[
        (grid["teff"] >= cmd_info["teff_low"])
        & (grid["teff"] <= cmd_info["teff_high"])
        & (grid["logg"] >= cmd_info["logg_low"])
        & (grid["logg"] <= cmd_info["logg_high"])
        & (grid["feh"] >= cmd_info["feh_low"])
        & (grid["feh"] <= cmd_info["feh_high"])
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= cmd_info["alpha_low"])
            & (grid["alpha"] <= cmd_info["alpha_high"])
        ].copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    return grid.reset_index(drop=True), used_geom


def restrict_grid_fallback(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    grid = grid_base[
        (grid_base["teff"] >= teff_sp - 1000)
        & (grid_base["teff"] <= teff_sp + 1000)
        & (grid_base["logg"] >= logg_sp - 1.5)
        & (grid_base["logg"] <= logg_sp + 1.5)
        & (grid_base["feh"] >= feh_sp - 0.75)
        & (grid_base["feh"] <= feh_sp + 0.75)
    ].copy()

    if np.isfinite(afe_sp):
        grid = grid[
            (grid["alpha"] >= afe_sp - 0.6)
            & (grid["alpha"] <= afe_sp + 0.6)
        ].copy()

    return grid.reset_index(drop=True)


# ------------------------------------------------------------
# FITTING HELPERS
# ------------------------------------------------------------

def vectorized_cached_fit(grid, sed):
    obs_flux = sed["nu_fnu"].to_numpy(dtype=float)
    obs_err = sed["nu_fnu_err"].to_numpy(dtype=float)
    model_cols = sed["model_col"].tolist()

    missing_cols = [c for c in model_cols if c not in grid.columns]
    if len(missing_cols) > 0:
        raise KeyError(f"Cache is missing model columns: {missing_cols}")

    M = grid[model_cols].to_numpy(dtype=float)

    y = obs_flux.astype(float)
    s = obs_err.astype(float)

    good_band = np.isfinite(y) & np.isfinite(s) & (y > 0) & (s > 0)

    if np.sum(good_band) < 2:
        raise ValueError("Not enough valid observed bands for SED fit.")

    M = M[:, good_band]
    y = y[good_band]
    s = s[good_band]

    good_model = np.all(np.isfinite(M) & (M > 0), axis=1)

    grid_fit = grid.loc[good_model].copy().reset_index(drop=True)
    M = M[good_model, :]

    if len(grid_fit) == 0:
        raise ValueError("No valid model rows after checking cached model fluxes.")

    w = 1.0 / s**2

    numerator = np.sum(M * y * w, axis=1)
    denominator = np.sum(M**2 * w, axis=1)

    scale = numerator / denominator

    resid = y[None, :] - scale[:, None] * M
    chi2 = np.sum((resid / s[None, :])**2, axis=1)

    nfit = int(np.sum(good_band))
    dof = nfit - 1

    if dof > 0:
        red_chi2 = chi2 / dof
    else:
        red_chi2 = np.full_like(chi2, np.nan)

    results_df = grid_fit.copy()
    results_df["scale"] = scale
    results_df["chi2"] = chi2
    results_df["red_chi2"] = red_chi2
    results_df["nfit"] = nfit

    results_df = results_df[np.isfinite(results_df["chi2"])].copy()
    results_df = results_df.sort_values("chi2").reset_index(drop=True)

    if len(results_df) == 0:
        raise ValueError("No valid cached fits produced.")

    return results_df


def compute_best_band_residuals(best, sed):
    model_cols = sed["model_col"].tolist()

    obs = sed["nu_fnu"].to_numpy(dtype=float)
    err = sed["nu_fnu_err"].to_numpy(dtype=float)

    model_raw = best[model_cols].to_numpy(dtype=float)
    model_scaled = best["scale"] * model_raw

    residual = obs - model_scaled
    residual_sigma = residual / err
    frac_residual = residual / model_scaled

    mag_residual = np.full_like(residual, np.nan, dtype=float)
    good = (obs > 0) & (model_scaled > 0)

    # Positive mag_residual means observed point is fainter/lower than model.
    mag_residual[good] = -2.5 * np.log10(obs[good] / model_scaled[good])

    resid_df = sed[["band", "lambda_um", "mag_obs", "mag0", "mag_err_fit"]].copy()
    resid_df["obs_nu_fnu"] = obs
    resid_df["model_nu_fnu"] = model_scaled
    resid_df["residual_nu_fnu"] = residual
    resid_df["residual_sigma"] = residual_sigma
    resid_df["frac_residual"] = frac_residual
    resid_df["mag_residual_obs_minus_model"] = mag_residual
    resid_df["abs_residual_sigma"] = np.abs(residual_sigma)

    resid_df = resid_df.sort_values("abs_residual_sigma", ascending=False).reset_index(drop=True)

    return resid_df


def local_sed_isolation_mag(sed, band):
    """
    Measure whether a band is locally isolated relative to its neighbors.
    Positive value means the point is low/faint relative to local interpolation.
    Negative value means the point is high/bright relative to local interpolation.
    """

    sed_sorted = sed.sort_values("lambda_um").reset_index(drop=True)

    idx_list = sed_sorted.index[sed_sorted["band"] == band].tolist()

    if len(idx_list) == 0:
        return np.nan

    idx = idx_list[0]

    # Need one neighbor on each side.
    if idx == 0 or idx == len(sed_sorted) - 1:
        return np.nan

    left = sed_sorted.iloc[idx - 1]
    mid = sed_sorted.iloc[idx]
    right = sed_sorted.iloc[idx + 1]

    if (
        left["nu_fnu"] <= 0
        or mid["nu_fnu"] <= 0
        or right["nu_fnu"] <= 0
        or left["lambda_um"] <= 0
        or mid["lambda_um"] <= 0
        or right["lambda_um"] <= 0
    ):
        return np.nan

    x_left = np.log10(left["lambda_um"])
    x_mid = np.log10(mid["lambda_um"])
    x_right = np.log10(right["lambda_um"])

    y_left = np.log10(left["nu_fnu"])
    y_mid = np.log10(mid["nu_fnu"])
    y_right = np.log10(right["nu_fnu"])

    y_pred = y_left + (y_right - y_left) * (x_mid - x_left) / (x_right - x_left)

    delta_mag = -2.5 * (y_mid - y_pred)

    return delta_mag


def evaluate_leave_one_out_candidate(grid, sed_work, band_to_test, current_red_chi2):
    """
    Try removing one band and refit.

    This asks:
      1. Does removing this band improve reduced chi2?
      2. Is the removed band discrepant from the leave-one-out refit?
      3. Is the removed band locally suspicious relative to neighboring bands?
    """

    sed_try = sed_work[sed_work["band"] != band_to_test].copy().reset_index(drop=True)

    if len(sed_try) < min_bands_after_rejection:
        return None

    results_try = vectorized_cached_fit(grid, sed_try)
    best_try = results_try.iloc[0]

    cand = sed_work[sed_work["band"] == band_to_test].iloc[0]

    obs = float(cand["nu_fnu"])
    err = float(cand["nu_fnu_err"])
    model_col = cand["model_col"]

    if (
        not np.isfinite(obs)
        or not np.isfinite(err)
        or err <= 0
        or model_col not in best_try.index
        or not np.isfinite(best_try[model_col])
        or best_try[model_col] <= 0
    ):
        return None

    pred = float(best_try["scale"] * best_try[model_col])

    if not np.isfinite(pred) or pred <= 0:
        return None

    residual_sigma = (obs - pred) / err
    abs_residual_sigma = abs(residual_sigma)

    # Positive means observed point is fainter/lower than the leave-one-out model.
    mag_resid = -2.5 * np.log10(obs / pred)

    local_delta_mag = local_sed_isolation_mag(sed_work, band_to_test)

    red_try = float(best_try["red_chi2"])
    red_improvement = current_red_chi2 - red_try

    if np.isfinite(current_red_chi2) and current_red_chi2 > 0 and np.isfinite(red_try):
        red_ratio = red_try / current_red_chi2
    else:
        red_ratio = np.nan

    if np.isfinite(local_delta_mag):
        locally_suspicious = abs(local_delta_mag) >= phot_outlier_local_mag_threshold
    else:
        # End bands have no two-sided local check, so require stronger evidence.
        locally_suspicious = abs_residual_sigma >= 5.0

    chi2_improves_enough = (
        np.isfinite(red_improvement)
        and np.isfinite(red_ratio)
        and (
            red_improvement >= phot_outlier_min_redchi2_improvement
            or red_ratio <= phot_outlier_min_redchi2_ratio
        )
    )

    passes_rejection_test = (
        abs_residual_sigma >= phot_outlier_sigma
        and chi2_improves_enough
        and locally_suspicious
    )

    local_weight = abs(local_delta_mag) if np.isfinite(local_delta_mag) else 0.0

    score = abs_residual_sigma * max(red_improvement, 0.0) * (1.0 + local_weight)

    return {
        "band": band_to_test,
        "sed_try": sed_try,
        "results_try": results_try,
        "best_try": best_try,
        "red_chi2_try": red_try,
        "red_chi2_current": current_red_chi2,
        "red_chi2_improvement": red_improvement,
        "red_chi2_ratio": red_ratio,
        "residual_sigma_leave_one_out": residual_sigma,
        "abs_residual_sigma_leave_one_out": abs_residual_sigma,
        "mag_residual_leave_one_out": mag_resid,
        "local_delta_mag": local_delta_mag,
        "locally_suspicious": locally_suspicious,
        "chi2_improves_enough": chi2_improves_enough,
        "passes_rejection_test": passes_rejection_test,
        "score": score,
    }


def fit_with_photometry_outlier_rejection(grid, sed_initial):
    """
    Improved automatic photometry rejection.

    Instead of rejecting the largest residual from a single global fit,
    this tests each band by leave-one-out refitting.
    """

    sed_work = sed_initial.copy().reset_index(drop=True)

    rejected_bands = []
    rejection_reasons = []
    rejection_diagnostics = []

    for rejection_pass in range(phot_outlier_max_reject + 1):

        results_current = vectorized_cached_fit(grid, sed_work)
        best_current = results_current.iloc[0]
        current_red_chi2 = float(best_current["red_chi2"])

        if not enable_photometry_outlier_rejection:
            break

        if len(rejected_bands) >= phot_outlier_max_reject:
            break

        if (len(sed_work) - 1) < min_bands_after_rejection:
            break

        candidates = []

        for band in sed_work["band"].tolist():
            cand = evaluate_leave_one_out_candidate(
                grid=grid,
                sed_work=sed_work,
                band_to_test=band,
                current_red_chi2=current_red_chi2
            )

            if cand is not None:
                candidates.append(cand)

        if len(candidates) == 0:
            break

        cand_df = pd.DataFrame([
            {
                "band": c["band"],
                "red_chi2_current": c["red_chi2_current"],
                "red_chi2_try": c["red_chi2_try"],
                "red_chi2_improvement": c["red_chi2_improvement"],
                "red_chi2_ratio": c["red_chi2_ratio"],
                "residual_sigma_leave_one_out": c["residual_sigma_leave_one_out"],
                "abs_residual_sigma_leave_one_out": c["abs_residual_sigma_leave_one_out"],
                "mag_residual_leave_one_out": c["mag_residual_leave_one_out"],
                "local_delta_mag": c["local_delta_mag"],
                "locally_suspicious": c["locally_suspicious"],
                "chi2_improves_enough": c["chi2_improves_enough"],
                "passes_rejection_test": c["passes_rejection_test"],
                "score": c["score"],
            }
            for c in candidates
        ])

        rejection_diagnostics.append(cand_df)

        passing = [c for c in candidates if c["passes_rejection_test"]]

        if len(passing) == 0:
            break

        best_candidate = sorted(passing, key=lambda c: c["score"], reverse=True)[0]

        reject_band = best_candidate["band"]
        rejected_bands.append(reject_band)

        direction = "high" if best_candidate["residual_sigma_leave_one_out"] > 0 else "low"

        reason = (
            f"{reject_band}: {direction}, "
            f"LOO_resid_sigma={best_candidate['residual_sigma_leave_one_out']:.2f}, "
            f"local_delta_mag={best_candidate['local_delta_mag']:.2f}, "
            f"red_chi2 {best_candidate['red_chi2_current']:.2f} -> {best_candidate['red_chi2_try']:.2f}"
        )

        rejection_reasons.append(reason)

        sed_work = sed_work[sed_work["band"] != reject_band].copy().reset_index(drop=True)

    final_results_df = vectorized_cached_fit(grid, sed_work)
    final_best = final_results_df.iloc[0]
    final_resid_df = compute_best_band_residuals(final_best, sed_work)

    if len(rejection_diagnostics) > 0:
        last_candidate_table = rejection_diagnostics[-1].copy()
        last_candidate_table = last_candidate_table.sort_values("score", ascending=False).reset_index(drop=True)
    else:
        last_candidate_table = pd.DataFrame()

    return {
        "results_df": final_results_df,
        "best": final_best,
        "sed_used": sed_work,
        "sed_initial": sed_initial,
        "rejected_bands": rejected_bands,
        "rejection_reasons": rejection_reasons,
        "rejection_diagnostics": rejection_diagnostics,
        "last_candidate_table": last_candidate_table,
        "final_residuals": final_resid_df,
    }


def make_candidate_summary(candidate_table):
    if candidate_table is None or len(candidate_table) == 0:
        return ""

    rows = []

    for _, r in candidate_table.iterrows():
        rows.append(
            f"{r['band']}: score={r['score']:.2f}, "
            f"LOO_sig={r['residual_sigma_leave_one_out']:.2f}, "
            f"dchi2nu={r['red_chi2_improvement']:.2f}, "
            f"localmag={r['local_delta_mag']:.2f}, "
            f"pass={r['passes_rejection_test']}"
        )

    return "; ".join(rows)


def make_variable_or_mismatch_flag(final_residuals, red_chi2):
    if final_residuals is None or len(final_residuals) == 0:
        return False, 0

    n_bad = int(np.sum(final_residuals["abs_residual_sigma"] >= variable_flag_abs_resid_sigma))

    flag = (
        n_bad >= variable_flag_min_bad_bands
        or (
            np.isfinite(red_chi2)
            and red_chi2 >= variable_flag_red_chi2
        )
    )

    return bool(flag), n_bad


# ------------------------------------------------------------
# MARCS MODEL CLOSEST TO SP_ACE
# ------------------------------------------------------------

def find_closest_marcs_to_sp_ace(grid_base, teff_sp, logg_sp, feh_sp, afe_sp):
    """
    Find the MARCS model closest to the SP_Ace atmospheric parameters.
    This is independent of the photometric chi2 fit.
    """

    grid = grid_base.copy()

    preferred_geom = "s" if logg_sp <= 3.5 else "p"
    sub = grid[grid["geom"] == preferred_geom].copy()

    if len(sub) > 0:
        grid = sub
        used_geom = preferred_geom
    else:
        used_geom = "mixed"

    grid["SP_distance"] = (
        ((grid["teff"] - teff_sp) / 100.0) ** 2
        + ((grid["logg"] - logg_sp) / 0.5) ** 2
        + ((grid["feh"] - feh_sp) / 0.25) ** 2
    )

    if np.isfinite(afe_sp):
        grid["SP_distance"] += ((grid["alpha"] - afe_sp) / 0.20) ** 2

    closest = grid.sort_values("SP_distance").iloc[0].copy()
    closest["SP_closest_used_geom"] = used_geom

    return closest


def fit_single_cached_model_to_sed(model_row, sed):
    """
    Scale one cached MARCS model to the observed SED and compute chi2.
    Used for the MARCS model closest to SP_Ace.
    """

    model_cols = sed["model_col"].tolist()

    obs = sed["nu_fnu"].to_numpy(dtype=float)
    err = sed["nu_fnu_err"].to_numpy(dtype=float)
    mod = model_row[model_cols].to_numpy(dtype=float)

    good = (
        np.isfinite(obs)
        & np.isfinite(err)
        & np.isfinite(mod)
        & (obs > 0)
        & (err > 0)
        & (mod > 0)
    )

    if np.sum(good) < 2:
        return np.nan, np.nan, np.nan, int(np.sum(good))

    y = obs[good]
    s = err[good]
    m = mod[good]

    w = 1.0 / s**2

    scale = np.sum(m * y * w) / np.sum(m**2 * w)

    chi2 = np.sum(((y - scale * m) / s) ** 2)

    nfit = int(np.sum(good))
    dof = nfit - 1

    red_chi2 = chi2 / dof if dof > 0 else np.nan

    return scale, chi2, red_chi2, nfit


# ------------------------------------------------------------
# PLOTTER
# ------------------------------------------------------------

def get_smoothed_model_curve(model_row, wave_A, model_curve_cache):
    """
    Read and smooth one MARCS model spectrum for plotting.
    Returns unscaled smoothed curve.
    """

    model_path = resolve_model_path(model_row["path"], model_row["filename"], MARCS_DIR)

    if model_path in model_curve_cache:
        return model_curve_cache[model_path]

    flux_lambda = read_numeric_file(model_path)

    n = min(len(wave_A), len(flux_lambda))

    lam_um, nu_fnu_surface = marcs_flux_to_nuFnu(
        wave_A[:n],
        flux_lambda[:n]
    )

    mrange = (lam_um >= 0.30) & (lam_um <= 1.20)

    lam_um = lam_um[mrange]
    nu_fnu_surface = nu_fnu_surface[mrange]

    lam_smooth, flux_smooth = bin_model_in_log_lambda(
        lam_um,
        nu_fnu_surface,
        n_bins=model_smoothing_bins
    )

    model_curve_cache[model_path] = (lam_smooth, flux_smooth)

    return lam_smooth, flux_smooth


def plot_best_model_for_fiber(
    fiberid,
    row,
    sed_initial,
    sed_used,
    rejected_bands,
    best_phot,
    closest_sp_model,
    closest_sp_scale,
    closest_sp_red_chi2,
    cmd_info,
    wave_A,
    model_curve_cache,
    output_png
):
    if skip_existing_png and os.path.exists(output_png):
        return output_png

    # Photometric best MARCS model
    lam_best, flux_best_unscaled = get_smoothed_model_curve(
        best_phot,
        wave_A,
        model_curve_cache
    )

    flux_best_scaled = best_phot["scale"] * flux_best_unscaled

    # MARCS model closest to SP_Ace parameters
    lam_sp, flux_sp_unscaled = get_smoothed_model_curve(
        closest_sp_model,
        wave_A,
        model_curve_cache
    )

    flux_sp_scaled = closest_sp_scale * flux_sp_unscaled

    # Axis limits include all original points, including rejected ones.
    y_all = sed_initial["nu_fnu"].to_numpy(dtype=float)
    yerr_all = sed_initial["nu_fnu_err"].to_numpy(dtype=float)

    y_low = y_all - yerr_all
    y_high = y_all + yerr_all

    positive_data = np.concatenate([
        y_low[np.isfinite(y_low) & (y_low > 0)],
        y_high[np.isfinite(y_high) & (y_high > 0)]
    ])

    positive_model = flux_best_scaled[np.isfinite(flux_best_scaled) & (flux_best_scaled > 0)]
    positive_sp = flux_sp_scaled[np.isfinite(flux_sp_scaled) & (flux_sp_scaled > 0)]

    ymin = np.nanmin(np.concatenate([positive_data, positive_model, positive_sp]))
    ymax = np.nanmax(np.concatenate([positive_data, positive_model, positive_sp]))

    log_ymin = np.log10(ymin)
    log_ymax = np.log10(ymax)
# ------------------------------------------------------------
# Asymmetric y-axis padding
# Leave extra vertical room above the spectrum so the legend
# does not cover the top photometry/model region.
# ------------------------------------------------------------

    yrange = log_ymax - log_ymin if log_ymax > log_ymin else 1.0

    ypad_low = 0.10 * yrange

# Larger top padding for internal legend.
# 0.45 dex means the top of the plot is about 2.8 times above the highest point.
    ypad_high = max(0.45, 0.22 * yrange)

    ylim_low = 10.0 ** (log_ymin - ypad_low)
    ylim_high = 10.0 ** (log_ymax + ypad_high)



    teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
    logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
    feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
    afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

    # Normal height; legend is inside the plot, not above it.
    fig, ax = plt.subplots(figsize=(8.8, 6.4))

    # Used photometry: filled black circles
    used = sed_initial[~sed_initial["band"].isin(rejected_bands)].copy()

    ax.errorbar(
        used["lambda_um"],
        used["nu_fnu"],
        yerr=used["nu_fnu_err"],
        fmt="o",
        markersize=4.2,
        capsize=2.5,
        elinewidth=0.9,
        markeredgewidth=0.6,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
        label=f"Fiber {fiberid} fitted photometry"
    )

    # Rejected photometry: open red squares
    rejected = sed_initial[sed_initial["band"].isin(rejected_bands)].copy()

    if len(rejected) > 0:
        ax.errorbar(
            rejected["lambda_um"],
            rejected["nu_fnu"],
            yerr=rejected["nu_fnu_err"],
            fmt="s",
            markersize=5.0,
            capsize=2.5,
            elinewidth=0.9,
            markeredgewidth=1.0,
            markerfacecolor="none",
            markeredgecolor="red",
            ecolor="red",
            linestyle="none",
            label="Rejected photometry"
        )

    for _, r in sed_initial.iterrows():
        label_color = "red" if r["band"] in rejected_bands else "black"
        ax.annotate(
            r["band"],
            (r["lambda_um"], r["nu_fnu"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
            color=label_color
        )

    # Best photometric MARCS model
    phot_label = (
        "Best photometric MARCS "
        f"T={best_phot['teff']:.0f} K, "
        f"logg={best_phot['logg']:.1f}, "
        f"[Fe/H]={best_phot['feh']:.2f}, "
        f"[α/Fe]={best_phot['alpha']:.2f}, "
        f"χ²ν={best_phot['red_chi2']:.2f}"
    )

    ax.plot(
        lam_best,
        flux_best_scaled,
        linewidth=1.1,
        alpha=0.9,
        color="tab:blue",
        label=phot_label
    )

    # MARCS model closest to SP_Ace
    spclose_label = (
        "MARCS closest to SP_Ace "
        f"T={closest_sp_model['teff']:.0f} K, "
        f"logg={closest_sp_model['logg']:.1f}, "
        f"[Fe/H]={closest_sp_model['feh']:.2f}, "
        f"[α/Fe]={closest_sp_model['alpha']:.2f}, "
        f"χ²ν={closest_sp_red_chi2:.2f}"
    )

    ax.plot(
        lam_sp,
        flux_sp_scaled,
        linewidth=0.95,
        alpha=0.9,
        color="tab:orange",
        label=spclose_label
    )

    CaT_lines_um = [0.849802, 0.854209, 0.866214]

    for line_um in CaT_lines_um:
        ax.axvline(line_um, linestyle=":", linewidth=0.8, color="gray", alpha=0.8)

    rejected_text = ",".join(rejected_bands) if len(rejected_bands) > 0 else "none"

    textstr = (
        f"CMD region: {cmd_info['cmd_region']}\n"
        f"SP_Ace: T={teff_sp:.0f} K, logg={logg_sp:.2f}, [Fe/H]={feh_sp:.2f}, [α/Fe]={afe_sp:.2f}\n"
        f"Phot best: T={best_phot['teff']:.0f} K, logg={best_phot['logg']:.1f}, [Fe/H]={best_phot['feh']:.2f}, [α/Fe]={best_phot['alpha']:.2f}\n"
        f"SP-nearest MARCS: T={closest_sp_model['teff']:.0f} K, logg={closest_sp_model['logg']:.1f}, [Fe/H]={closest_sp_model['feh']:.2f}, [α/Fe]={closest_sp_model['alpha']:.2f}\n"
        f"Rejected bands: {rejected_text}"
    )

    # Fixed information panel: lower-left
    ax.text(
        0.03,
        0.04,
        textstr,
        transform=ax.transAxes,
        fontsize=7.6,
        va="bottom",
        ha="left",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            edgecolor="black",
            alpha=0.75
        ),
        zorder=10
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(0.32, 1.10)
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_xlabel("Wavelength [micron]")
    ax.set_ylabel(r"$\nu F_{\nu}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_title(f"Fiber {fiberid}: photometric MARCS fit vs SP_Ace-nearest MARCS model")

    ax.grid(alpha=0.25, which="both")

    # Fixed legend panel INSIDE the plot, upper-left.
    # It stays on the plot and no longer collides with the title.
    legend = ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.01, 0.99),
        fontsize=7.0,
        framealpha=0.88,
        borderaxespad=0.0,
        handlelength=2.4,
        ncol=1
    )

    legend.set_zorder(20)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)

    return output_png


# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------

if not os.path.exists(cache_file):
    raise FileNotFoundError(
        "Cache file does not exist. Run the MARCS synthetic cache-building cell first:\n"
        f"{cache_file}"
    )

summary_file = find_existing_file(summary_file_candidates)

phot = pd.read_csv(phot_file)
summary = pd.read_csv(summary_file)
cache = pd.read_csv(cache_file)

phot["fiberid"] = pd.to_numeric(phot["fiberid"], errors="coerce")
summary["fiberid"] = pd.to_numeric(summary["fiberid"], errors="coerce")

df = phot.merge(summary, on="fiberid", how="left", suffixes=("", "_SPsummary"))

df = df[np.isfinite(df["fiberid"])].copy()
df["fiberid"] = df["fiberid"].astype(int)

df = df.drop_duplicates(subset=["fiberid"]).sort_values("fiberid").reset_index(drop=True)

if max_fibers_to_run is not None:
    df = df.iloc[:max_fibers_to_run].copy()

grid_base = cache.copy()

if abundance_filter is not None:
    grid_base = grid_base[grid_base["abund"] == abundance_filter].copy()

if geometry_filter is not None:
    grid_base = grid_base[grid_base["geom"] == geometry_filter].copy()

if len(grid_base) == 0:
    raise ValueError("Cached MARCS grid is empty after abundance/geometry filters.")

wave_file = find_wavelength_file(MARCS_DIR)
wave_A = read_numeric_file(wave_file)

print(f"Number of fibers to process: {len(df)}")
print(f"Number of cached MARCS models before per-fiber restrictions: {len(grid_base)}")
print(f"Automatic photometry outlier rejection: {enable_photometry_outlier_rejection}")
print(f"Outlier threshold: {phot_outlier_sigma} sigma")
print(f"Maximum rejected bands per star: {phot_outlier_max_reject}")
print(f"Master output table:\n{master_output_csv}")
print(f"PNG output directory:\n{output_dir}")


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

master_rows = []
model_curve_cache = {}

for idx, row in df.iterrows():

    fiberid = int(row["fiberid"])

    if (idx % 10) == 0:
        print(f"\nProcessing fiber {idx+1} / {len(df)}: fiberid={fiberid}", flush=True)

    output_png = os.path.join(output_dir, f"SED_Fiber_{fiberid:04d}_MARCS_phot_vs_SPnearest_LOOreject.png")

    failed_result = {
        "fiberid": fiberid,
        "status": "failed",
        "error": "",
        "png_file": output_png,
    }

    try:
        teff_sp = pd.to_numeric(row["Teff_avg_SP"], errors="coerce")
        logg_sp = pd.to_numeric(row["logg_avg_SP"], errors="coerce")
        feh_sp = pd.to_numeric(row["FeH_avg_SP"], errors="coerce")
        afe_sp = pd.to_numeric(row["aFe_avg_SP"], errors="coerce")

        if not np.isfinite(teff_sp) or not np.isfinite(logg_sp) or not np.isfinite(feh_sp):
            raise ValueError(
                f"Missing SP_Ace parameters: Teff={teff_sp}, logg={logg_sp}, FeH={feh_sp}"
            )

        manual_exclude = list(exclude_bands_global)

        if fiberid in per_fiber_exclude_bands:
            manual_exclude += list(per_fiber_exclude_bands[fiberid])

        manual_exclude = sorted(list(set(manual_exclude)))

        sed_initial = build_dereddened_sed(row, exclude_bands=manual_exclude)

        if len(sed_initial) < 2:
            raise ValueError("Fewer than two usable photometric bands.")

        cmd_info = get_cmd_region_and_prior(row, teff_sp, logg_sp, feh_sp, afe_sp)

        n_cache_before = len(grid_base)

        grid_restricted, used_geom = restrict_grid_for_fiber(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp,
            cmd_info
        )

        fallback_used = False

        if len(grid_restricted) == 0 and allow_fallback_wider_box:
            grid_restricted = restrict_grid_fallback(
                grid_base,
                teff_sp,
                logg_sp,
                feh_sp,
                afe_sp
            )
            used_geom = "fallback"
            fallback_used = True

        n_cache_after = len(grid_restricted)

        if n_cache_after == 0:
            raise ValueError("No cached MARCS models left after restricted CMD/SP_Ace cuts.")

        if (idx % 10) == 0:
            print(f"  restricted models: {n_cache_after} / {n_cache_before}", flush=True)

        fit_info = fit_with_photometry_outlier_rejection(
            grid_restricted,
            sed_initial
        )

        best = fit_info["best"]
        sed_used = fit_info["sed_used"]
        rejected_bands = fit_info["rejected_bands"]
        rejection_reasons = fit_info["rejection_reasons"]
        final_residuals = fit_info["final_residuals"]
        last_candidate_table = fit_info["last_candidate_table"]

        closest_sp_model = find_closest_marcs_to_sp_ace(
            grid_base,
            teff_sp,
            logg_sp,
            feh_sp,
            afe_sp
        )

        closest_sp_scale, closest_sp_chi2, closest_sp_red_chi2, closest_sp_nfit = fit_single_cached_model_to_sed(
            closest_sp_model,
            sed_used
        )

        possible_variable_or_mismatch, n_bad_bands_after_rejection = make_variable_or_mismatch_flag(
            final_residuals,
            best["red_chi2"]
        )

        if make_plots:
            png_file = plot_best_model_for_fiber(
                fiberid,
                row,
                sed_initial,
                sed_used,
                rejected_bands,
                best,
                closest_sp_model,
                closest_sp_scale,
                closest_sp_red_chi2,
                cmd_info,
                wave_A,
                model_curve_cache,
                output_png
            )
        else:
            png_file = ""

        initial_bands = ",".join(sed_initial["band"].tolist())
        used_bands = ",".join(sed_used["band"].tolist())
        rejected_bands_str = ",".join(rejected_bands)
        rejection_reasons_str = "; ".join(rejection_reasons)
        loo_candidate_summary = make_candidate_summary(last_candidate_table)

        if len(final_residuals) > 0:
            max_resid_band = final_residuals.iloc[0]["band"]
            max_abs_resid_sigma = final_residuals.iloc[0]["abs_residual_sigma"]
            max_signed_resid_sigma = final_residuals.iloc[0]["residual_sigma"]

            residual_summary = "; ".join([
                f"{r['band']}:{r['residual_sigma']:.2f}"
                for _, r in final_residuals.iterrows()
            ])
        else:
            max_resid_band = ""
            max_abs_resid_sigma = np.nan
            max_signed_resid_sigma = np.nan
            residual_summary = ""

        result_row = {
            "fiberid": fiberid,
            "status": "modeled",
            "error": "",
            "png_file": png_file,

            "RV": row["RV"] if "RV" in row.index else np.nan,
            "e_RV": row["e_RV"] if "e_RV" in row.index else np.nan,
            "S/N": row["S/N"] if "S/N" in row.index else np.nan,

            "u0": cmd_info["u0"],
            "i0": cmd_info["i0"],
            "ui0": cmd_info["ui0"],
            "cmd_region": cmd_info["cmd_region"],

            "initial_bands": initial_bands,
            "used_bands": used_bands,
            "manual_excluded_bands": ",".join(manual_exclude),
            "auto_rejected_bands": rejected_bands_str,
            "auto_rejection_reasons": rejection_reasons_str,
            "LOO_candidate_summary": loo_candidate_summary,

            "n_bands_initial": len(sed_initial),
            "n_bands_used": len(sed_used),

            "max_resid_band_after_rejection": max_resid_band,
            "max_abs_resid_sigma_after_rejection": max_abs_resid_sigma,
            "max_signed_resid_sigma_after_rejection": max_signed_resid_sigma,
            "band_residuals_after_rejection": residual_summary,

            "possible_variable_or_photometric_mismatch": possible_variable_or_mismatch,
            "n_bad_bands_after_rejection": n_bad_bands_after_rejection,

            "Teff_SP": teff_sp,
            "logg_SP": logg_sp,
            "FeH_SP": feh_sp,
            "aFe_SP": afe_sp,

            "best_Teff": best["teff"],
            "best_logg": best["logg"],
            "best_FeH": best["feh"],
            "best_aFe": best["alpha"],
            "best_geom": best["geom"],
            "best_abund": best["abund"],
            "best_vturb": best["vturb"],

            "best_scale": best["scale"],
            "chi2": best["chi2"],
            "red_chi2": best["red_chi2"],
            "nfit": best["nfit"],

            "best_filename": best["filename"],
            "best_path": best["path"],

            "SPnearest_Teff": closest_sp_model["teff"],
            "SPnearest_logg": closest_sp_model["logg"],
            "SPnearest_FeH": closest_sp_model["feh"],
            "SPnearest_aFe": closest_sp_model["alpha"],
            "SPnearest_geom": closest_sp_model["geom"],
            "SPnearest_abund": closest_sp_model["abund"],
            "SPnearest_vturb": closest_sp_model["vturb"],
            "SPnearest_scale": closest_sp_scale,
            "SPnearest_chi2": closest_sp_chi2,
            "SPnearest_red_chi2": closest_sp_red_chi2,
            "SPnearest_nfit": closest_sp_nfit,
            "SPnearest_distance": closest_sp_model["SP_distance"],
            "SPnearest_filename": closest_sp_model["filename"],
            "SPnearest_path": closest_sp_model["path"],

            "dTeff_phot_model_minus_SP": best["teff"] - teff_sp,
            "dlogg_phot_model_minus_SP": best["logg"] - logg_sp,
            "dFeH_phot_model_minus_SP": best["feh"] - feh_sp,
            "daFe_phot_model_minus_SP": best["alpha"] - afe_sp,

            "dTeff_SPnearest_minus_SP": closest_sp_model["teff"] - teff_sp,
            "dlogg_SPnearest_minus_SP": closest_sp_model["logg"] - logg_sp,
            "dFeH_SPnearest_minus_SP": closest_sp_model["feh"] - feh_sp,
            "daFe_SPnearest_minus_SP": closest_sp_model["alpha"] - afe_sp,

            "n_cache_before_priors": n_cache_before,
            "n_cache_after_priors": n_cache_after,
            "used_geom_preference": used_geom,
            "fallback_used": fallback_used,

            "teff_low": cmd_info["teff_low"],
            "teff_high": cmd_info["teff_high"],
            "logg_low": cmd_info["logg_low"],
            "logg_high": cmd_info["logg_high"],
            "feh_low": cmd_info["feh_low"],
            "feh_high": cmd_info["feh_high"],
            "alpha_low": cmd_info["alpha_low"],
            "alpha_high": cmd_info["alpha_high"],

            "teff_min_cmd": cmd_info["teff_min_cmd"],
            "teff_max_cmd": cmd_info["teff_max_cmd"],
            "logg_min_cmd": cmd_info["logg_min_cmd"],
            "logg_max_cmd": cmd_info["logg_max_cmd"],
            "teff_half_width": cmd_info["teff_half_width"],
            "logg_half_width": cmd_info["logg_half_width"],
            "feh_half_width": cmd_info["feh_half_width"],
            "alpha_half_width": cmd_info["alpha_half_width"],
        }

        master_rows.append(result_row)

    except Exception as e:
        failed_result["error"] = str(e)

        for col in [
            "RV", "e_RV", "S/N",
            "Teff_avg_SP", "logg_avg_SP", "FeH_avg_SP", "aFe_avg_SP",
            "umag", "Au", "imag", "Ai"
        ]:
            if col in row.index:
                failed_result[col] = row[col]

        master_rows.append(failed_result)


# ------------------------------------------------------------
# SAVE MASTER SUMMARY TABLE
# ------------------------------------------------------------

master = pd.DataFrame(master_rows)

master.to_csv(master_output_csv, index=False)

print("\nBatch restricted cached MARCS fitting complete.")
print(f"Modeled successfully: {(master['status'] == 'modeled').sum()}")
print(f"Failed/skipped:       {(master['status'] != 'modeled').sum()}")

print("\nMaster summary written to:")
print(master_output_csv)

print("\nPNG directory:")
print(output_dir)

print("\nFirst few rows of master table:")
print(master.head())

